# 04 ResNet SASC Analysis and Diagnostics

This notebook contains the ResNet paper-freeze analysis for Support-Aware Stream Calibration (SASC). It analyzes cached logits for ResNet-18 and ResNet-50 under ImageNet-C corruption shift.

Main steps covered in this notebook:

- validate cached target logits and source-bank logits;
- refit clean-source global temperature scaling from cached source calibration logits;
- regenerate source-bank and target-stream descriptors;
- fit and evaluate Safe V2 and frozen V3/SASC;
- run LOCO and LOSO condition-split evaluations;
- run frozen V3 ablations for fallback, cap, monotonicity, and high-entropy extrapolation;
- compute support/extrapolation diagnostics, bootstrap confidence intervals, and final export tables.

Large datasets, model checkpoints, and cached logits are not stored in this GitHub repository. The notebook assumes that required cached artifacts are available locally or through the external artifact link described in the reproducibility documentation.

Target labels are used only for evaluation metrics and target-oracle diagnostics. Deployable SASC temperatures are fitted from source-bank information and unlabeled target-stream descriptors only.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
import json
import math
import hashlib
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

AML_FINAL_ROOT = Path("/content/drive/MyDrive/AML_final")
AML_CLEAN_ROOT = Path("/content/drive/MyDrive/AML")

PAPER_ROOT = Path("/content/drive/MyDrive/AML_paper_freeze_v3")

assert AML_FINAL_ROOT.exists(), f"Missing AML_final root: {AML_FINAL_ROOT}"
assert AML_CLEAN_ROOT.exists(), f"Missing AML clean/reference root: {AML_CLEAN_ROOT}"

DIRS = {
    "root": PAPER_ROOT,

    "configs": PAPER_ROOT / "configs",
    "manifests": PAPER_ROOT / "manifests",
    "tables": PAPER_ROOT / "tables",
    "figures": PAPER_ROOT / "figures",
    "logs": PAPER_ROOT / "logs",

    "source_ts": PAPER_ROOT / "source_ts",
    "source_calibration": PAPER_ROOT / "source_calibration",

    "oracle_diagnostics": PAPER_ROOT / "oracle_diagnostics",

    "stream_methods": PAPER_ROOT / "stream_methods",
    "safe_v2": PAPER_ROOT / "safe_v2_binned_isotonic",
    "safe_v3": PAPER_ROOT / "safe_v3_frozen",

    "v3_ablations": PAPER_ROOT / "v3_ablations",
    "support_extrapolation": PAPER_ROOT / "support_extrapolation",
    "baselines": PAPER_ROOT / "baselines",
    "stream_size_sensitivity": PAPER_ROOT / "stream_size_sensitivity",

    "stats": PAPER_ROOT / "statistical_analysis",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Paper-freeze root:", PAPER_ROOT)
print("Created folders:")
for name, path in DIRS.items():
    print(f"{name:28s} -> {path}")

**Global freeze protocol**

In [ ]:
PROTOCOL = {
    "project": "AML calibration under corruption shift",
    "stage": "paper_freeze_v3",
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "input_roots": {
        "aml_final": str(AML_FINAL_ROOT),
        "aml_clean_reference": str(AML_CLEAN_ROOT),
    },

    "output_root": str(PAPER_ROOT),

    "backbones": ["resnet18", "resnet50"],
    "seeds": [1, 2, 3],

    "corruptions": [
        "brightness",
        "defocus_blur",
        "glass_blur",
        "motion_blur",
        "zoom_blur",
        "gaussian_noise",
        "shot_noise",
        "impulse_noise",
        "fog",
        "snow",
    ],

    "severities": [1, 2, 3, 4, 5],
    "num_classes": 100,

    "metrics": [
        "accuracy",
        "mean_confidence",
        "signed_gap",
        "abs_signed_gap",
        "nll",
        "brier",
        "ece15",
        "aece15",
    ],

    "condition_splits": [
        "main_all_source_bank_train",
        "leave_one_corruption_family_out",
        "leave_one_severity_out",
    ],

    "main_method_blocks": [
        "raw",
        "source_global_ts_refit_from_clean_source_logits",
        "stream_entropy_mean_only_ridge",
        "stream_entropy_stats_ridge",
        "stream_all_stats_ridge_no_class_entropy",
        "stream_all_stats_mlp_no_class_entropy",
        "safe_v2_binned_isotonic",
        "safe_v3_frozen",
        "target_oracle_diagnostic_only",
    ],

    "planned_additions_after_current_code_audit": [
        "refit source TS from cached clean/source calibration logits",
        "add V2 binned isotonic safe method evaluation",
        "add MLP stream baseline",
        "add stream-size sensitivity",
        "add V3 ablations for LOCO and LOSO",
        "add figures for oracle, gaps, support/extrapolation, ablations, and stream-size sensitivity",
    ],

    "important_notes": [
        "Target-oracle temperatures are diagnostic only and use target labels.",
        "Deployable stream methods must not use target labels for fitting.",
        "ECE values must be computed from logits; never use row counts as fallback.",
        "Source TS must be regenerated from cached clean/source calibration logits for the final rerun.",
        "V3 was developed during exploratory analysis, then frozen before the final paper-level rerun.",
        "Final reported V3 numbers use the frozen configuration and must not be adjusted after this freeze.",
        "stream_entropy_mean_only is treated as the closest simple adaptive entropy baseline.",
        "Extra architecture/dataset is optional and will be done after ResNet18/ResNet50 are completed.",
    ],
}

protocol_path = DIRS["configs"] / "paper_freeze_protocol_config.json"
run_config_path = DIRS["manifests"] / "run_config.json"

with open(protocol_path, "w") as f:
    json.dump(PROTOCOL, f, indent=2)

with open(run_config_path, "w") as f:
    json.dump(PROTOCOL, f, indent=2)

print("Saved protocol config:", protocol_path)
print("Saved run config:", run_config_path)

**Frozen V3 config**

In [ ]:
FROZEN_V3_CONFIG = {
    "method": "safe_stream_entropy_v3_capped_extrapolation",

    "n_entropy_bins": 8,
    "support_margin_std": 0.25,
    "t_max_absolute": 1.45,
    "t_max_additive": 0.35,
    "max_extrapolation_slope": 0.75,
    "lower_fallback_to_source_ts": True,

    "fit_data": "source_bank_only",
    "target_labels_used_for_fit": False,
    "target_metrics_used_for_hyperparameter_selection": False,

    "freeze_status": "frozen_before_final_paper_level_rerun",
    "honesty_note": (
        "V3 was developed during exploratory analysis, then frozen before the final paper-level rerun. "
        "Final reported numbers must use this frozen configuration, with no further adjustment after the freeze."
    ),
}

v3_config_path = DIRS["configs"] / "frozen_v3_config.json"
v3_manifest_config_path = DIRS["manifests"] / "frozen_v3_config.json"

with open(v3_config_path, "w") as f:
    json.dump(FROZEN_V3_CONFIG, f, indent=2)

with open(v3_manifest_config_path, "w") as f:
    json.dump(FROZEN_V3_CONFIG, f, indent=2)

print("Saved frozen V3 config:", v3_config_path)
print("Saved manifest copy:", v3_manifest_config_path)
print(json.dumps(FROZEN_V3_CONFIG, indent=2))

In [ ]:
INPUT_PATHS = {
    # Target ImageNet-C cached logits
    "target_logits_cache_root": AML_FINAL_ROOT / "logits_cache",

    # Source-bank pseudo-domain logits
    "source_bank_logits_root": AML_FINAL_ROOT / "results" / "conditioned_ts" / "bank_logits",

    # Candidate roots for clean/source calibration logits.
    # Stage 1 will search these and select the valid cached clean/source calibration logits.
    "candidate_source_calibration_roots": [
        AML_FINAL_ROOT / "logits_cache",
        AML_FINAL_ROOT / "results" / "ts",
        AML_FINAL_ROOT / "source_ts",
        AML_FINAL_ROOT / "results",
        AML_CLEAN_ROOT / "logits_cache",
        AML_CLEAN_ROOT / "source_ts",
        AML_CLEAN_ROOT / "tables",
    ],

    # Old outputs kept only for audit/comparison, not as final source of truth.
    "old_results_root": AML_FINAL_ROOT / "results",
    "old_theory_root": AML_FINAL_ROOT / "theory",
    "clean_tables_root": AML_CLEAN_ROOT / "tables",
    "clean_stream_root": AML_CLEAN_ROOT / "stream_level",
    "clean_condition_splits_root": AML_CLEAN_ROOT / "condition_level_splits",
    "clean_manifest_root": AML_CLEAN_ROOT / "manifest",

    # Old source TS summaries: reference only.
    "old_resnet18_source_ts_fit_summary": AML_FINAL_ROOT / "results" / "ts" / "fit_summary.csv",
    "old_resnet50_source_ts_fit_summary": AML_FINAL_ROOT / "theory" / "resnet50_source_ts_fit_summary.csv",
}

input_registry = []

for name, path_or_list in INPUT_PATHS.items():
    if isinstance(path_or_list, list):
        for i, path in enumerate(path_or_list):
            input_registry.append({
                "name": f"{name}_{i}",
                "path": str(path),
                "exists": path.exists(),
                "is_file": path.is_file(),
                "is_dir": path.is_dir(),
                "role": "candidate_source_calibration_root",
            })
    else:
        path = path_or_list
        input_registry.append({
            "name": name,
            "path": str(path),
            "exists": path.exists(),
            "is_file": path.is_file(),
            "is_dir": path.is_dir(),
            "role": "required_or_reference_input",
        })

input_registry_df = pd.DataFrame(input_registry)

input_registry_path = DIRS["manifests"] / "input_path_registry.csv"
input_manifest_path = DIRS["manifests"] / "input_manifest.csv"

input_registry_df.to_csv(input_registry_path, index=False)
input_registry_df.to_csv(input_manifest_path, index=False)

print("Saved input path registry:", input_registry_path)
print("Saved input manifest:", input_manifest_path)
display(input_registry_df)

required_inputs = [
    "target_logits_cache_root",
    "source_bank_logits_root",
]

missing_required = input_registry_df[
    (input_registry_df["name"].isin(required_inputs)) &
    (input_registry_df["exists"] == False)
]

assert len(missing_required) == 0, (
    "Missing required input paths:\n"
    + missing_required[["name", "path"]].to_string(index=False)
)

existing_source_calib_candidates = input_registry_df[
    (input_registry_df["role"] == "candidate_source_calibration_root") &
    (input_registry_df["exists"] == True)
]

assert len(existing_source_calib_candidates) > 0, (
    "No candidate source calibration roots exist. "
    "Stage 1 cannot regenerate source TS without clean/source calibration logits."
)

print("Existing candidate source calibration roots:")
display(existing_source_calib_candidates)

**Torch imports**

In [ ]:
import torch
import torch.nn.functional as F

print("Torch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

**STage 1**

**temperature fitting + source calibration logit discovery**

In [ ]:
def fit_temperature_nll(
    logits: torch.Tensor,
    labels: torch.Tensor,
    init_temperature: float = 1.0,
    max_iter: int = 100,
    lr: float = 0.05,
    min_temperature: float = 0.05,
    max_temperature: float = 10.0,
):
    """
    Fits scalar temperature by minimizing NLL on given logits/labels.
    Used for clean/source calibration TS and later oracle/source-bank fits.
    """
    logits = logits.detach().cpu().float()
    labels = labels.detach().cpu().long()

    assert logits.ndim == 2, f"Bad logits shape: {logits.shape}"
    assert labels.ndim == 1, f"Bad labels shape: {labels.shape}"
    assert logits.shape[0] == labels.shape[0], "Logits/labels length mismatch"
    assert logits.shape[1] == PROTOCOL["num_classes"], (
        f"Expected {PROTOCOL['num_classes']} classes, got {logits.shape[1]}"
    )

    log_temperature = torch.tensor(
        [math.log(init_temperature)],
        dtype=torch.float32,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=lr,
        max_iter=max_iter,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()

        temperature = torch.exp(log_temperature)
        temperature = torch.clamp(
            temperature,
            min=min_temperature,
            max=max_temperature,
        )

        loss = F.cross_entropy(logits / temperature, labels, reduction="mean")
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).item()
        temperature = float(np.clip(temperature, min_temperature, max_temperature))

        nll_before = F.cross_entropy(logits, labels, reduction="mean").item()
        nll_after = F.cross_entropy(logits / temperature, labels, reduction="mean").item()

    return {
        "temperature": float(temperature),
        "log_temperature": float(math.log(temperature)),
        "nll_before": float(nll_before),
        "nll_after": float(nll_after),
        "nll_improvement": float(nll_before - nll_after),
    }


def load_logits_labels_file(path: Path):
    """
    Loads logits/labels from .pt, .pth, or .npz files.
    Expected keys include logits and labels/targets/y.
    """
    path = Path(path)

    if path.suffix.lower() in [".pt", ".pth"]:
        blob = torch.load(path, map_location="cpu", weights_only=False)

        if isinstance(blob, dict):
            logits_key = None
            labels_key = None

            for k in ["logits", "outputs", "scores"]:
                if k in blob:
                    logits_key = k
                    break

            for k in ["labels", "targets", "y"]:
                if k in blob:
                    labels_key = k
                    break

            assert logits_key is not None, f"No logits key found in {path}. Keys: {list(blob.keys())}"
            assert labels_key is not None, f"No labels key found in {path}. Keys: {list(blob.keys())}"

            logits = blob[logits_key]
            labels = blob[labels_key]
        else:
            raise ValueError(f"Unsupported torch object type in {path}: {type(blob)}")

    elif path.suffix.lower() == ".npz":
        blob = np.load(path, allow_pickle=True)

        logits_key = None
        labels_key = None

        for k in ["logits", "outputs", "scores"]:
            if k in blob.files:
                logits_key = k
                break

        for k in ["labels", "targets", "y"]:
            if k in blob.files:
                labels_key = k
                break

        assert logits_key is not None, f"No logits key found in {path}. Keys: {blob.files}"
        assert labels_key is not None, f"No labels key found in {path}. Keys: {blob.files}"

        logits = blob[logits_key]
        labels = blob[labels_key]

    else:
        raise ValueError(f"Unsupported logits file type: {path}")

    logits = torch.as_tensor(logits).detach().cpu().float()
    labels = torch.as_tensor(labels).detach().cpu().long()

    assert logits.ndim == 2, f"Bad logits shape in {path}: {logits.shape}"
    assert labels.ndim == 1, f"Bad labels shape in {path}: {labels.shape}"
    assert logits.shape[0] == labels.shape[0], f"Logits/labels length mismatch in {path}"
    assert logits.shape[1] == PROTOCOL["num_classes"], (
        f"Expected {PROTOCOL['num_classes']} classes in {path}, got {logits.shape[1]}"
    )

    return logits, labels


def infer_seed_from_path(path: Path):
    text = str(path).lower()

    for seed in PROTOCOL["seeds"]:
        patterns = [
            f"seed_{seed}",
            f"seed{seed}",
            f"seed-{seed}",
            f"/{seed}/",
        ]

        if any(p in text for p in patterns):
            return int(seed)

    return None


def infer_backbone_from_path(path: Path):
    text = str(path).lower()

    if "resnet50" in text or "resnet_50" in text or "rn50" in text:
        return "resnet50"

    if "resnet18" in text or "resnet_18" in text or "rn18" in text:
        return "resnet18"

    return None


def source_calibration_priority(path: Path):
    """
    Lower score is better.
    Prefer explicit source/calibration logits over generic clean validation logits.
    """
    text = str(path).lower()
    name = path.name.lower()

    score = 100

    if "calib_logits" in name:
        score -= 50
    if "source" in text and "calib" in text:
        score -= 40
    if "calib" in text:
        score -= 30
    if "clean" in text:
        score -= 10
    if "val_logits_clean" in name:
        score -= 5

    if "imagenet-c" in text or "imagenetc" in text:
        score += 100
    if "corruption" in text:
        score += 100
    if "bank_logits" in text:
        score += 100

    return score


def discover_source_calibration_logits():
    """
    Finds candidate clean/source calibration logits.
    If auto-selection fails, inspect the saved manifest and set MANUAL_SOURCE_CALIB_LOGITS.
    """
    candidate_roots = INPUT_PATHS["candidate_source_calibration_roots"]

    patterns = [
        "*calib*logits*.npz",
        "*calib*logits*.pt",
        "*source*logits*.npz",
        "*source*logits*.pt",
        "*clean*logits*.npz",
        "*clean*logits*.pt",
        "calib_logits.npz",
        "calib_logits.pt",
        "val_logits_clean.npz",
        "val_logits_clean.pt",
    ]

    rows = []

    for root in candidate_roots:
        root = Path(root)
        if not root.exists():
            continue

        for pattern in patterns:
            for path in root.rglob(pattern):
                if not path.is_file():
                    continue

                backbone = infer_backbone_from_path(path)
                seed = infer_seed_from_path(path)

                rows.append({
                    "path": str(path),
                    "name": path.name,
                    "parent": str(path.parent),
                    "suffix": path.suffix.lower(),
                    "backbone_inferred": backbone,
                    "seed_inferred": seed,
                    "priority": source_calibration_priority(path),
                    "size_bytes": int(path.stat().st_size),
                    "modified_time": datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
                })

    df = pd.DataFrame(rows).drop_duplicates(subset=["path"]).reset_index(drop=True)

    if len(df) > 0:
        df = df.sort_values(
            ["backbone_inferred", "seed_inferred", "priority", "path"],
            na_position="last",
        ).reset_index(drop=True)

    return df

MANUAL_SOURCE_CALIB_LOGITS = {}

source_calib_candidates_df = discover_source_calibration_logits()

source_calib_candidates_path = DIRS["source_calibration"] / "source_calibration_logit_candidates.csv"
source_calib_candidates_df.to_csv(source_calib_candidates_path, index=False)

print("Saved source calibration candidate manifest:", source_calib_candidates_path)
print("Number of candidate files:", len(source_calib_candidates_df))
display(source_calib_candidates_df.head(50))

**select source calibration logits and refit source TS**

In [ ]:
# ============================================================
# Stage 1.1 — Refit source TS from cached clean/source calibration logits
# Robust source calibration logit discovery
# ============================================================

def infer_seed_from_path(path: Path):
    text = str(path).lower()

    for seed in PROTOCOL["seeds"]:
        patterns = [
            f"seed_{seed}",
            f"seed{seed}",
            f"seed-{seed}",
            f"/{seed}/",
        ]

        if any(p in text for p in patterns):
            return int(seed)

    return None


def infer_backbone_from_path(path: Path):
    text = str(path).lower()

    if "resnet50" in text or "resnet_50" in text or "rn50" in text:
        return "resnet50"

    if "resnet18" in text or "resnet_18" in text or "rn18" in text:
        return "resnet18"

    return None


def source_calibration_priority(path: Path):
    text = str(path).lower()
    name = path.name.lower()

    score = 100

    # prefer explicit calibration logits
    if name == "calib_logits.npz" or name == "calib_logits.pt":
        score -= 80
    if "calib_logits" in name:
        score -= 70
    if "calib" in text:
        score -= 40
    if "source" in text:
        score -= 20
    if "clean" in text:
        score -= 10
    if "val_logits_clean" in name:
        score -= 5

    # avoid target/corruption/bank files
    if "imagenet-c" in text or "imagenetc" in text:
        score += 100
    if "corruption" in text:
        score += 100
    if "bank_logits" in text:
        score += 100
    if "conditioned_ts" in text:
        score += 80

    return score


def discover_source_calibration_logits():
    candidate_roots = INPUT_PATHS["candidate_source_calibration_roots"]

    rows = []

    for root in candidate_roots:
        root = Path(root)
        if not root.exists():
            continue

        # Broad scan, then filter by loadable file type and likely name/path.
        for path in list(root.rglob("*.npz")) + list(root.rglob("*.pt")) + list(root.rglob("*.pth")):
            if not path.is_file():
                continue

            text = str(path).lower()
            name = path.name.lower()

            likely_source_calib = (
                "calib" in text
                or "source" in text
                or "clean" in text
                or "val_logits_clean" in name
            )

            clearly_bad = (
                "bank_logits" in text
                or "conditioned_ts" in text
                or "imagenet-c" in text
                or "imagenetc" in text
            )

            if not likely_source_calib or clearly_bad:
                continue

            rows.append({
                "path": str(path),
                "name": path.name,
                "parent": str(path.parent),
                "suffix": path.suffix.lower(),
                "backbone_inferred": infer_backbone_from_path(path),
                "seed_inferred": infer_seed_from_path(path),
                "priority": source_calibration_priority(path),
                "size_bytes": int(path.stat().st_size),
                "modified_time": datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            })

    columns = [
        "path",
        "name",
        "parent",
        "suffix",
        "backbone_inferred",
        "seed_inferred",
        "priority",
        "size_bytes",
        "modified_time",
    ]

    df = pd.DataFrame(rows, columns=columns).drop_duplicates(subset=["path"]).reset_index(drop=True)

    if len(df) > 0:
        df = df.sort_values(
            ["backbone_inferred", "seed_inferred", "priority", "path"],
            na_position="last",
        ).reset_index(drop=True)

    return df


MANUAL_SOURCE_CALIB_LOGITS = {}

source_calib_candidates_df = discover_source_calibration_logits()

source_calib_candidates_path = DIRS["source_calibration"] / "source_calibration_logit_candidates.csv"
source_calib_candidates_df.to_csv(source_calib_candidates_path, index=False)

print("Saved source calibration candidate manifest:", source_calib_candidates_path)
print("Number of candidate files:", len(source_calib_candidates_df))
display(source_calib_candidates_df.head(100))

assert len(source_calib_candidates_df) > 0, (
    "No source calibration logits were found. "
    "Open the candidate roots and manually set MANUAL_SOURCE_CALIB_LOGITS."
)

In [ ]:
# Fill missing backbone labels for ResNet18 source calibration files.
# Files in the main logits_cache root without resnet50 in path are ResNet18.
source_calib_candidates_df["backbone_inferred"] = source_calib_candidates_df.apply(
    lambda r: "resnet18"
    if pd.isna(r["backbone_inferred"]) and "resnet50" not in str(r["path"]).lower()
    else r["backbone_inferred"],
    axis=1,
)

source_calib_candidates_df = source_calib_candidates_df.sort_values(
    ["backbone_inferred", "seed_inferred", "priority", "path"],
    na_position="last",
).reset_index(drop=True)

source_calib_candidates_df.to_csv(source_calib_candidates_path, index=False)

display(source_calib_candidates_df)

In [ ]:
selected_source_calib_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        manual_key = (backbone, seed)

        if manual_key in MANUAL_SOURCE_CALIB_LOGITS:
            selected_path = Path(MANUAL_SOURCE_CALIB_LOGITS[manual_key])
            selection_mode = "manual_override"
        else:
            sub = source_calib_candidates_df[
                (source_calib_candidates_df["backbone_inferred"] == backbone) &
                (source_calib_candidates_df["seed_inferred"] == seed)
            ].copy()

            assert len(sub) > 0, (
                f"No source calibration logits found for {backbone}, seed={seed}. "
                f"Check {source_calib_candidates_path} and set MANUAL_SOURCE_CALIB_LOGITS."
            )

            sub = sub.sort_values(["priority", "path"]).reset_index(drop=True)
            selected_path = Path(sub.iloc[0]["path"])
            selection_mode = "auto_selected_lowest_priority"

        assert selected_path.exists(), f"Selected source calibration file does not exist: {selected_path}"

        logits, labels = load_logits_labels_file(selected_path)

        selected_source_calib_rows.append({
            "backbone": backbone,
            "seed": int(seed),
            "path": str(selected_path),
            "selection_mode": selection_mode,
            "n_samples": int(len(labels)),
            "n_classes": int(logits.shape[1]),
            "raw_nll": float(F.cross_entropy(logits, labels, reduction="mean").item()),
        })

        del logits, labels

selected_source_calib_df = pd.DataFrame(selected_source_calib_rows)

selected_source_calib_path = DIRS["source_calibration"] / "selected_source_calibration_logits.csv"
selected_source_calib_df.to_csv(selected_source_calib_path, index=False)

print("Saved selected source calibration logits:", selected_source_calib_path)
display(selected_source_calib_df)


source_ts_rows = []

for _, row in selected_source_calib_df.iterrows():
    backbone = row["backbone"]
    seed = int(row["seed"])
    path = Path(row["path"])

    print(f"\nRefitting source TS | {backbone} | seed={seed}")
    print("File:", path)

    logits, labels = load_logits_labels_file(path)

    fit = fit_temperature_nll(
        logits=logits,
        labels=labels,
        init_temperature=1.0,
        max_iter=100,
        lr=0.05,
        min_temperature=0.05,
        max_temperature=10.0,
    )

    source_ts_rows.append({
        "information_regime": "source_only",
        "method": "source_global_ts",
        "fit_data": "clean_source_calibration_logits",
        "backbone": backbone,
        "seed": int(seed),
        "source_calibration_logits_path": str(path),
        "n_samples": int(len(labels)),
        "n_classes": int(logits.shape[1]),
        "temperature": float(fit["temperature"]),
        "log_temperature": float(fit["log_temperature"]),
        "nll_before_ts": float(fit["nll_before"]),
        "nll_after_ts": float(fit["nll_after"]),
        "nll_improvement": float(fit["nll_improvement"]),
    })

    del logits, labels

source_ts_refit_df = pd.DataFrame(source_ts_rows)

assert len(source_ts_refit_df) == len(PROTOCOL["backbones"]) * len(PROTOCOL["seeds"])
assert source_ts_refit_df["temperature"].gt(0).all()
assert source_ts_refit_df["nll_after_ts"].notna().all()
assert (
    source_ts_refit_df["nll_after_ts"]
    <= source_ts_refit_df["nll_before_ts"] + 1e-6
).all(), "Some source TS fits worsened source calibration NLL."

source_ts_refit_path = DIRS["source_ts"] / "refit_source_ts_from_clean_source_logits.csv"
source_ts_refit_df.to_csv(source_ts_refit_path, index=False)

SOURCE_T_BY_BACKBONE = {
    backbone: {
        int(seed): float(temp)
        for seed, temp in zip(
            source_ts_refit_df[source_ts_refit_df["backbone"] == backbone]["seed"],
            source_ts_refit_df[source_ts_refit_df["backbone"] == backbone]["temperature"],
        )
    }
    for backbone in PROTOCOL["backbones"]
}

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        assert seed in SOURCE_T_BY_BACKBONE[backbone], (
            f"Missing refit source temperature for {backbone}, seed={seed}"
        )
        assert SOURCE_T_BY_BACKBONE[backbone][seed] > 0, (
            f"Invalid source temperature for {backbone}, seed={seed}"
        )

print("\nSaved refit source TS:", source_ts_refit_path)
print("Refit source temperatures:")
for backbone in PROTOCOL["backbones"]:
    print(backbone)
    for seed in PROTOCOL["seeds"]:
        print(f"  seed {seed}: T = {SOURCE_T_BY_BACKBONE[backbone][seed]:.6f}")

display(source_ts_refit_df.round(6))

**Cached logit path helpers**

In [ ]:
def logits_cache_path(backbone: str, seed: int, corruption: str, severity: int) -> Path:
    condition = f"{corruption}_sev{severity}"
    root = INPUT_PATHS["target_logits_cache_root"]

    if backbone == "resnet18":
        return root / f"seed{seed}_{condition}.pt"

    if backbone == "resnet50":
        return root / "resnet50" / f"seed{seed}_{condition}.pt"

    raise ValueError(f"Unknown backbone: {backbone}")


def load_cached_logits(backbone: str, seed: int, corruption: str, severity: int):
    path = logits_cache_path(backbone, seed, corruption, severity)
    assert path.exists(), f"Missing cached target logits: {path}"

    blob = torch.load(path, map_location="cpu", weights_only=False)

    logits = blob["logits"].detach().cpu().float()
    labels = blob["labels"].detach().cpu().long()

    assert logits.ndim == 2, f"Bad logits shape: {logits.shape}"
    assert labels.ndim == 1, f"Bad labels shape: {labels.shape}"
    assert logits.shape[0] == labels.shape[0], "Logits/labels length mismatch"
    assert logits.shape[1] == PROTOCOL["num_classes"], f"Expected {PROTOCOL['num_classes']} classes"

    return logits, labels


print("Cached target logit helpers ready.")

**Validate all cached targets logits**

In [ ]:
logit_manifest_rows = []
missing_logits = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                path = logits_cache_path(backbone, seed, corruption, severity)
                exists = path.exists()

                row = {
                    "backbone": backbone,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": f"{corruption}_sev{severity}",
                    "path": str(path),
                    "exists": bool(exists),
                    "size_bytes": int(path.stat().st_size) if exists else 0,
                }

                if not exists:
                    missing_logits.append(str(path))
                else:
                    logits, labels = load_cached_logits(backbone, seed, corruption, severity)
                    row["n_samples"] = int(len(labels))
                    row["n_classes"] = int(logits.shape[1])

                logit_manifest_rows.append(row)

logit_manifest_df = pd.DataFrame(logit_manifest_rows)

logit_manifest_path = DIRS["manifests"] / "cached_logits_manifest.csv"
logit_manifest_df.to_csv(logit_manifest_path, index=False)

print("Saved cached logits manifest:", logit_manifest_path)
print("Expected cached logit files:", len(PROTOCOL["backbones"]) * len(PROTOCOL["seeds"]) * len(PROTOCOL["corruptions"]) * len(PROTOCOL["severities"]))
print("Missing cached logit files:", len(missing_logits))

assert len(missing_logits) == 0, f"Missing logits examples: {missing_logits[:5]}"

display(logit_manifest_df.head())
display(
    logit_manifest_df
    .groupby(["backbone", "seed"], as_index=False)
    .agg(
        n_files=("exists", "sum"),
        total_samples=("n_samples", "sum"),
        min_samples=("n_samples", "min"),
        max_samples=("n_samples", "max"),
    )
)

**Metrics**

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    assert temperature > 0, f"Temperature must be positive, got {temperature}"
    return logits.detach().cpu().float() / float(temperature)


def nll_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    return F.cross_entropy(logits.float(), labels.long(), reduction="mean").item()


def brier_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    probs = F.softmax(logits.float(), dim=1)
    one_hot = F.one_hot(labels.long(), num_classes=probs.shape[1]).float()
    return ((probs - one_hot) ** 2).sum(dim=1).mean().item()


def ece_from_logits(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    probs = F.softmax(logits.float(), dim=1)
    confidences, predictions = probs.max(dim=1)
    accuracies = predictions.eq(labels.long()).float()

    bin_edges = torch.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        if i == 0:
            in_bin = (confidences >= lo) & (confidences <= hi)
        else:
            in_bin = (confidences > lo) & (confidences <= hi)

        prop = in_bin.float().mean().item()

        if prop > 0:
            acc_bin = accuracies[in_bin].mean().item()
            conf_bin = confidences[in_bin].mean().item()
            ece += abs(acc_bin - conf_bin) * prop

    return float(ece)


def adaptive_ece_from_logits(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    probs = F.softmax(logits.float(), dim=1)
    confidences, predictions = probs.max(dim=1)
    accuracies = predictions.eq(labels.long()).float()

    confidences = confidences.detach().cpu()
    accuracies = accuracies.detach().cpu()

    n = len(confidences)
    if n == 0:
        return 0.0

    order = torch.argsort(confidences)
    confidences = confidences[order]
    accuracies = accuracies[order]

    bin_sizes = [n // n_bins] * n_bins
    for i in range(n % n_bins):
        bin_sizes[i] += 1

    aece = 0.0
    start = 0

    for size in bin_sizes:
        if size == 0:
            continue

        end = start + size
        conf_bin = confidences[start:end]
        acc_bin = accuracies[start:end]

        aece += abs(conf_bin.mean().item() - acc_bin.mean().item()) * (size / n)
        start = end

    return float(aece)


def metrics_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    logits = logits.detach().cpu().float()
    labels = labels.detach().cpu().long()

    probs = F.softmax(logits, dim=1)
    confidence, prediction = probs.max(dim=1)

    accuracy = prediction.eq(labels).float().mean().item()
    mean_confidence = confidence.mean().item()
    signed_gap = mean_confidence - accuracy

    out = {
        "accuracy": float(accuracy),
        "mean_confidence": float(mean_confidence),
        "signed_gap": float(signed_gap),
        "abs_signed_gap": float(abs(signed_gap)),
        "nll": float(nll_from_logits(logits, labels)),
        "brier": float(brier_from_logits(logits, labels)),
        "ece15": float(ece_from_logits(logits, labels, n_bins=15)),
        "aece15": float(adaptive_ece_from_logits(logits, labels, n_bins=15)),
    }

    assert 0.0 <= out["ece15"] <= 1.0, f"Invalid ECE: {out['ece15']}"
    assert 0.0 <= out["aece15"] <= 1.0, f"Invalid AECE: {out['aece15']}"

    return out


print("Metric functions ready.")

**Regenerate raw and refit-source-TS target evaluation table**

In [ ]:
baseline_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

        print(f"Evaluating target conditions | {backbone}, seed={seed}, refit_source_T={source_T:.6f}")

        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                logits, labels = load_cached_logits(
                    backbone=backbone,
                    seed=seed,
                    corruption=corruption,
                    severity=severity,
                )

                method_to_temperature = {
                    "raw": 1.0,
                    "source_global_ts": source_T,
                }

                for method, temperature in method_to_temperature.items():
                    scaled_logits = apply_temperature(logits, temperature)
                    met = metrics_from_logits(scaled_logits, labels)

                    for metric_name, value in met.items():
                        baseline_rows.append({
                            "split_name": "main_all_source_bank_train",
                            "heldout_type": "none",
                            "heldout_value": "none",
                            "information_regime": "raw" if method == "raw" else "source_only",
                            "backbone": backbone,
                            "seed": int(seed),
                            "method": method,
                            "source_ts_origin": (
                                "none" if method == "raw"
                                else "refit_from_clean_source_calibration_logits"
                            ),
                            "corruption": corruption,
                            "severity": int(severity),
                            "condition": f"{corruption}_sev{severity}",
                            "temperature": float(temperature),
                            "metric": metric_name,
                            "value": float(value),
                        })

                del logits, labels

baseline_eval_long_corrected = pd.DataFrame(baseline_rows)

expected_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
    * 2
    * len(PROTOCOL["metrics"])
)

assert len(baseline_eval_long_corrected) == expected_rows, (
    f"Expected {expected_rows} rows, got {len(baseline_eval_long_corrected)}"
)

assert baseline_eval_long_corrected["value"].notna().all()

baseline_eval_path = DIRS["tables"] / "corrected_baseline_eval_long.csv"
baseline_eval_compat_path = DIRS["tables"] / "corrected_raw_source_ts_eval_long.csv"

baseline_eval_long_corrected.to_csv(baseline_eval_path, index=False)
baseline_eval_long_corrected.to_csv(baseline_eval_compat_path, index=False)

print("Saved:", baseline_eval_path)
print("Saved compatibility copy:", baseline_eval_compat_path)
print("Shape:", baseline_eval_long_corrected.shape)

display(baseline_eval_long_corrected.head())

**condition-wide table**

In [ ]:
corrected_condition_metric_wide = (
    baseline_eval_long_corrected
    .pivot_table(
        index=[
            "split_name",
            "heldout_type",
            "heldout_value",
            "information_regime",
            "backbone",
            "seed",
            "method",
            "source_ts_origin",
            "corruption",
            "severity",
            "condition",
            "temperature",
        ],
        columns="metric",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)

corrected_condition_metric_wide.columns.name = None

required_metric_cols = [
    "accuracy",
    "mean_confidence",
    "signed_gap",
    "abs_signed_gap",
    "nll",
    "brier",
    "ece15",
    "aece15",
]

missing_metric_cols = [c for c in required_metric_cols if c not in corrected_condition_metric_wide.columns]
assert len(missing_metric_cols) == 0, f"Missing metric columns: {missing_metric_cols}"

assert corrected_condition_metric_wide["ece15"].between(0, 1).all(), "ECE15 outside [0, 1]"
assert corrected_condition_metric_wide["aece15"].between(0, 1).all(), "AECE15 outside [0, 1]"

expected_wide_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
    * 2
)

assert len(corrected_condition_metric_wide) == expected_wide_rows, (
    f"Expected {expected_wide_rows} wide rows, got {len(corrected_condition_metric_wide)}"
)

condition_wide_path = DIRS["tables"] / "corrected_condition_metric_wide.csv"
condition_wide_compat_path = DIRS["tables"] / "corrected_raw_source_ts_condition_metric_wide.csv"

corrected_condition_metric_wide.to_csv(condition_wide_path, index=False)
corrected_condition_metric_wide.to_csv(condition_wide_compat_path, index=False)

print("Saved:", condition_wide_path)
print("Saved compatibility copy:", condition_wide_compat_path)
print("Shape:", corrected_condition_metric_wide.shape)

display(corrected_condition_metric_wide.head())

**source/reference summary**

In [ ]:
corrected_source_reference_summary = (
    corrected_condition_metric_wide
    .groupby(
        [
            "split_name",
            "heldout_type",
            "heldout_value",
            "information_regime",
            "backbone",
            "method",
            "source_ts_origin",
        ],
        as_index=False,
    )
    .agg(
        accuracy_mean=("accuracy", "mean"),
        mean_confidence_mean=("mean_confidence", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),
        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        temperature_mean=("temperature", "mean"),
        temperature_min=("temperature", "min"),
        temperature_max=("temperature", "max"),
        n_conditions=("nll", "count"),
    )
)

assert corrected_source_reference_summary["ece15_mean"].between(0, 1).all(), "Bad ECE summary"
assert corrected_source_reference_summary["aece15_mean"].between(0, 1).all(), "Bad AECE summary"
assert corrected_source_reference_summary["ece15_mean"].max() < 1.0, (
    "ECE summary is suspicious. ECE should be a calibration error in [0,1], not a count."
)

source_ref_path = DIRS["tables"] / "corrected_source_reference_summary.csv"
corrected_source_reference_summary.to_csv(source_ref_path, index=False)

print("Saved:", source_ref_path)

display(corrected_source_reference_summary.round(6))

**Add raw-to-source-TS deltas**

In [ ]:
raw_wide = corrected_condition_metric_wide[
    corrected_condition_metric_wide["method"] == "raw"
].copy()

source_wide = corrected_condition_metric_wide[
    corrected_condition_metric_wide["method"] == "source_global_ts"
].copy()

merge_keys = ["backbone", "seed", "corruption", "severity", "condition"]

raw_keep = merge_keys + required_metric_cols
source_keep = merge_keys + required_metric_cols + ["temperature"]

raw_wide = raw_wide[raw_keep].rename(columns={m: f"raw_{m}" for m in required_metric_cols})
source_wide = source_wide[source_keep].rename(
    columns={m: f"source_ts_{m}" for m in required_metric_cols}
).rename(columns={"temperature": "source_ts_temperature"})

raw_source_delta = source_wide.merge(
    raw_wide,
    on=merge_keys,
    how="inner",
)

expected_delta_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(raw_source_delta) == expected_delta_rows, (
    f"Expected {expected_delta_rows}, got {len(raw_source_delta)}"
)

for metric in required_metric_cols:
    raw_source_delta[f"delta_{metric}_source_ts_minus_raw"] = (
        raw_source_delta[f"source_ts_{metric}"] - raw_source_delta[f"raw_{metric}"]
    )

assert (raw_source_delta["delta_accuracy_source_ts_minus_raw"].abs() < 1e-10).all(), (
    "Source TS changed accuracy. This should not happen because positive temperature scaling preserves argmax."
)

assert raw_source_delta["source_ts_ece15"].between(0, 1).all()
assert raw_source_delta["source_ts_aece15"].between(0, 1).all()
assert raw_source_delta["raw_ece15"].between(0, 1).all()
assert raw_source_delta["raw_aece15"].between(0, 1).all()

raw_source_delta_path = DIRS["tables"] / "corrected_source_ts_delta_vs_raw_wide.csv"
raw_source_delta.to_csv(raw_source_delta_path, index=False)

print("Saved:", raw_source_delta_path)

raw_source_delta_summary = (
    raw_source_delta
    .groupby(["backbone"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_source_ts_minus_raw", "mean"),
        delta_brier_mean=("delta_brier_source_ts_minus_raw", "mean"),
        delta_ece15_mean=("delta_ece15_source_ts_minus_raw", "mean"),
        delta_aece15_mean=("delta_aece15_source_ts_minus_raw", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_source_ts_minus_raw", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_source_ts_minus_raw", "mean"),
        source_ts_temperature_mean=("source_ts_temperature", "mean"),
        source_ts_temperature_min=("source_ts_temperature", "min"),
        source_ts_temperature_max=("source_ts_temperature", "max"),
        n_conditions=("delta_nll_source_ts_minus_raw", "count"),
    )
)

raw_source_delta_summary_path = DIRS["tables"] / "corrected_source_ts_delta_vs_raw_summary.csv"
raw_source_delta_summary.to_csv(raw_source_delta_summary_path, index=False)

print("Saved:", raw_source_delta_summary_path)
display(raw_source_delta_summary.round(6))

**Compare corrected summary to old faulty table, if it exists**

In [ ]:
old_faulty_source_ref_path = Path("/content/drive/MyDrive/AML_stage2_safe_stream_entropy/tables/stage2_source_reference_summary.csv")

if old_faulty_source_ref_path.exists():
    old_source_ref = pd.read_csv(old_faulty_source_ref_path)

    old_compare_path = DIRS["tables"] / "old_faulty_stage2_source_reference_summary_copy.csv"
    old_source_ref.to_csv(old_compare_path, index=False)

    print("Old faulty table found and copied to:", old_compare_path)

    print("\nOld ECE values:")
    display(old_source_ref[[c for c in old_source_ref.columns if "ece" in c.lower()] + ["backbone", "method"]].head())

    print("\nCorrected ECE values:")
    display(
        corrected_source_reference_summary[
            ["backbone", "method", "ece15_mean", "aece15_mean"]
        ].round(6)
    )
else:
    print("Old faulty source reference table not found. Skipping comparison.")

In [ ]:
stage1_note = {
    "stage": "stage_1_corrected_metrics_and_refit_source_ts",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "main_outputs": {
        "source_calibration_logit_candidates": str(source_calib_candidates_path),
        "selected_source_calibration_logits": str(selected_source_calib_path),
        "refit_source_ts_from_clean_source_logits": str(source_ts_refit_path),
        "cached_target_logits_manifest": str(logit_manifest_path),
        "corrected_baseline_eval_long": str(baseline_eval_path),
        "corrected_condition_metric_wide": str(condition_wide_path),
        "corrected_source_reference_summary": str(source_ref_path),
        "corrected_source_ts_delta_vs_raw_wide": str(raw_source_delta_path),
        "corrected_source_ts_delta_vs_raw_summary": str(raw_source_delta_summary_path),
    },
    "checks": {
        "source_ts_refit_from_clean_source_calibration_logits": True,
        "old_source_ts_csv_not_used_as_final_source_of_truth": True,
        "all_cached_target_logits_present": len(missing_logits) == 0,
        "ece15_computed_from_logits": True,
        "aece15_computed_from_logits": True,
        "ece15_fake_count_fallback_removed": True,
        "ece15_values_between_0_and_1": bool(corrected_source_reference_summary["ece15_mean"].between(0, 1).all()),
        "aece15_values_between_0_and_1": bool(corrected_source_reference_summary["aece15_mean"].between(0, 1).all()),
        "source_ts_preserves_accuracy": bool((raw_source_delta["delta_accuracy_source_ts_minus_raw"].abs() < 1e-10).all()),
    },
    "important_note": (
        "Stage 1 refits source temperature scaling from cached clean/source calibration logits, "
        "then evaluates raw and refit source TS on cached target ImageNet-C logits. "
        "ECE15 and AECE15 are computed directly from logits and are never replaced by row counts."
    ),
}

stage1_note_path = DIRS["logs"] / "stage1_corrected_metrics_and_refit_source_ts_completion.json"

with open(stage1_note_path, "w") as f:
    json.dump(stage1_note, f, indent=2)

print("Saved Stage 1 completion note:", stage1_note_path)
print(json.dumps(stage1_note, indent=2))

**Stage 2**

**Source-bank logit paths**

In [ ]:

BANK_LOGITS_ROOT = INPUT_PATHS["source_bank_logits_root"]

assert BANK_LOGITS_ROOT.exists(), f"Missing bank logits root: {BANK_LOGITS_ROOT}"

BANK_LOGITS_PATHS = {
    "resnet18": BANK_LOGITS_ROOT / "resnet18",
    "resnet50": BANK_LOGITS_ROOT / "resnet50",
}

for backbone, path in BANK_LOGITS_PATHS.items():
    print(backbone, "| exists:", path.exists(), "| path:", path)
    assert path.exists(), f"Missing bank logits path for {backbone}: {path}"

**Source-bank logit helper**

In [ ]:
def source_bank_logits_path(backbone: str, seed: int, corruption: str, severity: int, n_samples: int = 4096) -> Path:
    condition = f"{corruption}_sev{severity}"
    filename = f"seed{seed}_{condition}_n{n_samples}.pt"

    if backbone not in BANK_LOGITS_PATHS:
        raise ValueError(f"Unknown backbone: {backbone}")

    return BANK_LOGITS_PATHS[backbone] / filename


def load_source_bank_logits(backbone: str, seed: int, corruption: str, severity: int, n_samples: int = 4096):
    path = source_bank_logits_path(
        backbone=backbone,
        seed=seed,
        corruption=corruption,
        severity=severity,
        n_samples=n_samples,
    )

    assert path.exists(), f"Missing source-bank logits: {path}"

    blob = torch.load(path, map_location="cpu", weights_only=False)

    logits = blob["logits"].detach().cpu().float()
    labels = blob["labels"].detach().cpu().long()

    assert logits.ndim == 2, f"Bad logits shape: {logits.shape}"
    assert labels.ndim == 1, f"Bad labels shape: {labels.shape}"
    assert logits.shape[0] == labels.shape[0], "Logits/labels length mismatch"
    assert logits.shape[1] == PROTOCOL["num_classes"], f"Expected {PROTOCOL['num_classes']} classes"

    return logits, labels

**Validate source-bank logits**

In [ ]:
bank_manifest_rows = []
missing_bank_logits = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                path = source_bank_logits_path(
                    backbone=backbone,
                    seed=seed,
                    corruption=corruption,
                    severity=severity,
                    n_samples=4096,
                )

                exists = path.exists()

                row = {
                    "backbone": backbone,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": f"{corruption}_sev{severity}",
                    "path": str(path),
                    "exists": bool(exists),
                    "size_bytes": int(path.stat().st_size) if exists else 0,
                }

                if not exists:
                    missing_bank_logits.append(str(path))
                else:
                    logits, labels = load_source_bank_logits(backbone, seed, corruption, severity)
                    row["n_samples"] = int(len(labels))
                    row["n_classes"] = int(logits.shape[1])

                bank_manifest_rows.append(row)

source_bank_logits_manifest = pd.DataFrame(bank_manifest_rows)

source_bank_manifest_path = DIRS["manifests"] / "source_bank_logits_manifest.csv"
source_bank_logits_manifest.to_csv(source_bank_manifest_path, index=False)

print("Saved source-bank logits manifest:", source_bank_manifest_path)
print("Expected source-bank files:", len(PROTOCOL["backbones"]) * len(PROTOCOL["seeds"]) * len(PROTOCOL["corruptions"]) * len(PROTOCOL["severities"]))
print("Missing source-bank files:", len(missing_bank_logits))

assert len(missing_bank_logits) == 0, f"Missing source-bank logits examples: {missing_bank_logits[:5]}"

display(source_bank_logits_manifest.head())
display(
    source_bank_logits_manifest
    .groupby(["backbone", "seed"], as_index=False)
    .agg(
        n_files=("exists", "sum"),
        total_samples=("n_samples", "sum"),
        min_samples=("n_samples", "min"),
        max_samples=("n_samples", "max"),
    )
)

**Stream descriptor helper with explicit ddof=0**

In [ ]:
def stream_descriptors_from_logits(logits: torch.Tensor) -> dict:
    """
    Compute unlabeled stream-level descriptors from logits only.
    Labels are not used.
    Standard deviation uses ddof=0 explicitly for consistency.
    """
    logits = logits.detach().cpu().float()
    probs = F.softmax(logits, dim=1)

    sorted_probs, _ = torch.sort(probs, dim=1, descending=True)

    top1_conf = sorted_probs[:, 0]
    top2_conf = sorted_probs[:, 1]
    margin = top1_conf - top2_conf

    entropy = -(probs * probs.clamp_min(1e-12).log()).sum(dim=1)
    free_energy = -torch.logsumexp(logits, dim=1)

    def stats(prefix: str, values: torch.Tensor) -> dict:
        values_np = values.detach().cpu().numpy().astype(float)

        return {
            f"{prefix}_mean": float(np.mean(values_np)),
            f"{prefix}_std": float(np.std(values_np, ddof=0)),
            f"{prefix}_q10": float(np.quantile(values_np, 0.10)),
            f"{prefix}_q25": float(np.quantile(values_np, 0.25)),
            f"{prefix}_q50": float(np.quantile(values_np, 0.50)),
            f"{prefix}_q75": float(np.quantile(values_np, 0.75)),
            f"{prefix}_q90": float(np.quantile(values_np, 0.90)),
        }

    out = {}
    out.update(stats("entropy", entropy))
    out.update(stats("top1_confidence", top1_conf))
    out.update(stats("margin", margin))
    out.update(stats("free_energy", free_energy))

    out["n_stream_samples"] = int(logits.shape[0])
    out["n_classes"] = int(logits.shape[1])

    return out


print("Stream descriptor helper ready.")

**Regenerate source-bank stream descriptors and temperatures**

In [ ]:
source_bank_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"\nRegenerating source-bank table | {backbone} | seed={seed}")

        source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                logits, labels = load_source_bank_logits(
                    backbone=backbone,
                    seed=seed,
                    corruption=corruption,
                    severity=severity,
                    n_samples=4096,
                )

                desc = stream_descriptors_from_logits(logits)

                temp_fit = fit_temperature_nll(
                    logits=logits,
                    labels=labels,
                    init_temperature=source_T,
                    max_iter=100,
                    lr=0.05,
                    min_temperature=0.05,
                    max_temperature=10.0,
                )

                row = {
                    "information_regime": "source_bank_pseudo_domain",
                    "backbone": backbone,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": f"{corruption}_sev{severity}",
                    "source_T": float(source_T),

                    "source_bank_target_temperature": float(temp_fit["temperature"]),
                    "source_bank_target_log_temperature": float(temp_fit["log_temperature"]),
                    "source_bank_nll_before_ts": float(temp_fit["nll_before"]),
                    "source_bank_nll_after_ts": float(temp_fit["nll_after"]),
                    "source_bank_nll_improvement": float(temp_fit["nll_improvement"]),
                }

                row.update(desc)
                source_bank_rows.append(row)

source_bank_stream_df = pd.DataFrame(source_bank_rows)

dup_source_bank = source_bank_stream_df.duplicated(
    subset=["backbone", "seed", "corruption", "severity"]
).sum()

assert dup_source_bank == 0, f"Duplicate source-bank rows found: {dup_source_bank}"

expected_source_bank_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(source_bank_stream_df) == expected_source_bank_rows, (
    f"Expected {expected_source_bank_rows}, got {len(source_bank_stream_df)}"
)

assert source_bank_stream_df["source_bank_target_temperature"].gt(0).all()

source_bank_stream_path = DIRS["stream_methods"] / "regenerated_source_bank_stream_descriptor_table.csv"
source_bank_stream_df.to_csv(source_bank_stream_path, index=False)

print("\nSaved regenerated source-bank stream table:", source_bank_stream_path)
print("Shape:", source_bank_stream_df.shape)

display(source_bank_stream_df.head())
display(
    source_bank_stream_df
    .groupby(["backbone", "seed"], as_index=False)
    .agg(
        n_conditions=("condition", "count"),
        entropy_min=("entropy_mean", "min"),
        entropy_max=("entropy_mean", "max"),
        target_T_min=("source_bank_target_temperature", "min"),
        target_T_max=("source_bank_target_temperature", "max"),
        target_T_mean=("source_bank_target_temperature", "mean"),
    )
    .round(6)
)

In [ ]:
assert source_bank_stream_df["source_bank_target_temperature"].between(0.05, 10.0).all()
assert source_bank_stream_df["source_bank_target_log_temperature"].notna().all()
assert source_bank_stream_df["source_bank_nll_after_ts"].notna().all()
assert source_bank_stream_df["source_bank_nll_before_ts"].notna().all()

bad_temp_fits = source_bank_stream_df[
    source_bank_stream_df["source_bank_nll_after_ts"]
    > source_bank_stream_df["source_bank_nll_before_ts"] + 1e-6
]

assert len(bad_temp_fits) == 0, (
    "Some source-bank temperature fits worsened NLL:\n"
    + bad_temp_fits[
        [
            "backbone",
            "seed",
            "corruption",
            "severity",
            "source_bank_nll_before_ts",
            "source_bank_nll_after_ts",
            "source_bank_target_temperature",
        ]
    ].head().to_string(index=False)
)

**Compare regenerated source-bank table to old source-bank table**

In [ ]:
OLD_SOURCE_BANK_STREAM_PATH = INPUT_PATHS["clean_stream_root"] / "source_bank_stream_descriptor_table.csv"

if OLD_SOURCE_BANK_STREAM_PATH.exists():
    old_source_bank_df = pd.read_csv(OLD_SOURCE_BANK_STREAM_PATH)

    compare_keys = ["backbone", "seed", "corruption", "severity"]

    compare_cols = [
        "entropy_mean",
        "entropy_std",
        "top1_confidence_mean",
        "top1_confidence_std",
        "margin_mean",
        "margin_std",
        "free_energy_mean",
        "free_energy_std",
        "source_bank_target_temperature",
        "source_bank_target_log_temperature",
    ]

    old_keep = compare_keys + [c for c in compare_cols if c in old_source_bank_df.columns]
    new_keep = compare_keys + [c for c in compare_cols if c in source_bank_stream_df.columns]

    source_bank_compare = source_bank_stream_df[new_keep].merge(
        old_source_bank_df[old_keep],
        on=compare_keys,
        how="inner",
        suffixes=("_new", "_old"),
    )

    diff_rows = []

    for col in compare_cols:
        new_col = f"{col}_new"
        old_col = f"{col}_old"

        if new_col in source_bank_compare.columns and old_col in source_bank_compare.columns:
            diff = source_bank_compare[new_col] - source_bank_compare[old_col]

            diff_rows.append({
                "quantity": col,
                "mean_abs_diff": float(np.mean(np.abs(diff))),
                "max_abs_diff": float(np.max(np.abs(diff))),
                "mean_diff": float(np.mean(diff)),
            })

    source_bank_diff_summary = pd.DataFrame(diff_rows)

    source_bank_compare_path = DIRS["stream_methods"] / "source_bank_regeneration_comparison_with_old.csv"
    source_bank_diff_path = DIRS["stream_methods"] / "source_bank_regeneration_diff_summary.csv"

    source_bank_compare.to_csv(source_bank_compare_path, index=False)
    source_bank_diff_summary.to_csv(source_bank_diff_path, index=False)

    print("Saved source-bank comparison:", source_bank_compare_path)
    print("Saved source-bank diff summary:", source_bank_diff_path)

    display(source_bank_diff_summary.round(10))
else:
    print("Old source-bank stream table not found. Skipping comparison.")

**Regenerate target stream descriptors from cached target logits**

In [ ]:
target_stream_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"\nRegenerating target stream descriptors | {backbone} | seed={seed}")

        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                logits, labels = load_cached_logits(
                    backbone=backbone,
                    seed=seed,
                    corruption=corruption,
                    severity=severity,
                )

                desc = stream_descriptors_from_logits(logits)

                row = {
                    "information_regime": "unlabeled_target_stream",
                    "backbone": backbone,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": f"{corruption}_sev{severity}",
                }

                row.update(desc)
                target_stream_rows.append(row)

target_stream_df = pd.DataFrame(target_stream_rows)

dup_target_stream = target_stream_df.duplicated(
    subset=["backbone", "seed", "corruption", "severity"]
).sum()

assert dup_target_stream == 0, f"Duplicate target stream rows found: {dup_target_stream}"

expected_target_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(target_stream_df) == expected_target_rows, (
    f"Expected {expected_target_rows}, got {len(target_stream_df)}"
)

target_stream_path = DIRS["stream_methods"] / "regenerated_target_stream_descriptor_table.csv"
target_stream_df.to_csv(target_stream_path, index=False)

print("\nSaved regenerated target stream descriptors:", target_stream_path)
print("Shape:", target_stream_df.shape)

display(target_stream_df.head())
display(
    target_stream_df
    .groupby(["backbone", "seed"], as_index=False)
    .agg(
        n_conditions=("condition", "count"),
        entropy_min=("entropy_mean", "min"),
        entropy_max=("entropy_mean", "max"),
        entropy_mean=("entropy_mean", "mean"),
        confidence_mean=("top1_confidence_mean", "mean"),
        margin_mean=("margin_mean", "mean"),
    )
    .round(6)
)

**Validate descriptor consistency**

In [ ]:
REQUIRED_DESCRIPTOR_COLS = [
    "entropy_mean",
    "entropy_std",
    "entropy_q10",
    "entropy_q25",
    "entropy_q50",
    "entropy_q75",
    "entropy_q90",

    "top1_confidence_mean",
    "top1_confidence_std",
    "top1_confidence_q10",
    "top1_confidence_q25",
    "top1_confidence_q50",
    "top1_confidence_q75",
    "top1_confidence_q90",

    "margin_mean",
    "margin_std",
    "margin_q10",
    "margin_q25",
    "margin_q50",
    "margin_q75",
    "margin_q90",

    "free_energy_mean",
    "free_energy_std",
    "free_energy_q10",
    "free_energy_q25",
    "free_energy_q50",
    "free_energy_q75",
    "free_energy_q90",
]

missing_source_desc = [c for c in REQUIRED_DESCRIPTOR_COLS if c not in source_bank_stream_df.columns]
missing_target_desc = [c for c in REQUIRED_DESCRIPTOR_COLS if c not in target_stream_df.columns]

assert len(missing_source_desc) == 0, f"Missing source descriptors: {missing_source_desc}"
assert len(missing_target_desc) == 0, f"Missing target descriptors: {missing_target_desc}"

for df_name, df in [
    ("source_bank_stream_df", source_bank_stream_df),
    ("target_stream_df", target_stream_df),
]:
    assert df["entropy_mean"].notna().all(), f"{df_name}: NaN entropy"
    assert df["top1_confidence_mean"].between(0, 1).all(), f"{df_name}: confidence outside [0,1]"
    assert df["margin_mean"].between(0, 1).all(), f"{df_name}: margin outside [0,1]"
    assert (df["n_classes"] == PROTOCOL["num_classes"]).all(), f"{df_name}: wrong n_classes"

print("Descriptor consistency checks passed.")

**Stream feature sets and ridge helpers**

In [ ]:
STREAM_FEATURE_SETS = {
    "stream_entropy_mean_only": [
        "entropy_mean",
    ],

    "stream_entropy_stats": [
        "entropy_mean",
        "entropy_std",
        "entropy_q10",
        "entropy_q25",
        "entropy_q50",
        "entropy_q75",
        "entropy_q90",
    ],

    "stream_all_stats_ridge_no_class_entropy": [
        "entropy_mean",
        "entropy_std",
        "entropy_q10",
        "entropy_q25",
        "entropy_q50",
        "entropy_q75",
        "entropy_q90",

        "top1_confidence_mean",
        "top1_confidence_std",
        "top1_confidence_q10",
        "top1_confidence_q25",
        "top1_confidence_q50",
        "top1_confidence_q75",
        "top1_confidence_q90",

        "margin_mean",
        "margin_std",
        "margin_q10",
        "margin_q25",
        "margin_q50",
        "margin_q75",
        "margin_q90",

        "free_energy_mean",
        "free_energy_std",
        "free_energy_q10",
        "free_energy_q25",
        "free_energy_q50",
        "free_energy_q75",
        "free_energy_q90",
    ],

    "stream_all_stats_mlp_no_class_entropy": [
        "entropy_mean",
        "entropy_std",
        "entropy_q10",
        "entropy_q25",
        "entropy_q50",
        "entropy_q75",
        "entropy_q90",

        "top1_confidence_mean",
        "top1_confidence_std",
        "top1_confidence_q10",
        "top1_confidence_q25",
        "top1_confidence_q50",
        "top1_confidence_q75",
        "top1_confidence_q90",

        "margin_mean",
        "margin_std",
        "margin_q10",
        "margin_q25",
        "margin_q50",
        "margin_q75",
        "margin_q90",

        "free_energy_mean",
        "free_energy_std",
        "free_energy_q10",
        "free_energy_q25",
        "free_energy_q50",
        "free_energy_q75",
        "free_energy_q90",
    ],
}

feature_set_path = DIRS["configs"] / "stream_feature_sets.json"

with open(feature_set_path, "w") as f:
    json.dump(STREAM_FEATURE_SETS, f, indent=2)

print("Saved stream feature sets:", feature_set_path)

for name, features in STREAM_FEATURE_SETS.items():
    print(f"{name:45s}: {len(features)} features")


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor


def fit_stream_ridge_model(
    train_df: pd.DataFrame,
    feature_names,
    target_col: str = "source_bank_target_log_temperature",
    alpha: float = 1.0,
):
    required = list(feature_names) + [target_col]
    missing = [c for c in required if c not in train_df.columns]
    assert len(missing) == 0, f"Missing columns for stream ridge: {missing}"

    X = train_df[list(feature_names)].to_numpy(dtype=np.float64)
    y = train_df[target_col].to_numpy(dtype=np.float64)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha)),
    ])

    model.fit(X, y)
    return model


def fit_stream_mlp_model(
    train_df: pd.DataFrame,
    feature_names,
    target_col: str = "source_bank_target_log_temperature",
    random_state: int = 123,
):
    required = list(feature_names) + [target_col]
    missing = [c for c in required if c not in train_df.columns]
    assert len(missing) == 0, f"Missing columns for stream MLP: {missing}"

    X = train_df[list(feature_names)].to_numpy(dtype=np.float64)
    y = train_df[target_col].to_numpy(dtype=np.float64)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(16,),
            activation="relu",
            alpha=1e-2,
            learning_rate_init=1e-3,
            max_iter=2000,
            early_stopping=True,
            validation_fraction=0.2,
            random_state=random_state,
        )),
    ])

    model.fit(X, y)
    return model


def predict_stream_temperature_model(
    model,
    test_df: pd.DataFrame,
    feature_names,
    t_min: float = 0.8,
    t_max: float = 2.5,
):
    missing = [c for c in feature_names if c not in test_df.columns]
    assert len(missing) == 0, f"Missing columns for prediction: {missing}"

    X = test_df[list(feature_names)].to_numpy(dtype=np.float64)

    pred_logT = model.predict(X)
    pred_T = np.exp(pred_logT)
    pred_T = np.clip(pred_T, t_min, t_max)

    return pred_T, pred_logT


# Backward-compatible alias for old later cells.
def predict_stream_temperature_ridge(
    model,
    test_df: pd.DataFrame,
    feature_names,
    t_min: float = 0.8,
    t_max: float = 2.5,
):
    return predict_stream_temperature_model(
        model=model,
        test_df=test_df,
        feature_names=feature_names,
        t_min=t_min,
        t_max=t_max,
    )


print("Stream ridge and MLP helpers ready.")

**Frozen V3 helpers**

In [ ]:
from sklearn.isotonic import IsotonicRegression

def make_entropy_bins(
    train_df: pd.DataFrame,
    entropy_col: str = "entropy_mean",
    target_T_col: str = "source_bank_target_temperature",
    n_bins: int = 8,
):
    df = train_df[[entropy_col, target_T_col]].dropna().copy()
    df = df.sort_values(entropy_col).reset_index(drop=True)

    df["bin_id"] = pd.qcut(
        df[entropy_col],
        q=min(n_bins, len(df)),
        labels=False,
        duplicates="drop",
    )

    binned = (
        df.groupby("bin_id", as_index=False)
        .agg(
            entropy_bin_mean=(entropy_col, "mean"),
            entropy_bin_min=(entropy_col, "min"),
            entropy_bin_max=(entropy_col, "max"),
            target_T_bin_mean=(target_T_col, "mean"),
            target_T_bin_median=(target_T_col, "median"),
            target_T_bin_std=(target_T_col, "std"),
            n_bin=(target_T_col, "count"),
        )
        .sort_values("entropy_bin_mean")
        .reset_index(drop=True)
    )

    assert len(binned) >= 2, "Too few bins for V3 isotonic fit"

    return binned


def fit_frozen_v3_model(
    train_df: pd.DataFrame,
    source_T: float,
    config: dict = FROZEN_V3_CONFIG,
    entropy_col: str = "entropy_mean",
    target_T_col: str = "source_bank_target_temperature",
):
    required = [entropy_col, target_T_col]
    missing = [c for c in required if c not in train_df.columns]
    assert len(missing) == 0, f"Missing columns for V3: {missing}"

    n_bins = int(config["n_entropy_bins"])
    support_margin_std = float(config["support_margin_std"])
    t_max_absolute = float(config["t_max_absolute"])
    t_max_additive = float(config["t_max_additive"])
    max_extrapolation_slope = float(config["max_extrapolation_slope"])

    binned = make_entropy_bins(
        train_df=train_df,
        entropy_col=entropy_col,
        target_T_col=target_T_col,
        n_bins=n_bins,
    )

    x_bin = binned["entropy_bin_mean"].to_numpy(dtype=np.float64)
    y_bin = binned["target_T_bin_mean"].to_numpy(dtype=np.float64)

    T_min = float(source_T)
    T_max_safe = float(min(t_max_absolute, source_T + t_max_additive))

    y_safe = np.clip(y_bin, T_min, T_max_safe)

    iso = IsotonicRegression(
        increasing=True,
        out_of_bounds="clip",
        y_min=T_min,
        y_max=T_max_safe,
    )

    iso.fit(x_bin, y_safe)

    x_train = train_df[entropy_col].to_numpy(dtype=np.float64)

    x_min = float(np.min(x_train))
    x_max = float(np.max(x_train))
    x_std = float(np.std(x_train, ddof=0))

    support_low = x_min - support_margin_std * x_std
    support_high = x_max + support_margin_std * x_std

    if len(x_bin) >= 2:
        last_dx = float(x_bin[-1] - x_bin[-2])
        last_dy = float(iso.predict([x_bin[-1]])[0] - iso.predict([x_bin[-2]])[0])

        if abs(last_dx) < 1e-12:
            raw_slope = 0.0
        else:
            raw_slope = last_dy / last_dx
    else:
        raw_slope = 0.0

    extrapolation_slope = float(np.clip(raw_slope, 0.0, max_extrapolation_slope))
    T_at_train_max = float(iso.predict([x_max])[0])

    model_info = {
        "model": iso,
        "binned": binned,

        "source_T": float(source_T),
        "T_min": float(T_min),
        "T_max_safe": float(T_max_safe),

        "entropy_train_min": float(x_min),
        "entropy_train_max": float(x_max),
        "entropy_train_mean": float(np.mean(x_train)),
        "entropy_train_std": float(x_std),

        "support_low": float(support_low),
        "support_high": float(support_high),

        "T_at_train_max": float(T_at_train_max),
        "raw_extrapolation_slope": float(raw_slope),
        "extrapolation_slope": float(extrapolation_slope),

        "n_train_conditions": int(len(train_df)),
        "n_bins_used": int(len(binned)),

        "config": dict(config),
    }

    return model_info


def predict_frozen_v3_temperature(
    model_info: dict,
    entropy_value: float,
):
    entropy_value = float(entropy_value)

    iso = model_info["model"]

    source_T = float(model_info["source_T"])
    T_min = float(model_info["T_min"])
    T_max_safe = float(model_info["T_max_safe"])

    support_low = float(model_info["support_low"])
    support_high = float(model_info["support_high"])
    entropy_train_max = float(model_info["entropy_train_max"])

    lower_fallback_enabled = bool(
        model_info["config"].get("lower_fallback_to_source_ts", True)
    )

    used_lower_fallback = False
    used_high_extrapolation = False

    below_support = entropy_value < support_low
    above_support = entropy_value > support_high
    inside_support = (not below_support) and (not above_support)

    if below_support and lower_fallback_enabled:
        raw_pred_T = source_T
        pred_T = source_T
        used_lower_fallback = True

    elif above_support:
        base_T = float(model_info["T_at_train_max"])
        slope = float(model_info["extrapolation_slope"])

        raw_pred_T = base_T + slope * (entropy_value - entropy_train_max)
        pred_T = float(np.clip(raw_pred_T, T_min, T_max_safe))

        used_high_extrapolation = True

    else:
        raw_pred_T = float(iso.predict([entropy_value])[0])
        pred_T = float(np.clip(raw_pred_T, T_min, T_max_safe))

    if used_lower_fallback:
        support_region = "lower_fallback"
    elif used_high_extrapolation:
        support_region = "high_extrapolation"
    else:
        support_region = "inside_support"

    return {
        "predicted_T": float(pred_T),
        "raw_predicted_T": float(raw_pred_T),

        "support_region": support_region,

        "used_lower_fallback": bool(used_lower_fallback),
        "used_high_extrapolation": bool(used_high_extrapolation),
        "inside_support": bool(inside_support),
        "below_support": bool(below_support),
        "above_support": bool(above_support),
    }


print("Frozen V3 helpers ready.")

**V2 helper**

In [ ]:
SAFE_V2_CONFIG = {
    "method": "safe_v2_binned_isotonic",
    "n_entropy_bins": 8,
    "t_min_rule": "source_T",
    "t_max_absolute": 1.25,
    "t_max_additive": 0.20,
    "out_of_bounds": "clip",
    "fit_data": "source_bank_only",
    "target_labels_used_for_fit": False,
    "target_metrics_used_for_hyperparameter_selection": False,
    "note": "V2 is conservative binned isotonic without high-entropy extrapolation.",
}

safe_v2_config_path = DIRS["configs"] / "safe_v2_config.json"

with open(safe_v2_config_path, "w") as f:
    json.dump(SAFE_V2_CONFIG, f, indent=2)

print("Saved safe V2 config:", safe_v2_config_path)


def fit_safe_v2_model(
    train_df: pd.DataFrame,
    source_T: float,
    config: dict = SAFE_V2_CONFIG,
    entropy_col: str = "entropy_mean",
    target_T_col: str = "source_bank_target_temperature",
):
    n_bins = int(config["n_entropy_bins"])

    binned = make_entropy_bins(
        train_df=train_df,
        entropy_col=entropy_col,
        target_T_col=target_T_col,
        n_bins=n_bins,
    )

    x_bin = binned["entropy_bin_mean"].to_numpy(dtype=np.float64)
    y_bin = binned["target_T_bin_mean"].to_numpy(dtype=np.float64)

    T_min = float(source_T)
    T_max_safe = float(min(
        config["t_max_absolute"],
        source_T + config["t_max_additive"],
    ))

    y_safe = np.clip(y_bin, T_min, T_max_safe)

    iso = IsotonicRegression(
        increasing=True,
        out_of_bounds="clip",
        y_min=T_min,
        y_max=T_max_safe,
    )

    iso.fit(x_bin, y_safe)

    x_train = train_df[entropy_col].to_numpy(dtype=np.float64)

    return {
        "model": iso,
        "binned": binned,
        "source_T": float(source_T),
        "T_min": float(T_min),
        "T_max_safe": float(T_max_safe),
        "entropy_train_min": float(np.min(x_train)),
        "entropy_train_max": float(np.max(x_train)),
        "entropy_train_mean": float(np.mean(x_train)),
        "entropy_train_std": float(np.std(x_train, ddof=0)),
        "n_train_conditions": int(len(train_df)),
        "n_bins_used": int(len(binned)),
        "config": dict(config),
    }


def predict_safe_v2_temperature(model_info: dict, entropy_value: float):
    entropy_value = float(entropy_value)

    iso = model_info["model"]
    T_min = float(model_info["T_min"])
    T_max_safe = float(model_info["T_max_safe"])

    raw_pred_T = float(iso.predict([entropy_value])[0])
    pred_T = float(np.clip(raw_pred_T, T_min, T_max_safe))

    return {
        "predicted_T": pred_T,
        "raw_predicted_T": raw_pred_T,
        "support_region": "clipped_isotonic_support",
        "used_lower_fallback": False,
        "used_high_extrapolation": False,
        "inside_support": True,
        "below_support": False,
        "above_support": False,
    }


print("Safe V2 helpers ready.")

**Fit Safe V2 and predict target temperatures**

In [ ]:
safe_v2_model_rows = []
safe_v2_prediction_rows = []
safe_v2_bin_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"Safe V2 fit/predict | {backbone} | seed={seed}")

        source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

        train_df = source_bank_stream_df[
            (source_bank_stream_df["backbone"] == backbone) &
            (source_bank_stream_df["seed"] == seed)
        ].copy()

        target_df = target_stream_df[
            (target_stream_df["backbone"] == backbone) &
            (target_stream_df["seed"] == seed)
        ].copy()

        assert len(train_df) == 50
        assert len(target_df) == 50

        model_info = fit_safe_v2_model(
            train_df=train_df,
            source_T=source_T,
            config=SAFE_V2_CONFIG,
        )

        safe_v2_model_rows.append({
            "backbone": backbone,
            "seed": int(seed),
            "source_T": float(source_T),
            "T_min": model_info["T_min"],
            "T_max_safe": model_info["T_max_safe"],
            "entropy_train_min": model_info["entropy_train_min"],
            "entropy_train_max": model_info["entropy_train_max"],
            "entropy_train_mean": model_info["entropy_train_mean"],
            "entropy_train_std": model_info["entropy_train_std"],
            "n_train_conditions": model_info["n_train_conditions"],
            "n_bins_used": model_info["n_bins_used"],
        })

        binned = model_info["binned"].copy()
        binned["backbone"] = backbone
        binned["seed"] = int(seed)
        binned["source_T"] = float(source_T)
        binned["T_max_safe"] = model_info["T_max_safe"]
        safe_v2_bin_rows.append(binned)

        for _, row in target_df.iterrows():
            entropy_mean = float(row["entropy_mean"])

            pred = predict_safe_v2_temperature(
                model_info=model_info,
                entropy_value=entropy_mean,
            )

            safe_v2_prediction_rows.append({
                "information_regime": "unlabeled_target_stream_safe_v2",
                "backbone": backbone,
                "seed": int(seed),
                "method": SAFE_V2_CONFIG["method"],
                "corruption": row["corruption"],
                "severity": int(row["severity"]),
                "condition": row["condition"],
                "entropy_mean": entropy_mean,
                "predicted_T": float(pred["predicted_T"]),
                "raw_predicted_T": float(pred["raw_predicted_T"]),
                "source_T": float(source_T),

                "support_region": pred["support_region"],
                "used_lower_fallback": bool(pred["used_lower_fallback"]),
                "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                "inside_support": bool(pred["inside_support"]),
                "below_support": bool(pred["below_support"]),
                "above_support": bool(pred["above_support"]),
            })

safe_v2_model_info_df = pd.DataFrame(safe_v2_model_rows)
safe_v2_predictions_df = pd.DataFrame(safe_v2_prediction_rows)
safe_v2_bins_df = pd.concat(safe_v2_bin_rows, ignore_index=True)

expected_v2_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(safe_v2_predictions_df) == expected_v2_rows
assert safe_v2_predictions_df["predicted_T"].gt(0).all()
assert (
    safe_v2_predictions_df["predicted_T"]
    >= safe_v2_predictions_df["source_T"] - 1e-12
).all(), "V2 predicted T below source_T"

safe_v2_model_info_path = DIRS["safe_v2"] / "safe_v2_model_info_regenerated_source_bank.csv"
safe_v2_prediction_path = DIRS["safe_v2"] / "safe_v2_target_temperature_predictions_unlabeled_only.csv"
safe_v2_bins_path = DIRS["safe_v2"] / "safe_v2_source_bank_entropy_bins.csv"

safe_v2_model_info_df.to_csv(safe_v2_model_info_path, index=False)
safe_v2_predictions_df.to_csv(safe_v2_prediction_path, index=False)
safe_v2_bins_df.to_csv(safe_v2_bins_path, index=False)

print("Saved V2 model info:", safe_v2_model_info_path)
print("Saved V2 target predictions:", safe_v2_prediction_path)
print("Saved V2 bins:", safe_v2_bins_path)

display(safe_v2_model_info_df.round(6))
display(safe_v2_predictions_df.head())

**Fit frozen V3 from regenerated source-bank table and predict target temperatures**

In [ ]:
# Fit frozen V3 on regenerated source-bank table
# and predict target temperatures from regenerated target descriptors
# No target labels are used here.

frozen_v3_model_rows = []
frozen_v3_prediction_rows = []
frozen_v3_bin_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"Frozen V3 fit/predict | {backbone} | seed={seed}")

        source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

        train_df = source_bank_stream_df[
            (source_bank_stream_df["backbone"] == backbone) &
            (source_bank_stream_df["seed"] == seed)
        ].copy()

        target_df = target_stream_df[
            (target_stream_df["backbone"] == backbone) &
            (target_stream_df["seed"] == seed)
        ].copy()

        assert len(train_df) == 50
        assert len(target_df) == 50

        model_info = fit_frozen_v3_model(
            train_df=train_df,
            source_T=source_T,
            config=FROZEN_V3_CONFIG,
        )

        frozen_v3_model_rows.append({
            "backbone": backbone,
            "seed": int(seed),
            "source_T": float(source_T),
            "T_min": model_info["T_min"],
            "T_max_safe": model_info["T_max_safe"],
            "entropy_train_min": model_info["entropy_train_min"],
            "entropy_train_max": model_info["entropy_train_max"],
            "entropy_train_mean": model_info["entropy_train_mean"],
            "entropy_train_std": model_info["entropy_train_std"],
            "support_low": model_info["support_low"],
            "support_high": model_info["support_high"],
            "T_at_train_max": model_info["T_at_train_max"],
            "raw_extrapolation_slope": model_info["raw_extrapolation_slope"],
            "extrapolation_slope": model_info["extrapolation_slope"],
            "n_train_conditions": model_info["n_train_conditions"],
            "n_bins_used": model_info["n_bins_used"],
        })

        binned = model_info["binned"].copy()
        binned["backbone"] = backbone
        binned["seed"] = int(seed)
        binned["source_T"] = float(source_T)
        binned["T_max_safe"] = model_info["T_max_safe"]
        frozen_v3_bin_rows.append(binned)

        for _, row in target_df.iterrows():
            entropy_mean = float(row["entropy_mean"])

            pred = predict_frozen_v3_temperature(
                model_info=model_info,
                entropy_value=entropy_mean,
            )

            frozen_v3_prediction_rows.append({
                "information_regime": "unlabeled_target_stream_safe_v3_frozen",
                "backbone": backbone,
                "seed": int(seed),
                "method": FROZEN_V3_CONFIG["method"],
                "corruption": row["corruption"],
                "severity": int(row["severity"]),
                "condition": row["condition"],
                "entropy_mean": entropy_mean,
                "predicted_T": float(pred["predicted_T"]),
                "raw_predicted_T": float(pred["raw_predicted_T"]),
                "source_T": float(source_T),

                "support_region": pred["support_region"],
                "used_lower_fallback": bool(pred["used_lower_fallback"]),
                "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                "inside_support": bool(pred["inside_support"]),
                "below_support": bool(pred["below_support"]),
                "above_support": bool(pred["above_support"]),
            })

frozen_v3_model_info_df = pd.DataFrame(frozen_v3_model_rows)
frozen_v3_predictions_df = pd.DataFrame(frozen_v3_prediction_rows)

expected_v3_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(frozen_v3_predictions_df) == expected_v3_rows, (
    f"Expected {expected_v3_rows} V3 predictions, got {len(frozen_v3_predictions_df)}"
)

assert frozen_v3_predictions_df["predicted_T"].gt(0).all(), "V3 predicted non-positive T"

assert (
    frozen_v3_predictions_df["predicted_T"]
    >= frozen_v3_predictions_df["source_T"] - 1e-12
).all(), "V3 predicted T below source_T"

assert (
    frozen_v3_predictions_df["support_region"]
    .isin(["inside_support", "lower_fallback", "high_extrapolation"])
    .all()
), "Unexpected support_region value"

assert (
    frozen_v3_predictions_df["used_lower_fallback"]
    == (frozen_v3_predictions_df["support_region"] == "lower_fallback")
).all(), "Mismatch in lower fallback support labels"

assert (
    frozen_v3_predictions_df["used_high_extrapolation"]
    == (frozen_v3_predictions_df["support_region"] == "high_extrapolation")
).all(), "Mismatch in high extrapolation support labels"

frozen_v3_bins_df = pd.concat(frozen_v3_bin_rows, ignore_index=True)

model_info_path = DIRS["safe_v3"] / "frozen_v3_model_info_regenerated_source_bank.csv"
prediction_path = DIRS["safe_v3"] / "frozen_v3_target_temperature_predictions_unlabeled_only.csv"
bins_path = DIRS["safe_v3"] / "frozen_v3_source_bank_entropy_bins.csv"

frozen_v3_model_info_df.to_csv(model_info_path, index=False)
frozen_v3_predictions_df.to_csv(prediction_path, index=False)
frozen_v3_bins_df.to_csv(bins_path, index=False)

print("Saved model info:", model_info_path)
print("Saved target predictions:", prediction_path)
print("Saved entropy bins:", bins_path)

display(frozen_v3_model_info_df.round(6))
display(frozen_v3_predictions_df.head())

**Support/extrapolation preview**

In [ ]:
support_preview_by_backbone = (
    frozen_v3_predictions_df
    .groupby(["backbone", "support_region"], as_index=False)
    .agg(
        count=("predicted_T", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        min_predicted_T=("predicted_T", "min"),
        max_predicted_T=("predicted_T", "max"),
        mean_entropy=("entropy_mean", "mean"),
    )
)

support_preview_by_backbone["fraction"] = (
    support_preview_by_backbone["count"] /
    support_preview_by_backbone.groupby("backbone")["count"].transform("sum")
)

support_preview_by_severity = (
    frozen_v3_predictions_df
    .groupby(["backbone", "severity", "support_region"], as_index=False)
    .agg(
        count=("predicted_T", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        mean_entropy=("entropy_mean", "mean"),
    )
)

support_preview_by_severity["fraction"] = (
    support_preview_by_severity["count"] /
    support_preview_by_severity.groupby(["backbone", "severity"])["count"].transform("sum")
)

support_preview_by_corruption = (
    frozen_v3_predictions_df
    .groupby(["backbone", "corruption", "support_region"], as_index=False)
    .agg(
        count=("predicted_T", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        mean_entropy=("entropy_mean", "mean"),
    )
)

support_preview_by_corruption["fraction"] = (
    support_preview_by_corruption["count"] /
    support_preview_by_corruption.groupby(["backbone", "corruption"])["count"].transform("sum")
)

support_backbone_path = DIRS["support_extrapolation"] / "frozen_v3_support_preview_by_backbone.csv"
support_severity_path = DIRS["support_extrapolation"] / "frozen_v3_support_preview_by_severity.csv"
support_corruption_path = DIRS["support_extrapolation"] / "frozen_v3_support_preview_by_corruption.csv"

support_preview_by_backbone.to_csv(support_backbone_path, index=False)
support_preview_by_severity.to_csv(support_severity_path, index=False)
support_preview_by_corruption.to_csv(support_corruption_path, index=False)

print("Saved:", support_backbone_path)
print("Saved:", support_severity_path)
print("Saved:", support_corruption_path)

print("\nSupport preview by backbone:")
display(support_preview_by_backbone.round(6))

print("\nSupport preview by severity:")
display(support_preview_by_severity.round(6).head(30))

print("\nSupport preview by corruption:")
display(support_preview_by_corruption.round(6).head(30))

In [ ]:
stage2_note = {
    "stage": "stage_2_regenerate_source_bank_target_stream_v2_v3_predictions",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "main_outputs": {
        "source_bank_logits_manifest": str(source_bank_manifest_path),
        "regenerated_source_bank_stream_descriptor_table": str(source_bank_stream_path),
        "regenerated_target_stream_descriptor_table": str(target_stream_path),
        "stream_feature_sets": str(feature_set_path),

        "safe_v2_config": str(safe_v2_config_path),
        "safe_v2_model_info": str(safe_v2_model_info_path),
        "safe_v2_target_temperature_predictions": str(safe_v2_prediction_path),
        "safe_v2_entropy_bins": str(safe_v2_bins_path),

        "frozen_v3_model_info": str(model_info_path),
        "frozen_v3_target_temperature_predictions": str(prediction_path),
        "frozen_v3_entropy_bins": str(bins_path),

        "support_preview_by_backbone": str(support_backbone_path),
        "support_preview_by_severity": str(support_severity_path),
        "support_preview_by_corruption": str(support_corruption_path),
    },

    "checks": {
        "source_bank_logits_regenerated_from_cached_pt_files": True,
        "source_bank_descriptors_regenerated": True,
        "source_bank_target_temperatures_regenerated": True,
        "target_stream_descriptors_regenerated": True,
        "std_uses_ddof_0_explicitly": True,

        "target_labels_used_for_target_stream_descriptors": False,
        "target_labels_used_for_v2_fit": False,
        "target_labels_used_for_v3_fit": False,

        "v2_fit_uses_regenerated_source_bank_only": True,
        "v3_fit_uses_regenerated_source_bank_only": True,
        "v3_config_frozen": True,

        "ridge_helpers_available": True,
        "mlp_helper_available": True,
    },

    "important_note": (
        "Stage 2 regenerates source-bank descriptors and source-bank fitted temperatures "
        "from cached source-bank logits. It also regenerates target stream descriptors from "
        "cached target logits. V2 and frozen V3 are fitted only on regenerated source-bank "
        "conditions and predict target temperatures using unlabeled target stream entropy."
    ),
}

stage2_note_path = DIRS["logs"] / "stage2_regenerated_stream_descriptors_v2_v3_completion.json"

with open(stage2_note_path, "w") as f:
    json.dump(stage2_note, f, indent=2)

print("Saved Stage 2 completion note:", stage2_note_path)
print(json.dumps(stage2_note, indent=2))

**Stage 3**

**Evaluation helper from predicted temperatures**

In [ ]:
def evaluate_temperature_on_target_condition(
    backbone: str,
    seed: int,
    corruption: str,
    severity: int,
    method: str,
    information_regime: str,
    temperature: float,
    split_name: str = "main_all_source_bank_train",
    heldout_type: str = "none",
    heldout_value: str = "none",
):
    logits, labels = load_cached_logits(
        backbone=backbone,
        seed=seed,
        corruption=corruption,
        severity=severity,
    )

    scaled_logits = apply_temperature(logits, float(temperature))
    met = metrics_from_logits(scaled_logits, labels)

    rows = []

    for metric_name, value in met.items():
        rows.append({
            "split_name": split_name,
            "heldout_type": heldout_type,
            "heldout_value": str(heldout_value),
            "information_regime": information_regime,
            "backbone": backbone,
            "seed": int(seed),
            "method": method,
            "corruption": corruption,
            "severity": int(severity),
            "condition": f"{corruption}_sev{severity}",
            "temperature": float(temperature),
            "metric": metric_name,
            "value": float(value),
        })

    del logits, labels
    return rows


print("Stage 3 evaluation helper ready.")

**generic safe-method evaluator**

In [ ]:
def evaluate_prediction_table_on_target(
    predictions_df: pd.DataFrame,
    method_label: str,
    output_dir: Path,
    eval_filename: str,
    wide_filename: str,
    split_name: str = "main_all_source_bank_train",
    heldout_type: str = "none",
    heldout_value: str = "none",
):
    required_pred_cols = [
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
        "predicted_T",
        "source_T",
        "support_region",
        "used_lower_fallback",
        "used_high_extrapolation",
    ]

    missing_pred_cols = [c for c in required_pred_cols if c not in predictions_df.columns]
    assert len(missing_pred_cols) == 0, (
        f"{method_label}: missing prediction columns: {missing_pred_cols}"
    )

    eval_rows = []

    for _, row in predictions_df.iterrows():
        eval_rows.extend(
            evaluate_temperature_on_target_condition(
                backbone=row["backbone"],
                seed=int(row["seed"]),
                corruption=row["corruption"],
                severity=int(row["severity"]),
                method=row["method"],
                information_regime=row["information_regime"],
                temperature=float(row["predicted_T"]),
                split_name=split_name,
                heldout_type=heldout_type,
                heldout_value=heldout_value,
            )
        )

    eval_long = pd.DataFrame(eval_rows)

    expected_eval_rows = (
        len(PROTOCOL["backbones"])
        * len(PROTOCOL["seeds"])
        * len(PROTOCOL["corruptions"])
        * len(PROTOCOL["severities"])
        * len(PROTOCOL["metrics"])
    )

    assert len(eval_long) == expected_eval_rows, (
        f"{method_label}: expected {expected_eval_rows} eval rows, got {len(eval_long)}"
    )

    assert eval_long["value"].notna().all(), f"{method_label}: NaN in evaluation values"

    eval_path = output_dir / eval_filename
    eval_long.to_csv(eval_path, index=False)

    condition_wide = (
        eval_long
        .pivot_table(
            index=[
                "split_name",
                "heldout_type",
                "heldout_value",
                "information_regime",
                "backbone",
                "seed",
                "method",
                "corruption",
                "severity",
                "condition",
                "temperature",
            ],
            columns="metric",
            values="value",
            aggfunc="mean",
        )
        .reset_index()
    )

    condition_wide.columns.name = None

    required_metric_cols = [
        "accuracy",
        "mean_confidence",
        "signed_gap",
        "abs_signed_gap",
        "nll",
        "brier",
        "ece15",
        "aece15",
    ]

    missing_metric_cols = [c for c in required_metric_cols if c not in condition_wide.columns]
    assert len(missing_metric_cols) == 0, (
        f"{method_label}: missing wide metric columns: {missing_metric_cols}"
    )

    assert condition_wide["ece15"].between(0, 1).all(), f"{method_label}: ECE outside [0,1]"
    assert condition_wide["aece15"].between(0, 1).all(), f"{method_label}: AECE outside [0,1]"

    wide_path = output_dir / wide_filename
    condition_wide.to_csv(wide_path, index=False)

    print(f"Saved {method_label} eval long:", eval_path)
    print(f"Saved {method_label} condition-wide:", wide_path)

    return eval_long, condition_wide, eval_path, wide_path


print("Generic safe-method evaluator ready.")

**Evaluate Safe V2 main setting**

In [ ]:
safe_v2_eval_long, safe_v2_condition_wide, safe_v2_eval_path, safe_v2_wide_path = (
    evaluate_prediction_table_on_target(
        predictions_df=safe_v2_predictions_df,
        method_label="safe_v2",
        output_dir=DIRS["safe_v2"],
        eval_filename="safe_v2_main_eval_long.csv",
        wide_filename="safe_v2_main_condition_metric_wide.csv",
        split_name="main_all_source_bank_train",
        heldout_type="none",
        heldout_value="none",
    )
)

display(safe_v2_eval_long.head())
display(safe_v2_condition_wide.head())

**Evaluate frozen V3 main setting**

In [ ]:
frozen_v3_eval_long, frozen_v3_condition_wide, v3_eval_path, v3_wide_path = (
    evaluate_prediction_table_on_target(
        predictions_df=frozen_v3_predictions_df,
        method_label="frozen_v3",
        output_dir=DIRS["safe_v3"],
        eval_filename="frozen_v3_main_eval_long.csv",
        wide_filename="frozen_v3_main_condition_metric_wide.csv",
        split_name="main_all_source_bank_train",
        heldout_type="none",
        heldout_value="none",
    )
)

display(frozen_v3_eval_long.head())
display(frozen_v3_condition_wide.head())

**Generic delta helper for safe methods**

In [ ]:
required_metric_cols = [
    "accuracy",
    "mean_confidence",
    "signed_gap",
    "abs_signed_gap",
    "nll",
    "brier",
    "ece15",
    "aece15",
]

def add_source_ts_deltas_and_support_metadata(
    condition_wide: pd.DataFrame,
    predictions_df: pd.DataFrame,
    method_label: str,
):
    source_ts_wide = corrected_condition_metric_wide[
        corrected_condition_metric_wide["method"] == "source_global_ts"
    ].copy()

    source_ts_keep_cols = [
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
    ] + required_metric_cols

    source_ts_baseline = source_ts_wide[source_ts_keep_cols].rename(
        columns={m: f"source_ts_{m}" for m in required_metric_cols}
    )

    dup_source_ts = source_ts_baseline.duplicated(
        subset=["backbone", "seed", "corruption", "severity", "condition"]
    ).sum()

    assert dup_source_ts == 0, f"{method_label}: duplicate source TS baseline rows: {dup_source_ts}"

    merge_keys = ["backbone", "seed", "corruption", "severity", "condition"]

    delta_wide = condition_wide.merge(
        source_ts_baseline,
        on=merge_keys,
        how="left",
    )

    assert len(delta_wide) == len(condition_wide), (
        f"{method_label}: merge with source TS changed row count"
    )

    assert delta_wide["source_ts_nll"].notna().all(), (
        f"{method_label}: missing source TS baseline after merge"
    )

    for metric in required_metric_cols:
        delta_wide[f"delta_{metric}_vs_source_ts"] = (
            delta_wide[metric] - delta_wide[f"source_ts_{metric}"]
        )

    assert (delta_wide["delta_accuracy_vs_source_ts"].abs() < 1e-10).all(), (
        f"{method_label}: temperature scaling changed accuracy relative to source TS"
    )

    support_meta_cols = [
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
        "entropy_mean",
        "predicted_T",
        "raw_predicted_T",
        "source_T",
        "support_region",
        "used_lower_fallback",
        "used_high_extrapolation",
        "inside_support",
        "below_support",
        "above_support",
    ]

    support_meta = predictions_df[support_meta_cols].copy()

    dup_support_meta = support_meta.duplicated(
        subset=["backbone", "seed", "corruption", "severity", "condition"]
    ).sum()

    assert dup_support_meta == 0, (
        f"{method_label}: duplicate support metadata rows: {dup_support_meta}"
    )

    delta_wide = delta_wide.merge(
        support_meta,
        on=merge_keys,
        how="left",
        suffixes=("", "_pred"),
    )

    assert len(delta_wide) == len(condition_wide), (
        f"{method_label}: merge with support metadata changed row count"
    )

    assert delta_wide["support_region"].notna().all(), (
        f"{method_label}: missing support metadata"
    )

    assert np.allclose(
        delta_wide["temperature"].to_numpy(dtype=float),
        delta_wide["predicted_T"].to_numpy(dtype=float),
        atol=1e-10,
    ), f"{method_label}: mismatch between evaluated temperature and predicted_T"

    assert delta_wide["ece15"].between(0, 1).all()
    assert delta_wide["aece15"].between(0, 1).all()
    assert delta_wide["source_ts_ece15"].between(0, 1).all()
    assert delta_wide["source_ts_aece15"].between(0, 1).all()

    return delta_wide


print("Safe-method delta helper ready.")

**Safe V2 deltas and summaries**

In [ ]:
safe_v2_delta_wide = add_source_ts_deltas_and_support_metadata(
    condition_wide=safe_v2_condition_wide,
    predictions_df=safe_v2_predictions_df,
    method_label="safe_v2",
)

safe_v2_delta_path = DIRS["safe_v2"] / "safe_v2_main_delta_vs_source_ts_wide.csv"
safe_v2_delta_wide.to_csv(safe_v2_delta_path, index=False)

safe_v2_main_delta_summary = (
    safe_v2_delta_wide
    .groupby(["split_name", "information_regime", "backbone", "method"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),

        source_ts_nll_mean=("source_ts_nll", "mean"),
        source_ts_brier_mean=("source_ts_brier", "mean"),
        source_ts_ece15_mean=("source_ts_ece15", "mean"),
        source_ts_aece15_mean=("source_ts_aece15", "mean"),
        source_ts_signed_gap_mean=("source_ts_signed_gap", "mean"),
        source_ts_abs_signed_gap_mean=("source_ts_abs_signed_gap", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        predicted_T_min=("predicted_T", "min"),
        predicted_T_max=("predicted_T", "max"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),

        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

assert safe_v2_main_delta_summary["ece15_mean"].between(0, 1).all()
assert safe_v2_main_delta_summary["source_ts_ece15_mean"].between(0, 1).all()

safe_v2_summary_path = DIRS["safe_v2"] / "safe_v2_main_delta_summary.csv"
safe_v2_main_delta_summary.to_csv(safe_v2_summary_path, index=False)

print("Saved Safe V2 delta wide:", safe_v2_delta_path)
print("Saved Safe V2 main summary:", safe_v2_summary_path)
display(safe_v2_main_delta_summary.round(6))

**Add source TS baseline and compute deltas**

In [ ]:
source_ts_wide = corrected_condition_metric_wide[
    corrected_condition_metric_wide["method"] == "source_global_ts"
].copy()

source_ts_keep_cols = [
    "backbone",
    "seed",
    "corruption",
    "severity",
    "condition",
] + required_metric_cols

source_ts_baseline = source_ts_wide[source_ts_keep_cols].rename(
    columns={m: f"source_ts_{m}" for m in required_metric_cols}
)

dup_source_ts = source_ts_baseline.duplicated(
    subset=["backbone", "seed", "corruption", "severity", "condition"]
).sum()

assert dup_source_ts == 0, f"Duplicate source TS baseline rows: {dup_source_ts}"

v3_merge_keys = ["backbone", "seed", "corruption", "severity", "condition"]

frozen_v3_delta_wide = frozen_v3_condition_wide.merge(
    source_ts_baseline,
    on=v3_merge_keys,
    how="left",
)

assert len(frozen_v3_delta_wide) == len(frozen_v3_condition_wide), (
    "Merge with source TS changed row count."
)

assert frozen_v3_delta_wide["source_ts_nll"].notna().all(), (
    "Missing source TS baseline after merge"
)

for metric in required_metric_cols:
    frozen_v3_delta_wide[f"delta_{metric}_vs_source_ts"] = (
        frozen_v3_delta_wide[metric] - frozen_v3_delta_wide[f"source_ts_{metric}"]
    )

assert (frozen_v3_delta_wide["delta_accuracy_vs_source_ts"].abs() < 1e-10).all(), (
    "V3 changed accuracy relative to source TS. This should not happen because both "
    "are positive temperature scalings and preserve argmax."
)

support_meta_cols = [
    "backbone",
    "seed",
    "corruption",
    "severity",
    "condition",
    "entropy_mean",
    "predicted_T",
    "raw_predicted_T",
    "source_T",
    "support_region",
    "used_lower_fallback",
    "used_high_extrapolation",
    "inside_support",
    "below_support",
    "above_support",
]

support_meta = frozen_v3_predictions_df[support_meta_cols].copy()

dup_support_meta = support_meta.duplicated(
    subset=["backbone", "seed", "corruption", "severity", "condition"]
).sum()

assert dup_support_meta == 0, f"Duplicate support metadata rows: {dup_support_meta}"

frozen_v3_delta_wide = frozen_v3_delta_wide.merge(
    support_meta,
    on=v3_merge_keys,
    how="left",
    suffixes=("", "_pred"),
)

assert len(frozen_v3_delta_wide) == len(frozen_v3_condition_wide), (
    "Merge with support metadata changed row count."
)

assert frozen_v3_delta_wide["support_region"].notna().all(), (
    "Missing support metadata"
)

assert np.allclose(
    frozen_v3_delta_wide["temperature"].to_numpy(dtype=float),
    frozen_v3_delta_wide["predicted_T"].to_numpy(dtype=float),
    atol=1e-10,
), "Mismatch between evaluated temperature and predicted_T"

assert frozen_v3_delta_wide["ece15"].between(0, 1).all()
assert frozen_v3_delta_wide["aece15"].between(0, 1).all()
assert frozen_v3_delta_wide["source_ts_ece15"].between(0, 1).all()
assert frozen_v3_delta_wide["source_ts_aece15"].between(0, 1).all()

v3_delta_path = DIRS["safe_v3"] / "frozen_v3_main_delta_vs_source_ts_wide.csv"
frozen_v3_delta_wide.to_csv(v3_delta_path, index=False)

print("Saved frozen V3 delta wide:", v3_delta_path)
print("Shape:", frozen_v3_delta_wide.shape)

display(frozen_v3_delta_wide.head())

**V3 summary table**

In [ ]:
frozen_v3_main_delta_summary = (
    frozen_v3_delta_wide
    .groupby(["split_name", "information_regime", "backbone", "method"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),

        source_ts_nll_mean=("source_ts_nll", "mean"),
        source_ts_brier_mean=("source_ts_brier", "mean"),
        source_ts_ece15_mean=("source_ts_ece15", "mean"),
        source_ts_aece15_mean=("source_ts_aece15", "mean"),
        source_ts_signed_gap_mean=("source_ts_signed_gap", "mean"),
        source_ts_abs_signed_gap_mean=("source_ts_abs_signed_gap", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        predicted_T_min=("predicted_T", "min"),
        predicted_T_max=("predicted_T", "max"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),

        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

assert frozen_v3_main_delta_summary["ece15_mean"].between(0, 1).all()
assert frozen_v3_main_delta_summary["source_ts_ece15_mean"].between(0, 1).all()

v3_summary_path = DIRS["safe_v3"] / "frozen_v3_main_delta_summary.csv"
frozen_v3_main_delta_summary.to_csv(v3_summary_path, index=False)

print("Saved frozen V3 main summary:", v3_summary_path)

display(frozen_v3_main_delta_summary.round(6))

**V3 summary by severity**

In [ ]:
frozen_v3_by_severity = (
    frozen_v3_delta_wide
    .groupby(["backbone", "severity"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        source_T_mean=("source_T", "mean"),
        entropy_mean=("entropy_mean", "mean"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),

        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

v3_by_severity_path = DIRS["safe_v3"] / "frozen_v3_main_delta_by_severity.csv"
frozen_v3_by_severity.to_csv(v3_by_severity_path, index=False)

print("Saved frozen V3 by severity:", v3_by_severity_path)
display(frozen_v3_by_severity.round(6))

**V3 summary by corruption**

In [ ]:
frozen_v3_by_corruption = (
    frozen_v3_delta_wide
    .groupby(["backbone", "corruption"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        source_T_mean=("source_T", "mean"),
        entropy_mean=("entropy_mean", "mean"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),

        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

v3_by_corruption_path = DIRS["safe_v3"] / "frozen_v3_main_delta_by_corruption.csv"
frozen_v3_by_corruption.to_csv(v3_by_corruption_path, index=False)

print("Saved frozen V3 by corruption:", v3_by_corruption_path)

display(
    frozen_v3_by_corruption
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**V3 harm/safety analysis**

In [ ]:
frozen_v3_harm_summary = (
    frozen_v3_delta_wide
    .groupby(["backbone"], as_index=False)
    .agg(
        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        median_delta_nll=("delta_nll_vs_source_ts", "median"),
        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
        improved_conditions=("delta_nll_vs_source_ts", lambda x: int((x < -1e-12).sum())),
        harmed_conditions=("delta_nll_vs_source_ts", lambda x: int((x > 1e-12).sum())),
        near_zero_conditions=("delta_nll_vs_source_ts", lambda x: int((np.abs(x) <= 1e-12).sum())),
        n_conditions=("delta_nll_vs_source_ts", "count"),

        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),
    )
)

frozen_v3_harm_summary["improved_fraction"] = (
    frozen_v3_harm_summary["improved_conditions"] /
    frozen_v3_harm_summary["n_conditions"]
)

frozen_v3_harm_summary["harmed_fraction"] = (
    frozen_v3_harm_summary["harmed_conditions"] /
    frozen_v3_harm_summary["n_conditions"]
)

v3_harm_path = DIRS["safe_v3"] / "frozen_v3_main_harm_summary.csv"
frozen_v3_harm_summary.to_csv(v3_harm_path, index=False)

print("Saved frozen V3 harm summary:", v3_harm_path)
display(frozen_v3_harm_summary.round(6))

**Full support/extrapolation table with metric deltas**

In [ ]:
support_extrapolation_by_backbone = (
    frozen_v3_delta_wide
    .groupby(["backbone", "support_region"], as_index=False)
    .agg(
        count=("delta_nll_vs_source_ts", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        mean_source_T=("source_T", "mean"),
        mean_entropy=("entropy_mean", "mean"),

        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
    )
)

support_extrapolation_by_backbone["fraction"] = (
    support_extrapolation_by_backbone["count"] /
    support_extrapolation_by_backbone.groupby("backbone")["count"].transform("sum")
)

frac_check = support_extrapolation_by_backbone.groupby("backbone")["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), (
    "Support fractions by backbone do not sum to 1"
)


support_extrapolation_by_severity = (
    frozen_v3_delta_wide
    .groupby(["backbone", "severity", "support_region"], as_index=False)
    .agg(
        count=("delta_nll_vs_source_ts", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        mean_source_T=("source_T", "mean"),
        mean_entropy=("entropy_mean", "mean"),

        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
    )
)

support_extrapolation_by_severity["fraction"] = (
    support_extrapolation_by_severity["count"] /
    support_extrapolation_by_severity.groupby(["backbone", "severity"])["count"].transform("sum")
)

frac_check = support_extrapolation_by_severity.groupby(["backbone", "severity"])["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), (
    "Support fractions by severity do not sum to 1"
)


support_extrapolation_by_corruption = (
    frozen_v3_delta_wide
    .groupby(["backbone", "corruption", "support_region"], as_index=False)
    .agg(
        count=("delta_nll_vs_source_ts", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        mean_source_T=("source_T", "mean"),
        mean_entropy=("entropy_mean", "mean"),

        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
    )
)

support_extrapolation_by_corruption["fraction"] = (
    support_extrapolation_by_corruption["count"] /
    support_extrapolation_by_corruption.groupby(["backbone", "corruption"])["count"].transform("sum")
)

frac_check = support_extrapolation_by_corruption.groupby(["backbone", "corruption"])["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), (
    "Support fractions by corruption do not sum to 1"
)


support_extrapolation_full = (
    frozen_v3_delta_wide
    .groupby(["backbone", "corruption", "severity", "support_region"], as_index=False)
    .agg(
        count=("delta_nll_vs_source_ts", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        mean_source_T=("source_T", "mean"),
        mean_entropy=("entropy_mean", "mean"),

        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
    )
)

support_extrapolation_full["fraction"] = (
    support_extrapolation_full["count"] /
    support_extrapolation_full.groupby(["backbone", "corruption", "severity"])["count"].transform("sum")
)

frac_check = support_extrapolation_full.groupby(["backbone", "corruption", "severity"])["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), (
    "Support fractions by corruption/severity do not sum to 1"
)


support_backbone_eval_path = DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_backbone.csv"
support_severity_eval_path = DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_severity.csv"
support_corruption_eval_path = DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_corruption.csv"
support_full_eval_path = DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_backbone_corruption_severity.csv"

support_extrapolation_by_backbone.to_csv(support_backbone_eval_path, index=False)
support_extrapolation_by_severity.to_csv(support_severity_eval_path, index=False)
support_extrapolation_by_corruption.to_csv(support_corruption_eval_path, index=False)
support_extrapolation_full.to_csv(support_full_eval_path, index=False)

print("Saved support/extrapolation eval tables:")
print(support_backbone_eval_path)
print(support_severity_eval_path)
print(support_corruption_eval_path)
print(support_full_eval_path)

print("\nBy backbone:")
display(support_extrapolation_by_backbone.round(6))

print("\nBy severity:")
display(support_extrapolation_by_severity.round(6).head(40))

print("\nBy corruption:")
display(support_extrapolation_by_corruption.round(6).head(40))

print("\nFull backbone/corruption/severity table:")
display(support_extrapolation_full.round(6).head(40))

**Bootstrap confidence intervals for frozen V3**

In [ ]:
DELTA_COLS = [
    "delta_nll_vs_source_ts",
    "delta_brier_vs_source_ts",
    "delta_ece15_vs_source_ts",
    "delta_aece15_vs_source_ts",
    "delta_signed_gap_vs_source_ts",
    "delta_abs_signed_gap_vs_source_ts",
]

def bootstrap_mean_ci(values, n_boot=5000, seed=123, ci_low=2.5, ci_high=97.5):
    values = np.asarray(values, dtype=np.float64)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return {
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n": 0,
        }

    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.empty(n_boot, dtype=np.float64)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[i] = values[idx].mean()

    return {
        "mean": float(values.mean()),
        "ci_low": float(np.percentile(boot_means, ci_low)),
        "ci_high": float(np.percentile(boot_means, ci_high)),
        "n": int(n),
    }


def make_ci_summary(df, group_cols, value_cols, seed_offset=0):
    rows = []

    for group_key, group in df.groupby(group_cols):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        base = dict(zip(group_cols, group_key))

        for col in value_cols:
            ci = bootstrap_mean_ci(
                group[col].to_numpy(),
                seed=123 + seed_offset,
            )

            row = base.copy()
            row["quantity"] = col
            row["mean"] = ci["mean"]
            row["ci_low"] = ci["ci_low"]
            row["ci_high"] = ci["ci_high"]
            row["n_units"] = ci["n"]
            rows.append(row)

    return pd.DataFrame(rows)


frozen_v3_main_ci = make_ci_summary(
    df=frozen_v3_delta_wide,
    group_cols=["split_name", "information_regime", "backbone", "method"],
    value_cols=DELTA_COLS,
    seed_offset=300,
)

v3_ci_path = DIRS["stats"] / "frozen_v3_main_delta_bootstrap_ci.csv"
frozen_v3_main_ci.to_csv(v3_ci_path, index=False)

print("Saved frozen V3 bootstrap CI:", v3_ci_path)
display(frozen_v3_main_ci.round(6))

**Wide CI table**

In [ ]:
ci_wide_rows = []

for key, group in frozen_v3_main_ci.groupby(
    ["split_name", "information_regime", "backbone", "method"]
):
    split_name, regime, backbone, method = key

    row = {
        "split_name": split_name,
        "information_regime": regime,
        "backbone": backbone,
        "method": method,
    }

    for _, r in group.iterrows():
        q = r["quantity"]
        row[f"{q}_mean"] = r["mean"]
        row[f"{q}_ci_low"] = r["ci_low"]
        row[f"{q}_ci_high"] = r["ci_high"]
        row[f"{q}_n"] = r["n_units"]

    ci_wide_rows.append(row)

frozen_v3_main_ci_wide = pd.DataFrame(ci_wide_rows)

v3_ci_wide_path = DIRS["stats"] / "frozen_v3_main_delta_bootstrap_ci_wide.csv"
frozen_v3_main_ci_wide.to_csv(v3_ci_wide_path, index=False)

print("Saved frozen V3 wide CI:", v3_ci_wide_path)
display(frozen_v3_main_ci_wide.round(6))

**Add Safe V2 bootstrap CI**

In [ ]:
safe_v2_main_ci = make_ci_summary(
    df=safe_v2_delta_wide,
    group_cols=["split_name", "information_regime", "backbone", "method"],
    value_cols=DELTA_COLS,
    seed_offset=250,
)

safe_v2_ci_path = DIRS["stats"] / "safe_v2_main_delta_bootstrap_ci.csv"
safe_v2_main_ci.to_csv(safe_v2_ci_path, index=False)

print("Saved Safe V2 bootstrap CI:", safe_v2_ci_path)
display(safe_v2_main_ci.round(6))


safe_v2_ci_wide_rows = []

for key, group in safe_v2_main_ci.groupby(
    ["split_name", "information_regime", "backbone", "method"]
):
    split_name, regime, backbone, method = key

    row = {
        "split_name": split_name,
        "information_regime": regime,
        "backbone": backbone,
        "method": method,
    }

    for _, r in group.iterrows():
        q = r["quantity"]
        row[f"{q}_mean"] = r["mean"]
        row[f"{q}_ci_low"] = r["ci_low"]
        row[f"{q}_ci_high"] = r["ci_high"]
        row[f"{q}_n"] = r["n_units"]

    safe_v2_ci_wide_rows.append(row)

safe_v2_main_ci_wide = pd.DataFrame(safe_v2_ci_wide_rows)

safe_v2_ci_wide_path = DIRS["stats"] / "safe_v2_main_delta_bootstrap_ci_wide.csv"
safe_v2_main_ci_wide.to_csv(safe_v2_ci_wide_path, index=False)

print("Saved Safe V2 wide CI:", safe_v2_ci_wide_path)
display(safe_v2_main_ci_wide.round(6))

**Combined Safe V2 + Frozen V3 main comparison**

In [ ]:
stage3_safe_methods_summary = pd.concat(
    [
        safe_v2_main_delta_summary.assign(safe_method_block="safe_v2"),
        frozen_v3_main_delta_summary.assign(safe_method_block="frozen_v3"),
    ],
    ignore_index=True,
    sort=False,
)

stage3_safe_methods_summary_path = DIRS["tables"] / "stage3_safe_v2_v3_main_delta_summary.csv"
stage3_safe_methods_summary.to_csv(stage3_safe_methods_summary_path, index=False)

stage3_safe_methods_delta_wide = pd.concat(
    [
        safe_v2_delta_wide.assign(safe_method_block="safe_v2"),
        frozen_v3_delta_wide.assign(safe_method_block="frozen_v3"),
    ],
    ignore_index=True,
    sort=False,
)

stage3_safe_methods_delta_path = DIRS["tables"] / "stage3_safe_v2_v3_main_delta_vs_source_ts_wide.csv"
stage3_safe_methods_delta_wide.to_csv(stage3_safe_methods_delta_path, index=False)

print("Saved Stage 3 safe-method summary:", stage3_safe_methods_summary_path)
print("Saved Stage 3 safe-method delta wide:", stage3_safe_methods_delta_path)

display(
    stage3_safe_methods_summary[
        [
            "safe_method_block",
            "backbone",
            "method",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "predicted_T_mean",
            "predicted_T_min",
            "predicted_T_max",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

In [ ]:
# Stage 3 completion note

stage3_note = {
    "stage": "stage_3_safe_v2_v3_downstream_evaluation",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "main_outputs": {
        "safe_v2_eval_long": str(safe_v2_eval_path),
        "safe_v2_condition_metric_wide": str(safe_v2_wide_path),
        "safe_v2_delta_vs_source_ts_wide": str(safe_v2_delta_path),
        "safe_v2_main_delta_summary": str(safe_v2_summary_path),
        "safe_v2_bootstrap_ci": str(safe_v2_ci_path),
        "safe_v2_bootstrap_ci_wide": str(safe_v2_ci_wide_path),

        "frozen_v3_eval_long": str(v3_eval_path),
        "frozen_v3_condition_metric_wide": str(v3_wide_path),
        "frozen_v3_delta_vs_source_ts_wide": str(v3_delta_path),
        "frozen_v3_main_delta_summary": str(v3_summary_path),
        "frozen_v3_delta_by_severity": str(v3_by_severity_path),
        "frozen_v3_delta_by_corruption": str(v3_by_corruption_path),
        "frozen_v3_harm_summary": str(v3_harm_path),
        "support_eval_by_backbone": str(support_backbone_eval_path),
        "support_eval_by_severity": str(support_severity_eval_path),
        "support_eval_by_corruption": str(support_corruption_eval_path),
        "support_eval_full": str(support_full_eval_path),
        "frozen_v3_bootstrap_ci": str(v3_ci_path),
        "frozen_v3_bootstrap_ci_wide": str(v3_ci_wide_path),

        "combined_safe_v2_v3_summary": str(stage3_safe_methods_summary_path),
        "combined_safe_v2_v3_delta_wide": str(stage3_safe_methods_delta_path),
    },

    "checks": {
        "target_labels_used_only_for_evaluation": True,
        "safe_v2_predictions_generated_before_label_evaluation": True,
        "frozen_v3_predictions_generated_before_label_evaluation": True,
        "ece15_computed_from_logits": True,
        "safe_v2_ece15_values_between_0_and_1": bool(safe_v2_condition_wide["ece15"].between(0, 1).all()),
        "safe_v2_aece15_values_between_0_and_1": bool(safe_v2_condition_wide["aece15"].between(0, 1).all()),
        "v3_ece15_values_between_0_and_1": bool(frozen_v3_condition_wide["ece15"].between(0, 1).all()),
        "v3_aece15_values_between_0_and_1": bool(frozen_v3_condition_wide["aece15"].between(0, 1).all()),
        "deltas_vs_corrected_refit_source_ts": True,
        "v3_support_extrapolation_tables_created": True,
        "safe_v2_and_v3_evaluated_downstream": True,
    },

    "important_note": (
        "Stage 3 evaluates Safe V2 and frozen V3 downstream on cached target ImageNet-C logits. "
        "Target labels are used only for evaluation metrics. V2 and V3 temperatures were generated "
        "before label-based evaluation using regenerated source-bank fits and unlabeled target stream descriptors."
    ),
}

stage3_note_path = DIRS["logs"] / "stage3_safe_v2_v3_downstream_evaluation_completion.json"

with open(stage3_note_path, "w") as f:
    json.dump(stage3_note, f, indent=2)

print("Saved Stage 3 completion note:", stage3_note_path)
print(json.dumps(stage3_note, indent=2))

**Stage 4**

**Generic delta helper**

In [ ]:
def eval_long_to_condition_wide(eval_long: pd.DataFrame) -> pd.DataFrame:
    wide = (
        eval_long
        .pivot_table(
            index=[
                "split_name",
                "heldout_type",
                "heldout_value",
                "information_regime",
                "backbone",
                "seed",
                "method",
                "corruption",
                "severity",
                "condition",
                "temperature",
            ],
            columns="metric",
            values="value",
            aggfunc="mean",
        )
        .reset_index()
    )

    wide.columns.name = None

    required_metric_cols = [
        "accuracy",
        "mean_confidence",
        "signed_gap",
        "abs_signed_gap",
        "nll",
        "brier",
        "ece15",
        "aece15",
    ]

    missing = [c for c in required_metric_cols if c not in wide.columns]
    assert len(missing) == 0, f"Missing metric columns after pivot: {missing}"

    assert wide["ece15"].between(0, 1).all(), "ECE15 outside [0,1]"
    assert wide["aece15"].between(0, 1).all(), "AECE15 outside [0,1]"

    return wide


def add_corrected_source_ts_deltas(condition_wide: pd.DataFrame) -> pd.DataFrame:
    required_metric_cols = [
        "accuracy",
        "mean_confidence",
        "signed_gap",
        "abs_signed_gap",
        "nll",
        "brier",
        "ece15",
        "aece15",
    ]

    source_ts_wide = corrected_condition_metric_wide[
        corrected_condition_metric_wide["method"] == "source_global_ts"
    ].copy()

    source_ts_keep_cols = [
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
    ] + required_metric_cols

    source_ts_baseline = source_ts_wide[source_ts_keep_cols].rename(
        columns={m: f"source_ts_{m}" for m in required_metric_cols}
    )

    dup_source_ts = source_ts_baseline.duplicated(
        subset=["backbone", "seed", "corruption", "severity", "condition"]
    ).sum()

    assert dup_source_ts == 0, f"Duplicate source TS baseline rows: {dup_source_ts}"

    merge_keys = ["backbone", "seed", "corruption", "severity", "condition"]

    out = condition_wide.merge(
        source_ts_baseline,
        on=merge_keys,
        how="left",
    )

    assert len(out) == len(condition_wide), "Merge with source TS changed row count."
    assert out["source_ts_nll"].notna().all(), "Missing source TS after merge"

    for metric in required_metric_cols:
        out[f"delta_{metric}_vs_source_ts"] = (
            out[metric] - out[f"source_ts_{metric}"]
        )

    # Positive scalar temperature scaling preserves argmax.
    assert (out["delta_accuracy_vs_source_ts"].abs() < 1e-10).all(), (
        "Temperature scaling changed accuracy relative to source TS. "
        "This should not happen for positive scalar temperatures."
    )

    assert out["ece15"].between(0, 1).all()
    assert out["aece15"].between(0, 1).all()
    assert out["source_ts_ece15"].between(0, 1).all()
    assert out["source_ts_aece15"].between(0, 1).all()

    return out


def summarize_delta_wide(delta_wide: pd.DataFrame) -> pd.DataFrame:
    summary = (
        delta_wide
        .groupby(
            ["split_name", "heldout_type", "information_regime", "backbone", "method"],
            as_index=False,
        )
        .agg(
            delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
            delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
            delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
            delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
            delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
            delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

            nll_mean=("nll", "mean"),
            brier_mean=("brier", "mean"),
            ece15_mean=("ece15", "mean"),
            aece15_mean=("aece15", "mean"),

            source_ts_nll_mean=("source_ts_nll", "mean"),
            source_ts_brier_mean=("source_ts_brier", "mean"),
            source_ts_ece15_mean=("source_ts_ece15", "mean"),
            source_ts_aece15_mean=("source_ts_aece15", "mean"),

            temperature_mean=("temperature", "mean"),
            temperature_min=("temperature", "min"),
            temperature_max=("temperature", "max"),

            n_conditions=("delta_nll_vs_source_ts", "count"),
        )
    )

    assert summary["ece15_mean"].between(0, 1).all()
    assert summary["source_ts_ece15_mean"].between(0, 1).all()

    return summary


print("Stage 4 generic delta helpers ready.")

**Run stream ridge baselines under LOCO/LOSO**

In [ ]:
def fit_stream_model_by_method(method: str, train_df: pd.DataFrame, features, seed: int):
    if "mlp" in method:
        return fit_stream_mlp_model(
            train_df=train_df,
            feature_names=features,
            target_col="source_bank_target_log_temperature",
            random_state=1000 + int(seed),
        )

    return fit_stream_ridge_model(
        train_df=train_df,
        feature_names=features,
        target_col="source_bank_target_log_temperature",
        alpha=1.0,
    )


def run_stream_baselines_condition_split(split_name: str):
    assert split_name in ["leave_one_corruption_family_out", "leave_one_severity_out"]

    eval_rows = []
    pred_rows = []

    if split_name == "leave_one_corruption_family_out":
        heldout_values = PROTOCOL["corruptions"]
        heldout_type = "corruption"
    else:
        heldout_values = PROTOCOL["severities"]
        heldout_type = "severity"

    for backbone in PROTOCOL["backbones"]:
        for seed in PROTOCOL["seeds"]:
            print(f"\nStream baselines | {split_name} | {backbone} | seed={seed}")

            source_bank_seed_df = source_bank_stream_df[
                (source_bank_stream_df["backbone"] == backbone) &
                (source_bank_stream_df["seed"] == seed)
            ].copy()

            target_seed_df = target_stream_df[
                (target_stream_df["backbone"] == backbone) &
                (target_stream_df["seed"] == seed)
            ].copy()

            assert len(source_bank_seed_df) == 50
            assert len(target_seed_df) == 50

            for heldout_value in heldout_values:
                if split_name == "leave_one_corruption_family_out":
                    train_df = source_bank_seed_df[
                        source_bank_seed_df["corruption"] != heldout_value
                    ].copy()

                    test_target_df = target_seed_df[
                        target_seed_df["corruption"] == heldout_value
                    ].copy()

                    assert len(train_df) == 45
                    assert len(test_target_df) == 5

                else:
                    train_df = source_bank_seed_df[
                        source_bank_seed_df["severity"] != heldout_value
                    ].copy()

                    test_target_df = target_seed_df[
                        target_seed_df["severity"] == heldout_value
                    ].copy()

                    assert len(train_df) == 40
                    assert len(test_target_df) == 10

                for method, features in STREAM_FEATURE_SETS.items():
                    model = fit_stream_model_by_method(
                        method=method,
                        train_df=train_df,
                        features=features,
                        seed=seed,
                    )

                    pred_T, pred_logT = predict_stream_temperature_model(
                        model=model,
                        test_df=test_target_df,
                        feature_names=features,
                        t_min=0.8,
                        t_max=2.5,
                    )

                    test_target_df_reset = test_target_df.reset_index(drop=True)

                    for i, row in test_target_df_reset.iterrows():
                        corruption = row["corruption"]
                        severity = int(row["severity"])
                        condition = row["condition"]
                        T = float(pred_T[i])

                        pred_rows.append({
                            "split_name": split_name,
                            "heldout_type": heldout_type,
                            "heldout_value": str(heldout_value),
                            "information_regime": "unlabeled_target_stream_condition_split",
                            "backbone": backbone,
                            "seed": int(seed),
                            "method": method,
                            "corruption": corruption,
                            "severity": severity,
                            "condition": condition,
                            "predicted_T": T,
                            "predicted_logT": float(pred_logT[i]),
                        })

                        eval_rows.extend(
                            evaluate_temperature_on_target_condition(
                                backbone=backbone,
                                seed=seed,
                                corruption=corruption,
                                severity=severity,
                                method=method,
                                information_regime="unlabeled_target_stream_condition_split",
                                temperature=T,
                                split_name=split_name,
                                heldout_type=heldout_type,
                                heldout_value=str(heldout_value),
                            )
                        )

    return pd.DataFrame(eval_rows), pd.DataFrame(pred_rows)


stream_loco_eval_long, stream_loco_predictions = run_stream_baselines_condition_split(
    "leave_one_corruption_family_out"
)

stream_loso_eval_long, stream_loso_predictions = run_stream_baselines_condition_split(
    "leave_one_severity_out"
)

stream_loco_eval_path = DIRS["stream_methods"] / "stream_baselines_loco_eval_long.csv"
stream_loco_pred_path = DIRS["stream_methods"] / "stream_baselines_loco_predictions.csv"
stream_loso_eval_path = DIRS["stream_methods"] / "stream_baselines_loso_eval_long.csv"
stream_loso_pred_path = DIRS["stream_methods"] / "stream_baselines_loso_predictions.csv"

stream_loco_eval_long.to_csv(stream_loco_eval_path, index=False)
stream_loco_predictions.to_csv(stream_loco_pred_path, index=False)
stream_loso_eval_long.to_csv(stream_loso_eval_path, index=False)
stream_loso_predictions.to_csv(stream_loso_pred_path, index=False)

print("Saved stream baseline LOCO/LOSO evals and predictions:")
print(stream_loco_eval_path)
print(stream_loco_pred_path)
print(stream_loso_eval_path)
print(stream_loso_pred_path)

display(stream_loco_eval_long.head())
display(stream_loso_eval_long.head())

**Stream baseline LOCO/LOSO deltas and summaries**

In [ ]:
stream_loco_condition_wide = eval_long_to_condition_wide(stream_loco_eval_long)
stream_loso_condition_wide = eval_long_to_condition_wide(stream_loso_eval_long)

stream_loco_delta_wide = add_corrected_source_ts_deltas(stream_loco_condition_wide)
stream_loso_delta_wide = add_corrected_source_ts_deltas(stream_loso_condition_wide)

stream_loco_delta_path = DIRS["stream_methods"] / "stream_baselines_loco_delta_vs_source_ts_wide.csv"
stream_loso_delta_path = DIRS["stream_methods"] / "stream_baselines_loso_delta_vs_source_ts_wide.csv"

stream_loco_delta_wide.to_csv(stream_loco_delta_path, index=False)
stream_loso_delta_wide.to_csv(stream_loso_delta_path, index=False)

stream_loco_summary = summarize_delta_wide(stream_loco_delta_wide)
stream_loso_summary = summarize_delta_wide(stream_loso_delta_wide)

stream_loco_summary_path = DIRS["stream_methods"] / "stream_baselines_loco_delta_summary.csv"
stream_loso_summary_path = DIRS["stream_methods"] / "stream_baselines_loso_delta_summary.csv"

stream_loco_summary.to_csv(stream_loco_summary_path, index=False)
stream_loso_summary.to_csv(stream_loso_summary_path, index=False)

print("Saved stream baseline delta tables:")
print(stream_loco_delta_path)
print(stream_loso_delta_path)
print(stream_loco_summary_path)
print(stream_loso_summary_path)

print("\nStream LOCO summary:")
display(
    stream_loco_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

print("\nStream LOSO summary:")
display(
    stream_loso_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Safe V2 under LOCO/LOSO**

In [ ]:
def run_safe_v2_condition_split(split_name: str):
    assert split_name in ["leave_one_corruption_family_out", "leave_one_severity_out"]

    eval_rows = []
    pred_rows = []
    model_rows = []
    bin_rows = []

    if split_name == "leave_one_corruption_family_out":
        heldout_values = PROTOCOL["corruptions"]
        heldout_type = "corruption"
    else:
        heldout_values = PROTOCOL["severities"]
        heldout_type = "severity"

    for backbone in PROTOCOL["backbones"]:
        for seed in PROTOCOL["seeds"]:
            print(f"\nSafe V2 | {split_name} | {backbone} | seed={seed}")

            source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

            source_bank_seed_df = source_bank_stream_df[
                (source_bank_stream_df["backbone"] == backbone) &
                (source_bank_stream_df["seed"] == seed)
            ].copy()

            target_seed_df = target_stream_df[
                (target_stream_df["backbone"] == backbone) &
                (target_stream_df["seed"] == seed)
            ].copy()

            assert len(source_bank_seed_df) == 50
            assert len(target_seed_df) == 50

            for heldout_value in heldout_values:
                if split_name == "leave_one_corruption_family_out":
                    train_df = source_bank_seed_df[
                        source_bank_seed_df["corruption"] != heldout_value
                    ].copy()

                    test_target_df = target_seed_df[
                        target_seed_df["corruption"] == heldout_value
                    ].copy()

                    assert len(train_df) == 45
                    assert len(test_target_df) == 5

                else:
                    train_df = source_bank_seed_df[
                        source_bank_seed_df["severity"] != heldout_value
                    ].copy()

                    test_target_df = target_seed_df[
                        target_seed_df["severity"] == heldout_value
                    ].copy()

                    assert len(train_df) == 40
                    assert len(test_target_df) == 10

                model_info = fit_safe_v2_model(
                    train_df=train_df,
                    source_T=source_T,
                    config=SAFE_V2_CONFIG,
                )

                model_rows.append({
                    "split_name": split_name,
                    "heldout_type": heldout_type,
                    "heldout_value": str(heldout_value),
                    "backbone": backbone,
                    "seed": int(seed),
                    "source_T": float(source_T),
                    "T_min": model_info["T_min"],
                    "T_max_safe": model_info["T_max_safe"],
                    "entropy_train_min": model_info["entropy_train_min"],
                    "entropy_train_max": model_info["entropy_train_max"],
                    "entropy_train_mean": model_info["entropy_train_mean"],
                    "entropy_train_std": model_info["entropy_train_std"],
                    "n_train_conditions": model_info["n_train_conditions"],
                    "n_bins_used": model_info["n_bins_used"],
                })

                binned = model_info["binned"].copy()
                binned["split_name"] = split_name
                binned["heldout_type"] = heldout_type
                binned["heldout_value"] = str(heldout_value)
                binned["backbone"] = backbone
                binned["seed"] = int(seed)
                binned["source_T"] = float(source_T)
                binned["T_max_safe"] = model_info["T_max_safe"]
                bin_rows.append(binned)

                for _, row in test_target_df.iterrows():
                    entropy_mean = float(row["entropy_mean"])

                    pred = predict_safe_v2_temperature(
                        model_info=model_info,
                        entropy_value=entropy_mean,
                    )

                    T = float(pred["predicted_T"])

                    pred_rows.append({
                        "split_name": split_name,
                        "heldout_type": heldout_type,
                        "heldout_value": str(heldout_value),
                        "information_regime": "unlabeled_target_stream_safe_v2_condition_split",
                        "backbone": backbone,
                        "seed": int(seed),
                        "method": SAFE_V2_CONFIG["method"],
                        "corruption": row["corruption"],
                        "severity": int(row["severity"]),
                        "condition": row["condition"],
                        "entropy_mean": entropy_mean,
                        "predicted_T": T,
                        "raw_predicted_T": float(pred["raw_predicted_T"]),
                        "source_T": float(source_T),
                        "support_region": pred["support_region"],
                        "used_lower_fallback": bool(pred["used_lower_fallback"]),
                        "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                        "inside_support": bool(pred["inside_support"]),
                        "below_support": bool(pred["below_support"]),
                        "above_support": bool(pred["above_support"]),
                    })

                    eval_rows.extend(
                        evaluate_temperature_on_target_condition(
                            backbone=backbone,
                            seed=seed,
                            corruption=row["corruption"],
                            severity=int(row["severity"]),
                            method=SAFE_V2_CONFIG["method"],
                            information_regime="unlabeled_target_stream_safe_v2_condition_split",
                            temperature=T,
                            split_name=split_name,
                            heldout_type=heldout_type,
                            heldout_value=str(heldout_value),
                        )
                    )

    eval_long = pd.DataFrame(eval_rows)
    predictions = pd.DataFrame(pred_rows)
    model_info_df = pd.DataFrame(model_rows)
    bins_df = pd.concat(bin_rows, ignore_index=True)

    return eval_long, predictions, model_info_df, bins_df


v2_loco_eval_long, v2_loco_predictions, v2_loco_model_info, v2_loco_bins = run_safe_v2_condition_split(
    "leave_one_corruption_family_out"
)

v2_loso_eval_long, v2_loso_predictions, v2_loso_model_info, v2_loso_bins = run_safe_v2_condition_split(
    "leave_one_severity_out"
)

v2_loco_eval_path = DIRS["safe_v2"] / "safe_v2_loco_eval_long.csv"
v2_loco_pred_path = DIRS["safe_v2"] / "safe_v2_loco_predictions.csv"
v2_loco_model_path = DIRS["safe_v2"] / "safe_v2_loco_model_info.csv"
v2_loco_bins_path = DIRS["safe_v2"] / "safe_v2_loco_entropy_bins.csv"

v2_loso_eval_path = DIRS["safe_v2"] / "safe_v2_loso_eval_long.csv"
v2_loso_pred_path = DIRS["safe_v2"] / "safe_v2_loso_predictions.csv"
v2_loso_model_path = DIRS["safe_v2"] / "safe_v2_loso_model_info.csv"
v2_loso_bins_path = DIRS["safe_v2"] / "safe_v2_loso_entropy_bins.csv"

v2_loco_eval_long.to_csv(v2_loco_eval_path, index=False)
v2_loco_predictions.to_csv(v2_loco_pred_path, index=False)
v2_loco_model_info.to_csv(v2_loco_model_path, index=False)
v2_loco_bins.to_csv(v2_loco_bins_path, index=False)

v2_loso_eval_long.to_csv(v2_loso_eval_path, index=False)
v2_loso_predictions.to_csv(v2_loso_pred_path, index=False)
v2_loso_model_info.to_csv(v2_loso_model_path, index=False)
v2_loso_bins.to_csv(v2_loso_bins_path, index=False)

print("Saved Safe V2 LOCO/LOSO outputs:")
print(v2_loco_eval_path)
print(v2_loco_pred_path)
print(v2_loso_eval_path)
print(v2_loso_pred_path)

**Safe V2 LOCO/LOSO deltas and summaries**

In [ ]:
v2_loco_condition_wide = eval_long_to_condition_wide(v2_loco_eval_long)
v2_loso_condition_wide = eval_long_to_condition_wide(v2_loso_eval_long)

v2_loco_delta_wide = add_corrected_source_ts_deltas(v2_loco_condition_wide)
v2_loso_delta_wide = add_corrected_source_ts_deltas(v2_loso_condition_wide)

def merge_safe_prediction_meta_for_split(delta_wide: pd.DataFrame, predictions: pd.DataFrame) -> pd.DataFrame:
    merge_keys = [
        "split_name",
        "heldout_type",
        "heldout_value",
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
    ]

    support_cols = merge_keys + [
        "entropy_mean",
        "predicted_T",
        "raw_predicted_T",
        "source_T",
        "support_region",
        "used_lower_fallback",
        "used_high_extrapolation",
        "inside_support",
        "below_support",
        "above_support",
    ]

    support_meta = predictions[support_cols].copy()

    dup_support = support_meta.duplicated(subset=merge_keys).sum()
    assert dup_support == 0, f"Duplicate safe-method metadata rows: {dup_support}"

    out = delta_wide.merge(
        support_meta,
        on=merge_keys,
        how="left",
        suffixes=("", "_pred"),
    )

    assert len(out) == len(delta_wide), "Prediction metadata merge changed row count."
    assert out["support_region"].notna().all(), "Missing prediction metadata after merge"

    assert np.allclose(
        out["temperature"].to_numpy(dtype=float),
        out["predicted_T"].to_numpy(dtype=float),
        atol=1e-10,
    ), "Mismatch between evaluated temperature and predicted_T"

    return out


v2_loco_delta_wide = merge_safe_prediction_meta_for_split(v2_loco_delta_wide, v2_loco_predictions)
v2_loso_delta_wide = merge_safe_prediction_meta_for_split(v2_loso_delta_wide, v2_loso_predictions)

v2_loco_delta_path = DIRS["safe_v2"] / "safe_v2_loco_delta_vs_source_ts_wide.csv"
v2_loso_delta_path = DIRS["safe_v2"] / "safe_v2_loso_delta_vs_source_ts_wide.csv"

v2_loco_delta_wide.to_csv(v2_loco_delta_path, index=False)
v2_loso_delta_wide.to_csv(v2_loso_delta_path, index=False)

v2_loco_summary = summarize_delta_wide(v2_loco_delta_wide)
v2_loso_summary = summarize_delta_wide(v2_loso_delta_wide)

v2_loco_summary_path = DIRS["safe_v2"] / "safe_v2_loco_delta_summary.csv"
v2_loso_summary_path = DIRS["safe_v2"] / "safe_v2_loso_delta_summary.csv"

v2_loco_summary.to_csv(v2_loco_summary_path, index=False)
v2_loso_summary.to_csv(v2_loso_summary_path, index=False)

print("Saved Safe V2 LOCO/LOSO delta outputs:")
print(v2_loco_delta_path)
print(v2_loso_delta_path)
print(v2_loco_summary_path)
print(v2_loso_summary_path)

print("\nSafe V2 LOCO summary:")
display(v2_loco_summary.round(6))

print("\nSafe V2 LOSO summary:")
display(v2_loso_summary.round(6))

**Frozen V3 under LOCO/LOSO**

In [ ]:
def run_frozen_v3_condition_split(split_name: str):
    assert split_name in ["leave_one_corruption_family_out", "leave_one_severity_out"]

    eval_rows = []
    pred_rows = []
    model_rows = []
    bin_rows = []

    if split_name == "leave_one_corruption_family_out":
        heldout_values = PROTOCOL["corruptions"]
        heldout_type = "corruption"
    else:
        heldout_values = PROTOCOL["severities"]
        heldout_type = "severity"

    for backbone in PROTOCOL["backbones"]:
        for seed in PROTOCOL["seeds"]:
            print(f"\nFrozen V3 | {split_name} | {backbone} | seed={seed}")

            source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

            source_bank_seed_df = source_bank_stream_df[
                (source_bank_stream_df["backbone"] == backbone) &
                (source_bank_stream_df["seed"] == seed)
            ].copy()

            target_seed_df = target_stream_df[
                (target_stream_df["backbone"] == backbone) &
                (target_stream_df["seed"] == seed)
            ].copy()

            assert len(source_bank_seed_df) == 50
            assert len(target_seed_df) == 50

            for heldout_value in heldout_values:
                if split_name == "leave_one_corruption_family_out":
                    train_df = source_bank_seed_df[
                        source_bank_seed_df["corruption"] != heldout_value
                    ].copy()

                    test_target_df = target_seed_df[
                        target_seed_df["corruption"] == heldout_value
                    ].copy()

                    assert len(train_df) == 45
                    assert len(test_target_df) == 5

                else:
                    train_df = source_bank_seed_df[
                        source_bank_seed_df["severity"] != heldout_value
                    ].copy()

                    test_target_df = target_seed_df[
                        target_seed_df["severity"] == heldout_value
                    ].copy()

                    assert len(train_df) == 40
                    assert len(test_target_df) == 10

                model_info = fit_frozen_v3_model(
                    train_df=train_df,
                    source_T=source_T,
                    config=FROZEN_V3_CONFIG,
                )

                model_rows.append({
                    "split_name": split_name,
                    "heldout_type": heldout_type,
                    "heldout_value": str(heldout_value),
                    "backbone": backbone,
                    "seed": int(seed),
                    "source_T": float(source_T),
                    "T_min": model_info["T_min"],
                    "T_max_safe": model_info["T_max_safe"],
                    "entropy_train_min": model_info["entropy_train_min"],
                    "entropy_train_max": model_info["entropy_train_max"],
                    "entropy_train_mean": model_info["entropy_train_mean"],
                    "entropy_train_std": model_info["entropy_train_std"],
                    "support_low": model_info["support_low"],
                    "support_high": model_info["support_high"],
                    "T_at_train_max": model_info["T_at_train_max"],
                    "raw_extrapolation_slope": model_info["raw_extrapolation_slope"],
                    "extrapolation_slope": model_info["extrapolation_slope"],
                    "n_train_conditions": model_info["n_train_conditions"],
                    "n_bins_used": model_info["n_bins_used"],
                })

                binned = model_info["binned"].copy()
                binned["split_name"] = split_name
                binned["heldout_type"] = heldout_type
                binned["heldout_value"] = str(heldout_value)
                binned["backbone"] = backbone
                binned["seed"] = int(seed)
                binned["source_T"] = float(source_T)
                binned["T_max_safe"] = model_info["T_max_safe"]
                bin_rows.append(binned)

                for _, row in test_target_df.iterrows():
                    entropy_mean = float(row["entropy_mean"])

                    pred = predict_frozen_v3_temperature(
                        model_info=model_info,
                        entropy_value=entropy_mean,
                    )

                    T = float(pred["predicted_T"])

                    pred_rows.append({
                        "split_name": split_name,
                        "heldout_type": heldout_type,
                        "heldout_value": str(heldout_value),
                        "information_regime": "unlabeled_target_stream_safe_v3_frozen_condition_split",
                        "backbone": backbone,
                        "seed": int(seed),
                        "method": FROZEN_V3_CONFIG["method"],
                        "corruption": row["corruption"],
                        "severity": int(row["severity"]),
                        "condition": row["condition"],
                        "entropy_mean": entropy_mean,
                        "predicted_T": T,
                        "raw_predicted_T": float(pred["raw_predicted_T"]),
                        "source_T": float(source_T),

                        "support_region": pred["support_region"],
                        "used_lower_fallback": bool(pred["used_lower_fallback"]),
                        "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                        "inside_support": bool(pred["inside_support"]),
                        "below_support": bool(pred["below_support"]),
                        "above_support": bool(pred["above_support"]),
                    })

                    eval_rows.extend(
                        evaluate_temperature_on_target_condition(
                            backbone=backbone,
                            seed=seed,
                            corruption=row["corruption"],
                            severity=int(row["severity"]),
                            method=FROZEN_V3_CONFIG["method"],
                            information_regime="unlabeled_target_stream_safe_v3_frozen_condition_split",
                            temperature=T,
                            split_name=split_name,
                            heldout_type=heldout_type,
                            heldout_value=str(heldout_value),
                        )
                    )

    eval_long = pd.DataFrame(eval_rows)
    predictions = pd.DataFrame(pred_rows)
    model_info_df = pd.DataFrame(model_rows)
    bins_df = pd.concat(bin_rows, ignore_index=True)

    return eval_long, predictions, model_info_df, bins_df


v3_loco_eval_long, v3_loco_predictions, v3_loco_model_info, v3_loco_bins = run_frozen_v3_condition_split(
    "leave_one_corruption_family_out"
)

v3_loso_eval_long, v3_loso_predictions, v3_loso_model_info, v3_loso_bins = run_frozen_v3_condition_split(
    "leave_one_severity_out"
)

v3_loco_eval_path = DIRS["safe_v3"] / "frozen_v3_loco_eval_long.csv"
v3_loco_pred_path = DIRS["safe_v3"] / "frozen_v3_loco_predictions.csv"
v3_loco_model_path = DIRS["safe_v3"] / "frozen_v3_loco_model_info.csv"
v3_loco_bins_path = DIRS["safe_v3"] / "frozen_v3_loco_entropy_bins.csv"

v3_loso_eval_path = DIRS["safe_v3"] / "frozen_v3_loso_eval_long.csv"
v3_loso_pred_path = DIRS["safe_v3"] / "frozen_v3_loso_predictions.csv"
v3_loso_model_path = DIRS["safe_v3"] / "frozen_v3_loso_model_info.csv"
v3_loso_bins_path = DIRS["safe_v3"] / "frozen_v3_loso_entropy_bins.csv"

v3_loco_eval_long.to_csv(v3_loco_eval_path, index=False)
v3_loco_predictions.to_csv(v3_loco_pred_path, index=False)
v3_loco_model_info.to_csv(v3_loco_model_path, index=False)
v3_loco_bins.to_csv(v3_loco_bins_path, index=False)

v3_loso_eval_long.to_csv(v3_loso_eval_path, index=False)
v3_loso_predictions.to_csv(v3_loso_pred_path, index=False)
v3_loso_model_info.to_csv(v3_loso_model_path, index=False)
v3_loso_bins.to_csv(v3_loso_bins_path, index=False)

print("Saved frozen V3 LOCO/LOSO outputs:")
print(v3_loco_eval_path)
print(v3_loco_pred_path)
print(v3_loso_eval_path)
print(v3_loso_pred_path)

**Frozen V3 LOCO/LOSO deltas and summaries**

In [ ]:
v3_loco_condition_wide = eval_long_to_condition_wide(v3_loco_eval_long)
v3_loso_condition_wide = eval_long_to_condition_wide(v3_loso_eval_long)

v3_loco_delta_wide = add_corrected_source_ts_deltas(v3_loco_condition_wide)
v3_loso_delta_wide = add_corrected_source_ts_deltas(v3_loso_condition_wide)

def merge_support_meta_for_split(delta_wide: pd.DataFrame, predictions: pd.DataFrame) -> pd.DataFrame:
    merge_keys = [
        "split_name",
        "heldout_type",
        "heldout_value",
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
    ]

    support_cols = merge_keys + [
        "entropy_mean",
        "predicted_T",
        "raw_predicted_T",
        "source_T",
        "support_region",
        "used_lower_fallback",
        "used_high_extrapolation",
        "inside_support",
        "below_support",
        "above_support",
    ]

    support_meta = predictions[support_cols].copy()

    dup_support = support_meta.duplicated(subset=merge_keys).sum()
    assert dup_support == 0, f"Duplicate support metadata rows: {dup_support}"

    out = delta_wide.merge(
        support_meta,
        on=merge_keys,
        how="left",
        suffixes=("", "_pred"),
    )

    assert len(out) == len(delta_wide), "Support metadata merge changed row count."
    assert out["support_region"].notna().all(), "Missing support metadata after merge"

    assert np.allclose(
        out["temperature"].to_numpy(dtype=float),
        out["predicted_T"].to_numpy(dtype=float),
        atol=1e-10,
    ), "Mismatch between evaluated temperature and predicted_T"

    return out


v3_loco_delta_wide = merge_support_meta_for_split(v3_loco_delta_wide, v3_loco_predictions)
v3_loso_delta_wide = merge_support_meta_for_split(v3_loso_delta_wide, v3_loso_predictions)

v3_loco_delta_path = DIRS["safe_v3"] / "frozen_v3_loco_delta_vs_source_ts_wide.csv"
v3_loso_delta_path = DIRS["safe_v3"] / "frozen_v3_loso_delta_vs_source_ts_wide.csv"

v3_loco_delta_wide.to_csv(v3_loco_delta_path, index=False)
v3_loso_delta_wide.to_csv(v3_loso_delta_path, index=False)

v3_loco_summary = summarize_delta_wide(v3_loco_delta_wide)
v3_loso_summary = summarize_delta_wide(v3_loso_delta_wide)

v3_loco_summary_path = DIRS["safe_v3"] / "frozen_v3_loco_delta_summary.csv"
v3_loso_summary_path = DIRS["safe_v3"] / "frozen_v3_loso_delta_summary.csv"

v3_loco_summary.to_csv(v3_loco_summary_path, index=False)
v3_loso_summary.to_csv(v3_loso_summary_path, index=False)

print("Saved frozen V3 LOCO/LOSO delta outputs:")
print(v3_loco_delta_path)
print(v3_loso_delta_path)
print(v3_loco_summary_path)
print(v3_loso_summary_path)

print("\nFrozen V3 LOCO summary:")
display(v3_loco_summary.round(6))

print("\nFrozen V3 LOSO summary:")
display(v3_loso_summary.round(6))

**Combined Stage 4 comparison table**

In [ ]:
stage4_condition_split_summary = pd.concat(
    [
        stream_loco_summary.assign(block="stream_baselines_loco"),
        stream_loso_summary.assign(block="stream_baselines_loso"),
        v2_loco_summary.assign(block="safe_v2_loco"),
        v2_loso_summary.assign(block="safe_v2_loso"),
        v3_loco_summary.assign(block="frozen_v3_loco"),
        v3_loso_summary.assign(block="frozen_v3_loso"),
    ],
    ignore_index=True,
)

stage4_summary_path = DIRS["tables"] / "stage4_loco_loso_downstream_delta_summary.csv"
stage4_condition_split_summary.to_csv(stage4_summary_path, index=False)

print("Saved Stage 4 combined summary:", stage4_summary_path)

display(
    stage4_condition_split_summary[
        [
            "block",
            "split_name",
            "heldout_type",
            "information_regime",
            "backbone",
            "method",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "temperature_mean",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "split_name", "delta_nll_mean"])
    .round(6)
)

**Support/extrapolation tables for V3 LOCO/LOSO**

In [ ]:
def make_support_eval_tables(delta_wide: pd.DataFrame, split_tag: str):
    by_backbone = (
        delta_wide
        .groupby(["split_name", "backbone", "support_region"], as_index=False)
        .agg(
            count=("delta_nll_vs_source_ts", "count"),
            mean_predicted_T=("predicted_T", "mean"),
            mean_source_T=("source_T", "mean"),
            mean_entropy=("entropy_mean", "mean"),

            mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
            mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
            mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
            mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
            mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

            worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
            best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
        )
    )

    by_backbone["fraction"] = (
        by_backbone["count"] /
        by_backbone.groupby(["split_name", "backbone"])["count"].transform("sum")
    )

    frac_check = by_backbone.groupby(["split_name", "backbone"])["fraction"].sum()
    assert np.allclose(frac_check.values, 1.0), f"{split_tag}: fractions by backbone do not sum to 1"

    by_severity = (
        delta_wide
        .groupby(["split_name", "backbone", "severity", "support_region"], as_index=False)
        .agg(
            count=("delta_nll_vs_source_ts", "count"),
            mean_predicted_T=("predicted_T", "mean"),
            mean_source_T=("source_T", "mean"),
            mean_entropy=("entropy_mean", "mean"),

            mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
            mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
            mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
            mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
            mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

            worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
            best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
        )
    )

    by_severity["fraction"] = (
        by_severity["count"] /
        by_severity.groupby(["split_name", "backbone", "severity"])["count"].transform("sum")
    )

    frac_check = by_severity.groupby(["split_name", "backbone", "severity"])["fraction"].sum()
    assert np.allclose(frac_check.values, 1.0), f"{split_tag}: fractions by severity do not sum to 1"

    by_corruption = (
        delta_wide
        .groupby(["split_name", "backbone", "corruption", "support_region"], as_index=False)
        .agg(
            count=("delta_nll_vs_source_ts", "count"),
            mean_predicted_T=("predicted_T", "mean"),
            mean_source_T=("source_T", "mean"),
            mean_entropy=("entropy_mean", "mean"),

            mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
            mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
            mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
            mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
            mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

            worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
            best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
        )
    )

    by_corruption["fraction"] = (
        by_corruption["count"] /
        by_corruption.groupby(["split_name", "backbone", "corruption"])["count"].transform("sum")
    )

    frac_check = by_corruption.groupby(["split_name", "backbone", "corruption"])["fraction"].sum()
    assert np.allclose(frac_check.values, 1.0), f"{split_tag}: fractions by corruption do not sum to 1"

    by_full = (
        delta_wide
        .groupby(["split_name", "backbone", "corruption", "severity", "support_region"], as_index=False)
        .agg(
            count=("delta_nll_vs_source_ts", "count"),
            mean_predicted_T=("predicted_T", "mean"),
            mean_source_T=("source_T", "mean"),
            mean_entropy=("entropy_mean", "mean"),

            mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
            mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
            mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
            mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
            mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

            worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
            best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
        )
    )

    by_full["fraction"] = (
        by_full["count"] /
        by_full.groupby(["split_name", "backbone", "corruption", "severity"])["count"].transform("sum")
    )

    frac_check = by_full.groupby(["split_name", "backbone", "corruption", "severity"])["fraction"].sum()
    assert np.allclose(frac_check.values, 1.0), f"{split_tag}: fractions by corruption/severity do not sum to 1"

    return by_backbone, by_severity, by_corruption, by_full


v3_loco_support_backbone, v3_loco_support_severity, v3_loco_support_corruption, v3_loco_support_full = make_support_eval_tables(
    v3_loco_delta_wide,
    split_tag="loco",
)

v3_loso_support_backbone, v3_loso_support_severity, v3_loso_support_corruption, v3_loso_support_full = make_support_eval_tables(
    v3_loso_delta_wide,
    split_tag="loso",
)

v3_loco_support_backbone_path = DIRS["support_extrapolation"] / "frozen_v3_loco_support_eval_by_backbone.csv"
v3_loco_support_severity_path = DIRS["support_extrapolation"] / "frozen_v3_loco_support_eval_by_severity.csv"
v3_loco_support_corruption_path = DIRS["support_extrapolation"] / "frozen_v3_loco_support_eval_by_corruption.csv"
v3_loco_support_full_path = DIRS["support_extrapolation"] / "frozen_v3_loco_support_eval_by_backbone_corruption_severity.csv"

v3_loso_support_backbone_path = DIRS["support_extrapolation"] / "frozen_v3_loso_support_eval_by_backbone.csv"
v3_loso_support_severity_path = DIRS["support_extrapolation"] / "frozen_v3_loso_support_eval_by_severity.csv"
v3_loso_support_corruption_path = DIRS["support_extrapolation"] / "frozen_v3_loso_support_eval_by_corruption.csv"
v3_loso_support_full_path = DIRS["support_extrapolation"] / "frozen_v3_loso_support_eval_by_backbone_corruption_severity.csv"

v3_loco_support_backbone.to_csv(v3_loco_support_backbone_path, index=False)
v3_loco_support_severity.to_csv(v3_loco_support_severity_path, index=False)
v3_loco_support_corruption.to_csv(v3_loco_support_corruption_path, index=False)
v3_loco_support_full.to_csv(v3_loco_support_full_path, index=False)

v3_loso_support_backbone.to_csv(v3_loso_support_backbone_path, index=False)
v3_loso_support_severity.to_csv(v3_loso_support_severity_path, index=False)
v3_loso_support_corruption.to_csv(v3_loso_support_corruption_path, index=False)
v3_loso_support_full.to_csv(v3_loso_support_full_path, index=False)

print("Saved V3 LOCO/LOSO support/extrapolation tables.")
display(v3_loco_support_backbone.round(6))
display(v3_loso_support_backbone.round(6))

**Bootstrap CIs for Stage 4**

In [ ]:
stage4_all_delta_wide = pd.concat(
    [
        stream_loco_delta_wide.assign(stage4_block="stream_loco"),
        stream_loso_delta_wide.assign(stage4_block="stream_loso"),
        v2_loco_delta_wide.assign(stage4_block="safe_v2_loco"),
        v2_loso_delta_wide.assign(stage4_block="safe_v2_loso"),
        v3_loco_delta_wide.assign(stage4_block="v3_loco"),
        v3_loso_delta_wide.assign(stage4_block="v3_loso"),
    ],
    ignore_index=True,
    sort=False,
)

stage4_all_delta_path = DIRS["tables"] / "stage4_all_loco_loso_delta_vs_source_ts_wide.csv"
stage4_all_delta_wide.to_csv(stage4_all_delta_path, index=False)

stage4_ci = make_ci_summary(
    df=stage4_all_delta_wide,
    group_cols=[
        "stage4_block",
        "split_name",
        "heldout_type",
        "information_regime",
        "backbone",
        "method",
    ],
    value_cols=DELTA_COLS,
    seed_offset=400,
)

stage4_ci_path = DIRS["stats"] / "stage4_loco_loso_delta_bootstrap_ci.csv"
stage4_ci.to_csv(stage4_ci_path, index=False)

print("Saved Stage 4 all delta wide:", stage4_all_delta_path)
print("Saved Stage 4 bootstrap CI:", stage4_ci_path)

display(stage4_ci.round(6).head(60))

**Wide CI table**

In [ ]:
stage4_ci_wide_rows = []

for key, group in stage4_ci.groupby(
    [
        "stage4_block",
        "split_name",
        "heldout_type",
        "information_regime",
        "backbone",
        "method",
    ]
):
    stage4_block, split_name, heldout_type, regime, backbone, method = key

    row = {
        "stage4_block": stage4_block,
        "split_name": split_name,
        "heldout_type": heldout_type,
        "information_regime": regime,
        "backbone": backbone,
        "method": method,
    }

    for _, r in group.iterrows():
        q = r["quantity"]
        row[f"{q}_mean"] = r["mean"]
        row[f"{q}_ci_low"] = r["ci_low"]
        row[f"{q}_ci_high"] = r["ci_high"]
        row[f"{q}_n"] = r["n_units"]

    stage4_ci_wide_rows.append(row)

stage4_ci_wide = pd.DataFrame(stage4_ci_wide_rows)

stage4_ci_wide_path = DIRS["stats"] / "stage4_loco_loso_delta_bootstrap_ci_wide.csv"
stage4_ci_wide.to_csv(stage4_ci_wide_path, index=False)

print("Saved Stage 4 wide CI:", stage4_ci_wide_path)
display(stage4_ci_wide.round(6))

In [ ]:
stage4_note = {
    "stage": "stage_4_loco_loso_downstream_evaluation",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "main_outputs": {
        "stream_loco_eval_long": str(stream_loco_eval_path),
        "stream_loco_predictions": str(stream_loco_pred_path),
        "stream_loco_delta_wide": str(stream_loco_delta_path),
        "stream_loco_summary": str(stream_loco_summary_path),

        "stream_loso_eval_long": str(stream_loso_eval_path),
        "stream_loso_predictions": str(stream_loso_pred_path),
        "stream_loso_delta_wide": str(stream_loso_delta_path),
        "stream_loso_summary": str(stream_loso_summary_path),

        "safe_v2_loco_eval_long": str(v2_loco_eval_path),
        "safe_v2_loco_predictions": str(v2_loco_pred_path),
        "safe_v2_loco_delta_wide": str(v2_loco_delta_path),
        "safe_v2_loco_summary": str(v2_loco_summary_path),

        "safe_v2_loso_eval_long": str(v2_loso_eval_path),
        "safe_v2_loso_predictions": str(v2_loso_pred_path),
        "safe_v2_loso_delta_wide": str(v2_loso_delta_path),
        "safe_v2_loso_summary": str(v2_loso_summary_path),

        "v3_loco_eval_long": str(v3_loco_eval_path),
        "v3_loco_predictions": str(v3_loco_pred_path),
        "v3_loco_delta_wide": str(v3_loco_delta_path),
        "v3_loco_summary": str(v3_loco_summary_path),

        "v3_loso_eval_long": str(v3_loso_eval_path),
        "v3_loso_predictions": str(v3_loso_pred_path),
        "v3_loso_delta_wide": str(v3_loso_delta_path),
        "v3_loso_summary": str(v3_loso_summary_path),

        "combined_stage4_summary": str(stage4_summary_path),
        "stage4_all_delta_wide": str(stage4_all_delta_path),
        "stage4_bootstrap_ci": str(stage4_ci_path),
        "stage4_bootstrap_ci_wide": str(stage4_ci_wide_path),
    },

    "checks": {
        "loco_evaluates_downstream_target_metrics": True,
        "loso_evaluates_downstream_target_metrics": True,
        "metrics_include_nll_brier_ece_aece_gap": True,
        "stream_ridge_baselines_included": True,
        "stream_mlp_baseline_included": True,
        "safe_v2_included": True,
        "frozen_v3_included": True,
        "deltas_use_corrected_refit_source_ts": True,
        "target_labels_used_only_for_evaluation": True,
        "ece15_values_valid": True,
        "accuracy_invariant_checked": True,
        "support_extrapolation_tables_for_v3_loco_loso_created": True,
    },

    "important_note": (
        "Stage 4 evaluates LOCO/LOSO predicted temperatures on target ImageNet-C logits. "
        "This addresses the supervisor request to evaluate downstream NLL, Brier, ECE/AECE, "
        "and confidence-accuracy gap, not only temperature prediction error. "
        "It includes stream ridge baselines, stream MLP baseline, Safe V2, and frozen V3."
    ),
}

stage4_note_path = DIRS["logs"] / "stage4_loco_loso_downstream_evaluation_completion.json"

with open(stage4_note_path, "w") as f:
    json.dump(stage4_note, f, indent=2)

print("Saved Stage 4 completion note:", stage4_note_path)
print(json.dumps(stage4_note, indent=2))

display(
    stage4_condition_split_summary[
        [
            "split_name",
            "backbone",
            "method",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "split_name", "delta_nll_mean"])
    .round(6)
)

**STage 5**

**Define V3 ablation configs**

In [ ]:
V3_ABLATION_CONFIGS = {
    "full_v3": {
        "description": "Full frozen V3: monotone isotonic, source lower fallback, bounded cap, high-entropy capped extrapolation.",
        "use_monotonicity": True,
        "use_lower_fallback": True,
        "use_cap": True,
        "use_high_extrapolation": True,
    },

    "no_fallback": {
        "description": "No lower-support fallback. Below-support entropy is handled by model prediction instead of source TS fallback.",
        "use_monotonicity": True,
        "use_lower_fallback": False,
        "use_cap": True,
        "use_high_extrapolation": True,
    },

    "no_cap": {
        "description": "No maximum temperature cap. Tests whether bounded softening is necessary.",
        "use_monotonicity": True,
        "use_lower_fallback": True,
        "use_cap": False,
        "use_high_extrapolation": True,
    },

    "no_monotonicity": {
        "description": "No monotonicity constraint. Uses linear ridge entropy-to-temperature mapping instead of isotonic.",
        "use_monotonicity": False,
        "use_lower_fallback": True,
        "use_cap": True,
        "use_high_extrapolation": True,
    },

    "no_extrapolation": {
        "description": "No high-entropy extrapolation. Above-support entropy is clipped to edge prediction.",
        "use_monotonicity": True,
        "use_lower_fallback": True,
        "use_cap": True,
        "use_high_extrapolation": False,
    },
}

v3_ablation_config_path = DIRS["v3_ablations"] / "v3_ablation_configs.json"

with open(v3_ablation_config_path, "w") as f:
    json.dump(V3_ABLATION_CONFIGS, f, indent=2)

print("Saved V3 ablation configs:", v3_ablation_config_path)
print(json.dumps(V3_ABLATION_CONFIGS, indent=2))

**Ablation fitting helpers**

In [ ]:
from sklearn.linear_model import LinearRegression

def fit_v3_ablation_model(
    train_df: pd.DataFrame,
    source_T: float,
    ablation_name: str,
    ablation_config: dict,
    frozen_config: dict = FROZEN_V3_CONFIG,
    entropy_col: str = "entropy_mean",
    target_T_col: str = "source_bank_target_temperature",
):
    required = [entropy_col, target_T_col]
    missing = [c for c in required if c not in train_df.columns]
    assert len(missing) == 0, f"Missing columns for V3 ablation: {missing}"

    x_train = train_df[entropy_col].to_numpy(dtype=np.float64)
    y_train = train_df[target_T_col].to_numpy(dtype=np.float64)

    assert np.all(y_train > 0), "Training target temperatures must be positive"

    n_bins = int(frozen_config["n_entropy_bins"])
    support_margin_std = float(frozen_config["support_margin_std"])
    t_max_absolute = float(frozen_config["t_max_absolute"])
    t_max_additive = float(frozen_config["t_max_additive"])
    max_extrapolation_slope = float(frozen_config["max_extrapolation_slope"])

    T_min = float(source_T)

    if ablation_config["use_cap"]:
        T_max_safe = float(min(t_max_absolute, source_T + t_max_additive))
    else:
        # Deliberately loose cap for numerical safety only.
        T_max_safe = 10.0

    x_min = float(np.min(x_train))
    x_max = float(np.max(x_train))
    x_mean = float(np.mean(x_train))
    x_std = float(np.std(x_train, ddof=0))

    support_low = x_min - support_margin_std * x_std
    support_high = x_max + support_margin_std * x_std

    if ablation_config["use_monotonicity"]:
        binned = make_entropy_bins(
            train_df=train_df,
            entropy_col=entropy_col,
            target_T_col=target_T_col,
            n_bins=n_bins,
        )

        x_fit = binned["entropy_bin_mean"].to_numpy(dtype=np.float64)
        y_fit = binned["target_T_bin_mean"].to_numpy(dtype=np.float64)

        y_safe = np.clip(y_fit, T_min, T_max_safe)

        model = IsotonicRegression(
            increasing=True,
            out_of_bounds="clip",
            y_min=T_min,
            y_max=T_max_safe,
        )
        model.fit(x_fit, y_safe)

        model_type = "isotonic"

        if len(x_fit) >= 2:
            last_dx = float(x_fit[-1] - x_fit[-2])
            last_dy = float(model.predict([x_fit[-1]])[0] - model.predict([x_fit[-2]])[0])

            if abs(last_dx) < 1e-12:
                raw_slope = 0.0
            else:
                raw_slope = last_dy / last_dx
        else:
            raw_slope = 0.0

        T_at_train_max = float(model.predict([x_max])[0])

    else:
        # Non-monotone ablation: linear regression on raw source-bank condition points.
        # We predict log-temperature for positivity, then exponentiate at prediction time.
        X = x_train.reshape(-1, 1)
        y_log = np.log(np.clip(y_train, 1e-8, None))

        model = LinearRegression()
        model.fit(X, y_log)

        model_type = "linear_log_temperature"

        binned = make_entropy_bins(
            train_df=train_df,
            entropy_col=entropy_col,
            target_T_col=target_T_col,
            n_bins=n_bins,
        )

        # Slope in temperature space near train max for extrapolation bookkeeping.
        eps = max(1e-6, 0.01 * max(x_std, 1e-6))
        T_xmax = float(np.exp(model.predict([[x_max]])[0]))
        T_xprev = float(np.exp(model.predict([[x_max - eps]])[0]))
        raw_slope = (T_xmax - T_xprev) / eps
        T_at_train_max = T_xmax

    extrapolation_slope = float(np.clip(raw_slope, 0.0, max_extrapolation_slope))

    model_info = {
        "ablation_name": ablation_name,
        "ablation_config": dict(ablation_config),
        "frozen_config": dict(frozen_config),

        "model": model,
        "model_type": model_type,
        "binned": binned,

        "source_T": float(source_T),
        "T_min": float(T_min),
        "T_max_safe": float(T_max_safe),

        "entropy_train_min": x_min,
        "entropy_train_max": x_max,
        "entropy_train_mean": x_mean,
        "entropy_train_std": x_std,
        "support_low": float(support_low),
        "support_high": float(support_high),

        "T_at_train_max": float(T_at_train_max),
        "raw_extrapolation_slope": float(raw_slope),
        "extrapolation_slope": float(extrapolation_slope),

        "n_train_conditions": int(len(train_df)),
        "n_bins_used": int(len(binned)),
    }

    return model_info


def predict_v3_ablation_temperature(
    model_info: dict,
    entropy_value: float,
):
    entropy_value = float(entropy_value)

    model = model_info["model"]
    model_type = model_info["model_type"]
    ablation_config = model_info["ablation_config"]

    source_T = float(model_info["source_T"])
    T_min = float(model_info["T_min"])
    T_max_safe = float(model_info["T_max_safe"])

    support_low = float(model_info["support_low"])
    support_high = float(model_info["support_high"])
    entropy_train_max = float(model_info["entropy_train_max"])

    below_support = entropy_value < support_low
    above_support = entropy_value > support_high
    inside_support = (not below_support) and (not above_support)

    used_lower_fallback = False
    used_high_extrapolation = False

    def model_predict_T(x_value: float) -> float:
        if model_type == "isotonic":
            return float(model.predict([x_value])[0])

        if model_type == "linear_log_temperature":
            logT = float(model.predict([[x_value]])[0])
            return float(np.exp(logT))

        raise ValueError(f"Unknown model_type: {model_type}")

    if below_support and ablation_config["use_lower_fallback"]:
        raw_pred_T = source_T
        pred_T = source_T
        used_lower_fallback = True

    elif above_support and ablation_config["use_high_extrapolation"]:
        base_T = float(model_info["T_at_train_max"])
        slope = float(model_info["extrapolation_slope"])

        raw_pred_T = base_T + slope * (entropy_value - entropy_train_max)
        pred_T = raw_pred_T
        used_high_extrapolation = True

    else:
        # Inside support, below support without fallback, or above support without extrapolation.
        # Isotonic clips out-of-bounds automatically; linear predicts directly.
        raw_pred_T = model_predict_T(entropy_value)
        pred_T = raw_pred_T

    # Always protect positivity. Cap only if cap ablation allows it.
    pred_T = max(float(pred_T), 1e-6)

    if ablation_config["use_cap"]:
        pred_T = float(np.clip(pred_T, T_min, T_max_safe))
    else:
        # Numerical safety only, not the V3 safety cap.
        pred_T = float(np.clip(pred_T, 1e-6, 10.0))

    if used_lower_fallback:
        support_region = "lower_fallback"
    elif used_high_extrapolation:
        support_region = "high_extrapolation"
    else:
        support_region = "inside_support"

    return {
        "predicted_T": float(pred_T),
        "raw_predicted_T": float(raw_pred_T),

        "support_region": support_region,
        "used_lower_fallback": bool(used_lower_fallback),
        "used_high_extrapolation": bool(used_high_extrapolation),
        "inside_support": bool(inside_support),
        "below_support": bool(below_support),
        "above_support": bool(above_support),
    }


print("V3 ablation helpers ready.")

**Fit/evaluate all ablations in main setting**

In [ ]:
v3_ablation_pred_rows = []
v3_ablation_eval_rows = []
v3_ablation_model_rows = []
v3_ablation_bin_rows = []

for ablation_name, ablation_config in V3_ABLATION_CONFIGS.items():
    print(f"\n{'='*100}")
    print(f"Running V3 ablation: {ablation_name}")
    print(f"{'='*100}")

    for backbone in PROTOCOL["backbones"]:
        for seed in PROTOCOL["seeds"]:
            print(f"Ablation={ablation_name} | {backbone} | seed={seed}")

            source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

            train_df = source_bank_stream_df[
                (source_bank_stream_df["backbone"] == backbone) &
                (source_bank_stream_df["seed"] == seed)
            ].copy()

            target_df = target_stream_df[
                (target_stream_df["backbone"] == backbone) &
                (target_stream_df["seed"] == seed)
            ].copy()

            assert len(train_df) == 50
            assert len(target_df) == 50

            model_info = fit_v3_ablation_model(
                train_df=train_df,
                source_T=source_T,
                ablation_name=ablation_name,
                ablation_config=ablation_config,
                frozen_config=FROZEN_V3_CONFIG,
            )

            v3_ablation_model_rows.append({
                "ablation_name": ablation_name,
                "ablation_description": ablation_config["description"],
                "split_name": "main_all_source_bank_train",
                "information_regime": "unlabeled_target_stream_safe_v3_ablation",
                "backbone": backbone,
                "seed": int(seed),
                "source_T": float(source_T),
                "model_type": model_info["model_type"],
                "T_min": model_info["T_min"],
                "T_max_safe": model_info["T_max_safe"],
                "entropy_train_min": model_info["entropy_train_min"],
                "entropy_train_max": model_info["entropy_train_max"],
                "entropy_train_mean": model_info["entropy_train_mean"],
                "entropy_train_std": model_info["entropy_train_std"],
                "support_low": model_info["support_low"],
                "support_high": model_info["support_high"],
                "T_at_train_max": model_info["T_at_train_max"],
                "raw_extrapolation_slope": model_info["raw_extrapolation_slope"],
                "extrapolation_slope": model_info["extrapolation_slope"],
                "n_train_conditions": model_info["n_train_conditions"],
                "n_bins_used": model_info["n_bins_used"],
                "use_monotonicity": bool(ablation_config["use_monotonicity"]),
                "use_lower_fallback": bool(ablation_config["use_lower_fallback"]),
                "use_cap": bool(ablation_config["use_cap"]),
                "use_high_extrapolation": bool(ablation_config["use_high_extrapolation"]),
            })

            binned = model_info["binned"].copy()
            binned["ablation_name"] = ablation_name
            binned["backbone"] = backbone
            binned["seed"] = int(seed)
            binned["source_T"] = float(source_T)
            binned["T_max_safe"] = model_info["T_max_safe"]
            v3_ablation_bin_rows.append(binned)

            for _, row in target_df.iterrows():
                entropy_mean = float(row["entropy_mean"])

                pred = predict_v3_ablation_temperature(
                    model_info=model_info,
                    entropy_value=entropy_mean,
                )

                T = float(pred["predicted_T"])

                v3_ablation_pred_rows.append({
                    "ablation_name": ablation_name,
                    "ablation_description": ablation_config["description"],
                    "split_name": "main_all_source_bank_train",
                    "heldout_type": "none",
                    "heldout_value": "none",
                    "information_regime": "unlabeled_target_stream_safe_v3_ablation",
                    "backbone": backbone,
                    "seed": int(seed),
                    "method": f"v3_ablation_{ablation_name}",
                    "corruption": row["corruption"],
                    "severity": int(row["severity"]),
                    "condition": row["condition"],
                    "entropy_mean": entropy_mean,
                    "predicted_T": T,
                    "raw_predicted_T": float(pred["raw_predicted_T"]),
                    "source_T": float(source_T),
                    "support_region": pred["support_region"],
                    "used_lower_fallback": bool(pred["used_lower_fallback"]),
                    "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                    "inside_support": bool(pred["inside_support"]),
                    "below_support": bool(pred["below_support"]),
                    "above_support": bool(pred["above_support"]),
                })

                v3_ablation_eval_rows.extend(
                    evaluate_temperature_on_target_condition(
                        backbone=backbone,
                        seed=seed,
                        corruption=row["corruption"],
                        severity=int(row["severity"]),
                        method=f"v3_ablation_{ablation_name}",
                        information_regime="unlabeled_target_stream_safe_v3_ablation",
                        temperature=T,
                        split_name="main_all_source_bank_train",
                        heldout_type="none",
                        heldout_value="none",
                    )
                )

v3_ablation_predictions = pd.DataFrame(v3_ablation_pred_rows)
v3_ablation_eval_long = pd.DataFrame(v3_ablation_eval_rows)
v3_ablation_model_info = pd.DataFrame(v3_ablation_model_rows)
v3_ablation_bins = pd.concat(v3_ablation_bin_rows, ignore_index=True)

expected_ablation_predictions = (
    len(V3_ABLATION_CONFIGS)
    * len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(v3_ablation_predictions) == expected_ablation_predictions, (
    f"Expected {expected_ablation_predictions} ablation predictions, got {len(v3_ablation_predictions)}"
)

assert v3_ablation_predictions["predicted_T"].gt(0).all(), "Ablation predicted non-positive T"

expected_ablation_eval_rows = expected_ablation_predictions * len(PROTOCOL["metrics"])

assert len(v3_ablation_eval_long) == expected_ablation_eval_rows, (
    f"Expected {expected_ablation_eval_rows} eval rows, got {len(v3_ablation_eval_long)}"
)

v3_ablation_pred_path = DIRS["v3_ablations"] / "v3_ablation_main_predictions.csv"
v3_ablation_eval_path = DIRS["v3_ablations"] / "v3_ablation_main_eval_long.csv"
v3_ablation_model_path = DIRS["v3_ablations"] / "v3_ablation_main_model_info.csv"
v3_ablation_bins_path = DIRS["v3_ablations"] / "v3_ablation_main_entropy_bins.csv"

v3_ablation_predictions.to_csv(v3_ablation_pred_path, index=False)
v3_ablation_eval_long.to_csv(v3_ablation_eval_path, index=False)
v3_ablation_model_info.to_csv(v3_ablation_model_path, index=False)
v3_ablation_bins.to_csv(v3_ablation_bins_path, index=False)

print("Saved V3 ablation outputs:")
print(v3_ablation_pred_path)
print(v3_ablation_eval_path)
print(v3_ablation_model_path)
print(v3_ablation_bins_path)

display(v3_ablation_predictions.head())
display(v3_ablation_model_info.head())

**Ablation deltas vs source TS**

In [ ]:
v3_ablation_condition_wide = eval_long_to_condition_wide(v3_ablation_eval_long)
v3_ablation_delta_wide = add_corrected_source_ts_deltas(v3_ablation_condition_wide)

# Merge prediction/support metadata
merge_keys = [
    "split_name",
    "heldout_type",
    "heldout_value",
    "backbone",
    "seed",
    "method",
    "corruption",
    "severity",
    "condition",
]

support_cols = merge_keys + [
    "ablation_name",
    "ablation_description",
    "entropy_mean",
    "predicted_T",
    "raw_predicted_T",
    "source_T",
    "support_region",
    "used_lower_fallback",
    "used_high_extrapolation",
    "inside_support",
    "below_support",
    "above_support",
]

support_meta = v3_ablation_predictions[support_cols].copy()

dup_support = support_meta.duplicated(subset=merge_keys).sum()
assert dup_support == 0, f"Duplicate ablation support metadata rows: {dup_support}"

v3_ablation_delta_wide = v3_ablation_delta_wide.merge(
    support_meta,
    on=merge_keys,
    how="left",
    suffixes=("", "_pred"),
)

assert len(v3_ablation_delta_wide) == len(v3_ablation_condition_wide), (
    "Ablation support metadata merge changed row count."
)

assert v3_ablation_delta_wide["support_region"].notna().all(), (
    "Missing ablation support metadata after merge"
)

assert np.allclose(
    v3_ablation_delta_wide["temperature"].to_numpy(dtype=float),
    v3_ablation_delta_wide["predicted_T"].to_numpy(dtype=float),
    atol=1e-10,
), "Mismatch between evaluated ablation temperature and predicted_T"

assert v3_ablation_delta_wide["ece15"].between(0, 1).all()
assert v3_ablation_delta_wide["aece15"].between(0, 1).all()
assert v3_ablation_delta_wide["source_ts_ece15"].between(0, 1).all()
assert v3_ablation_delta_wide["source_ts_aece15"].between(0, 1).all()

v3_ablation_delta_path = DIRS["v3_ablations"] / "v3_ablation_main_delta_vs_source_ts_wide.csv"
v3_ablation_delta_wide.to_csv(v3_ablation_delta_path, index=False)

print("Saved V3 ablation delta wide:", v3_ablation_delta_path)
print("Shape:", v3_ablation_delta_wide.shape)

display(v3_ablation_delta_wide.head())

**Ablation summary table**

In [ ]:
v3_ablation_summary = (
    v3_ablation_delta_wide
    .groupby(
        [
            "split_name",
            "information_regime",
            "backbone",
            "method",
            "ablation_name",
        ],
        as_index=False,
    )
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),

        source_ts_nll_mean=("source_ts_nll", "mean"),
        source_ts_brier_mean=("source_ts_brier", "mean"),
        source_ts_ece15_mean=("source_ts_ece15", "mean"),
        source_ts_aece15_mean=("source_ts_aece15", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        predicted_T_min=("predicted_T", "min"),
        predicted_T_max=("predicted_T", "max"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),

        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

v3_ablation_summary_path = DIRS["v3_ablations"] / "v3_ablation_main_delta_summary.csv"
v3_ablation_summary.to_csv(v3_ablation_summary_path, index=False)

print("Saved V3 ablation summary:", v3_ablation_summary_path)

display(
    v3_ablation_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Ablation harm/safety summary**

In [ ]:
v3_ablation_harm_summary = (
    v3_ablation_delta_wide
    .groupby(["backbone", "ablation_name", "method"], as_index=False)
    .agg(
        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        median_delta_nll=("delta_nll_vs_source_ts", "median"),
        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),

        improved_conditions=("delta_nll_vs_source_ts", lambda x: int((x < -1e-12).sum())),
        harmed_conditions=("delta_nll_vs_source_ts", lambda x: int((x > 1e-12).sum())),
        near_zero_conditions=("delta_nll_vs_source_ts", lambda x: int((np.abs(x) <= 1e-12).sum())),
        n_conditions=("delta_nll_vs_source_ts", "count"),

        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        predicted_T_min=("predicted_T", "min"),
        predicted_T_max=("predicted_T", "max"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),
    )
)

v3_ablation_harm_summary["improved_fraction"] = (
    v3_ablation_harm_summary["improved_conditions"] /
    v3_ablation_harm_summary["n_conditions"]
)

v3_ablation_harm_summary["harmed_fraction"] = (
    v3_ablation_harm_summary["harmed_conditions"] /
    v3_ablation_harm_summary["n_conditions"]
)

v3_ablation_harm_path = DIRS["v3_ablations"] / "v3_ablation_main_harm_summary.csv"
v3_ablation_harm_summary.to_csv(v3_ablation_harm_path, index=False)

print("Saved V3 ablation harm summary:", v3_ablation_harm_path)

display(
    v3_ablation_harm_summary
    .sort_values(["backbone", "mean_delta_nll"])
    .round(6)
)

**Ablation support/extrapolation summary**

In [ ]:
v3_ablation_support_summary = (
    v3_ablation_delta_wide
    .groupby(["backbone", "ablation_name", "support_region"], as_index=False)
    .agg(
        count=("delta_nll_vs_source_ts", "count"),
        mean_predicted_T=("predicted_T", "mean"),
        min_predicted_T=("predicted_T", "min"),
        max_predicted_T=("predicted_T", "max"),
        mean_entropy=("entropy_mean", "mean"),

        mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
        mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
        mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
        mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
        mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

        worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
        best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),
    )
)

v3_ablation_support_summary["fraction"] = (
    v3_ablation_support_summary["count"] /
    v3_ablation_support_summary.groupby(["backbone", "ablation_name"])["count"].transform("sum")
)

frac_check = v3_ablation_support_summary.groupby(["backbone", "ablation_name"])["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), (
    "Ablation support fractions do not sum to 1"
)

v3_ablation_support_path = DIRS["v3_ablations"] / "v3_ablation_main_support_summary.csv"
v3_ablation_support_summary.to_csv(v3_ablation_support_path, index=False)

print("Saved V3 ablation support summary:", v3_ablation_support_path)

display(
    v3_ablation_support_summary
    .sort_values(["backbone", "ablation_name", "support_region"])
    .round(6)
)

**Bootstrap CIs for ablations**

In [ ]:
v3_ablation_ci = make_ci_summary(
    df=v3_ablation_delta_wide,
    group_cols=[
        "split_name",
        "information_regime",
        "backbone",
        "method",
        "ablation_name",
    ],
    value_cols=DELTA_COLS,
    seed_offset=500,
)

v3_ablation_ci_path = DIRS["stats"] / "v3_ablation_main_delta_bootstrap_ci.csv"
v3_ablation_ci.to_csv(v3_ablation_ci_path, index=False)

print("Saved V3 ablation CI:", v3_ablation_ci_path)

display(
    v3_ablation_ci
    .sort_values(["backbone", "quantity", "mean"])
    .round(6)
    .head(80)
)

**Wide CI table**

In [ ]:
v3_ablation_ci_wide_rows = []

for key, group in v3_ablation_ci.groupby(
    [
        "split_name",
        "information_regime",
        "backbone",
        "method",
        "ablation_name",
    ]
):
    split_name, regime, backbone, method, ablation_name = key

    row = {
        "split_name": split_name,
        "information_regime": regime,
        "backbone": backbone,
        "method": method,
        "ablation_name": ablation_name,
    }

    for _, r in group.iterrows():
        q = r["quantity"]
        row[f"{q}_mean"] = r["mean"]
        row[f"{q}_ci_low"] = r["ci_low"]
        row[f"{q}_ci_high"] = r["ci_high"]
        row[f"{q}_n"] = r["n_units"]

    v3_ablation_ci_wide_rows.append(row)

v3_ablation_ci_wide = pd.DataFrame(v3_ablation_ci_wide_rows)

v3_ablation_ci_wide_path = DIRS["stats"] / "v3_ablation_main_delta_bootstrap_ci_wide.csv"
v3_ablation_ci_wide.to_csv(v3_ablation_ci_wide_path, index=False)

print("Saved V3 ablation wide CI:", v3_ablation_ci_wide_path)

display(
    v3_ablation_ci_wide
    .sort_values(["backbone", "ablation_name"])
    .round(6)
)

**LOCO/LOSO V3 ablations**

In [ ]:
def run_v3_ablation_condition_split(split_name: str):
    assert split_name in ["leave_one_corruption_family_out", "leave_one_severity_out"]

    eval_rows = []
    pred_rows = []
    model_rows = []
    bin_rows = []

    if split_name == "leave_one_corruption_family_out":
        heldout_values = PROTOCOL["corruptions"]
        heldout_type = "corruption"
    else:
        heldout_values = PROTOCOL["severities"]
        heldout_type = "severity"

    for ablation_name, ablation_config in V3_ABLATION_CONFIGS.items():
        print(f"\n{'='*100}")
        print(f"Running V3 ablation split={split_name} | ablation={ablation_name}")
        print(f"{'='*100}")

        for backbone in PROTOCOL["backbones"]:
            for seed in PROTOCOL["seeds"]:
                print(f"Ablation={ablation_name} | {split_name} | {backbone} | seed={seed}")

                source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

                source_bank_seed_df = source_bank_stream_df[
                    (source_bank_stream_df["backbone"] == backbone) &
                    (source_bank_stream_df["seed"] == seed)
                ].copy()

                target_seed_df = target_stream_df[
                    (target_stream_df["backbone"] == backbone) &
                    (target_stream_df["seed"] == seed)
                ].copy()

                assert len(source_bank_seed_df) == 50
                assert len(target_seed_df) == 50

                for heldout_value in heldout_values:
                    if split_name == "leave_one_corruption_family_out":
                        train_df = source_bank_seed_df[
                            source_bank_seed_df["corruption"] != heldout_value
                        ].copy()

                        test_target_df = target_seed_df[
                            target_seed_df["corruption"] == heldout_value
                        ].copy()

                        assert len(train_df) == 45
                        assert len(test_target_df) == 5

                    else:
                        train_df = source_bank_seed_df[
                            source_bank_seed_df["severity"] != heldout_value
                        ].copy()

                        test_target_df = target_seed_df[
                            target_seed_df["severity"] == heldout_value
                        ].copy()

                        assert len(train_df) == 40
                        assert len(test_target_df) == 10

                    model_info = fit_v3_ablation_model(
                        train_df=train_df,
                        source_T=source_T,
                        ablation_name=ablation_name,
                        ablation_config=ablation_config,
                        frozen_config=FROZEN_V3_CONFIG,
                    )

                    model_rows.append({
                        "ablation_name": ablation_name,
                        "ablation_description": ablation_config["description"],
                        "split_name": split_name,
                        "heldout_type": heldout_type,
                        "heldout_value": str(heldout_value),
                        "information_regime": "unlabeled_target_stream_safe_v3_ablation_condition_split",
                        "backbone": backbone,
                        "seed": int(seed),
                        "source_T": float(source_T),
                        "model_type": model_info["model_type"],
                        "T_min": model_info["T_min"],
                        "T_max_safe": model_info["T_max_safe"],
                        "entropy_train_min": model_info["entropy_train_min"],
                        "entropy_train_max": model_info["entropy_train_max"],
                        "entropy_train_mean": model_info["entropy_train_mean"],
                        "entropy_train_std": model_info["entropy_train_std"],
                        "support_low": model_info["support_low"],
                        "support_high": model_info["support_high"],
                        "T_at_train_max": model_info["T_at_train_max"],
                        "raw_extrapolation_slope": model_info["raw_extrapolation_slope"],
                        "extrapolation_slope": model_info["extrapolation_slope"],
                        "n_train_conditions": model_info["n_train_conditions"],
                        "n_bins_used": model_info["n_bins_used"],
                        "use_monotonicity": bool(ablation_config["use_monotonicity"]),
                        "use_lower_fallback": bool(ablation_config["use_lower_fallback"]),
                        "use_cap": bool(ablation_config["use_cap"]),
                        "use_high_extrapolation": bool(ablation_config["use_high_extrapolation"]),
                    })

                    binned = model_info["binned"].copy()
                    binned["ablation_name"] = ablation_name
                    binned["split_name"] = split_name
                    binned["heldout_type"] = heldout_type
                    binned["heldout_value"] = str(heldout_value)
                    binned["backbone"] = backbone
                    binned["seed"] = int(seed)
                    binned["source_T"] = float(source_T)
                    binned["T_max_safe"] = model_info["T_max_safe"]
                    bin_rows.append(binned)

                    for _, row in test_target_df.iterrows():
                        entropy_mean = float(row["entropy_mean"])

                        pred = predict_v3_ablation_temperature(
                            model_info=model_info,
                            entropy_value=entropy_mean,
                        )

                        T = float(pred["predicted_T"])

                        pred_rows.append({
                            "ablation_name": ablation_name,
                            "ablation_description": ablation_config["description"],
                            "split_name": split_name,
                            "heldout_type": heldout_type,
                            "heldout_value": str(heldout_value),
                            "information_regime": "unlabeled_target_stream_safe_v3_ablation_condition_split",
                            "backbone": backbone,
                            "seed": int(seed),
                            "method": f"v3_ablation_{ablation_name}",
                            "corruption": row["corruption"],
                            "severity": int(row["severity"]),
                            "condition": row["condition"],
                            "entropy_mean": entropy_mean,
                            "predicted_T": T,
                            "raw_predicted_T": float(pred["raw_predicted_T"]),
                            "source_T": float(source_T),
                            "support_region": pred["support_region"],
                            "used_lower_fallback": bool(pred["used_lower_fallback"]),
                            "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                            "inside_support": bool(pred["inside_support"]),
                            "below_support": bool(pred["below_support"]),
                            "above_support": bool(pred["above_support"]),
                        })

                        eval_rows.extend(
                            evaluate_temperature_on_target_condition(
                                backbone=backbone,
                                seed=seed,
                                corruption=row["corruption"],
                                severity=int(row["severity"]),
                                method=f"v3_ablation_{ablation_name}",
                                information_regime="unlabeled_target_stream_safe_v3_ablation_condition_split",
                                temperature=T,
                                split_name=split_name,
                                heldout_type=heldout_type,
                                heldout_value=str(heldout_value),
                            )
                        )

    eval_long = pd.DataFrame(eval_rows)
    predictions = pd.DataFrame(pred_rows)
    model_info_df = pd.DataFrame(model_rows)
    bins_df = pd.concat(bin_rows, ignore_index=True)

    return eval_long, predictions, model_info_df, bins_df


v3_ablation_loco_eval_long, v3_ablation_loco_predictions, v3_ablation_loco_model_info, v3_ablation_loco_bins = run_v3_ablation_condition_split(
    "leave_one_corruption_family_out"
)

v3_ablation_loso_eval_long, v3_ablation_loso_predictions, v3_ablation_loso_model_info, v3_ablation_loso_bins = run_v3_ablation_condition_split(
    "leave_one_severity_out"
)

v3_ablation_loco_eval_path = DIRS["v3_ablations"] / "v3_ablation_loco_eval_long.csv"
v3_ablation_loco_pred_path = DIRS["v3_ablations"] / "v3_ablation_loco_predictions.csv"
v3_ablation_loco_model_path = DIRS["v3_ablations"] / "v3_ablation_loco_model_info.csv"
v3_ablation_loco_bins_path = DIRS["v3_ablations"] / "v3_ablation_loco_entropy_bins.csv"

v3_ablation_loso_eval_path = DIRS["v3_ablations"] / "v3_ablation_loso_eval_long.csv"
v3_ablation_loso_pred_path = DIRS["v3_ablations"] / "v3_ablation_loso_predictions.csv"
v3_ablation_loso_model_path = DIRS["v3_ablations"] / "v3_ablation_loso_model_info.csv"
v3_ablation_loso_bins_path = DIRS["v3_ablations"] / "v3_ablation_loso_entropy_bins.csv"

v3_ablation_loco_eval_long.to_csv(v3_ablation_loco_eval_path, index=False)
v3_ablation_loco_predictions.to_csv(v3_ablation_loco_pred_path, index=False)
v3_ablation_loco_model_info.to_csv(v3_ablation_loco_model_path, index=False)
v3_ablation_loco_bins.to_csv(v3_ablation_loco_bins_path, index=False)

v3_ablation_loso_eval_long.to_csv(v3_ablation_loso_eval_path, index=False)
v3_ablation_loso_predictions.to_csv(v3_ablation_loso_pred_path, index=False)
v3_ablation_loso_model_info.to_csv(v3_ablation_loso_model_path, index=False)
v3_ablation_loso_bins.to_csv(v3_ablation_loso_bins_path, index=False)

print("Saved V3 ablation LOCO/LOSO outputs:")
print(v3_ablation_loco_eval_path)
print(v3_ablation_loco_pred_path)
print(v3_ablation_loso_eval_path)
print(v3_ablation_loso_pred_path)

**Add LOCO/LOSO ablation deltas and summaries**

In [ ]:
v3_ablation_loco_condition_wide = eval_long_to_condition_wide(v3_ablation_loco_eval_long)
v3_ablation_loso_condition_wide = eval_long_to_condition_wide(v3_ablation_loso_eval_long)

v3_ablation_loco_delta_wide = add_corrected_source_ts_deltas(v3_ablation_loco_condition_wide)
v3_ablation_loso_delta_wide = add_corrected_source_ts_deltas(v3_ablation_loso_condition_wide)


def merge_v3_ablation_prediction_meta_for_split(
    delta_wide: pd.DataFrame,
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge V3 ablation prediction metadata into LOCO/LOSO delta tables.

    Important:
    V3 ablations have multiple methods for the same condition, so 'method'
    must be included in the merge key. Otherwise duplicate metadata rows occur.
    """
    merge_keys = [
        "split_name",
        "heldout_type",
        "heldout_value",
        "backbone",
        "seed",
        "method",
        "corruption",
        "severity",
        "condition",
    ]

    support_cols = merge_keys + [
        "ablation_name",
        "ablation_description",
        "entropy_mean",
        "predicted_T",
        "raw_predicted_T",
        "source_T",
        "support_region",
        "used_lower_fallback",
        "used_high_extrapolation",
        "inside_support",
        "below_support",
        "above_support",
    ]

    missing_cols = [c for c in support_cols if c not in predictions.columns]
    assert len(missing_cols) == 0, f"Missing columns in ablation predictions: {missing_cols}"

    support_meta = predictions[support_cols].copy()

    dup_support = support_meta.duplicated(subset=merge_keys).sum()
    assert dup_support == 0, f"Duplicate V3 ablation metadata rows: {dup_support}"

    out = delta_wide.merge(
        support_meta,
        on=merge_keys,
        how="left",
        suffixes=("", "_pred"),
    )

    assert len(out) == len(delta_wide), "Ablation metadata merge changed row count."
    assert out["support_region"].notna().all(), "Missing ablation metadata after merge"

    assert np.allclose(
        out["temperature"].to_numpy(dtype=float),
        out["predicted_T"].to_numpy(dtype=float),
        atol=1e-10,
    ), "Mismatch between evaluated temperature and predicted_T"

    return out


v3_ablation_loco_delta_wide = merge_v3_ablation_prediction_meta_for_split(
    delta_wide=v3_ablation_loco_delta_wide,
    predictions=v3_ablation_loco_predictions,
)

v3_ablation_loso_delta_wide = merge_v3_ablation_prediction_meta_for_split(
    delta_wide=v3_ablation_loso_delta_wide,
    predictions=v3_ablation_loso_predictions,
)

v3_ablation_loco_delta_path = DIRS["v3_ablations"] / "v3_ablation_loco_delta_vs_source_ts_wide.csv"
v3_ablation_loso_delta_path = DIRS["v3_ablations"] / "v3_ablation_loso_delta_vs_source_ts_wide.csv"

v3_ablation_loco_delta_wide.to_csv(v3_ablation_loco_delta_path, index=False)
v3_ablation_loso_delta_wide.to_csv(v3_ablation_loso_delta_path, index=False)


def summarize_v3_ablation_delta_wide(delta_wide: pd.DataFrame) -> pd.DataFrame:
    summary = (
        delta_wide
        .groupby(
            [
                "split_name",
                "heldout_type",
                "information_regime",
                "backbone",
                "method",
                "ablation_name",
            ],
            as_index=False,
        )
        .agg(
            delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
            delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
            delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
            delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
            delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
            delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

            nll_mean=("nll", "mean"),
            brier_mean=("brier", "mean"),
            ece15_mean=("ece15", "mean"),
            aece15_mean=("aece15", "mean"),

            source_ts_nll_mean=("source_ts_nll", "mean"),
            source_ts_brier_mean=("source_ts_brier", "mean"),
            source_ts_ece15_mean=("source_ts_ece15", "mean"),
            source_ts_aece15_mean=("source_ts_aece15", "mean"),

            predicted_T_mean=("predicted_T", "mean"),
            predicted_T_min=("predicted_T", "min"),
            predicted_T_max=("predicted_T", "max"),

            lower_fallback_rate=("used_lower_fallback", "mean"),
            high_extrapolation_rate=("used_high_extrapolation", "mean"),
            inside_support_rate=("inside_support", "mean"),

            n_conditions=("delta_nll_vs_source_ts", "count"),
        )
    )

    assert summary["ece15_mean"].between(0, 1).all()
    assert summary["source_ts_ece15_mean"].between(0, 1).all()

    return summary


v3_ablation_loco_summary = summarize_v3_ablation_delta_wide(v3_ablation_loco_delta_wide)
v3_ablation_loso_summary = summarize_v3_ablation_delta_wide(v3_ablation_loso_delta_wide)

v3_ablation_loco_summary_path = DIRS["v3_ablations"] / "v3_ablation_loco_delta_summary.csv"
v3_ablation_loso_summary_path = DIRS["v3_ablations"] / "v3_ablation_loso_delta_summary.csv"

v3_ablation_loco_summary.to_csv(v3_ablation_loco_summary_path, index=False)
v3_ablation_loso_summary.to_csv(v3_ablation_loso_summary_path, index=False)

print("Saved V3 ablation LOCO/LOSO delta outputs:")
print(v3_ablation_loco_delta_path)
print(v3_ablation_loso_delta_path)
print(v3_ablation_loco_summary_path)
print(v3_ablation_loso_summary_path)

print("\nV3 ablation LOCO summary:")
display(
    v3_ablation_loco_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

print("\nV3 ablation LOSO summary:")
display(
    v3_ablation_loso_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Add combined harm summary for main + LOCO + LOSO**

In [ ]:
def make_v3_ablation_harm_summary(delta_wide: pd.DataFrame, block_name: str) -> pd.DataFrame:
    harm = (
        delta_wide
        .groupby(["split_name", "backbone", "ablation_name", "method"], as_index=False)
        .agg(
            mean_delta_nll=("delta_nll_vs_source_ts", "mean"),
            median_delta_nll=("delta_nll_vs_source_ts", "median"),
            worst_harm_delta_nll=("delta_nll_vs_source_ts", "max"),
            best_improvement_delta_nll=("delta_nll_vs_source_ts", "min"),

            improved_conditions=("delta_nll_vs_source_ts", lambda x: int((x < -1e-12).sum())),
            harmed_conditions=("delta_nll_vs_source_ts", lambda x: int((x > 1e-12).sum())),
            near_zero_conditions=("delta_nll_vs_source_ts", lambda x: int((np.abs(x) <= 1e-12).sum())),
            n_conditions=("delta_nll_vs_source_ts", "count"),

            mean_delta_brier=("delta_brier_vs_source_ts", "mean"),
            mean_delta_ece15=("delta_ece15_vs_source_ts", "mean"),
            mean_delta_aece15=("delta_aece15_vs_source_ts", "mean"),
            mean_delta_abs_gap=("delta_abs_signed_gap_vs_source_ts", "mean"),

            predicted_T_mean=("predicted_T", "mean"),
            predicted_T_min=("predicted_T", "min"),
            predicted_T_max=("predicted_T", "max"),

            lower_fallback_rate=("used_lower_fallback", "mean"),
            high_extrapolation_rate=("used_high_extrapolation", "mean"),
            inside_support_rate=("inside_support", "mean"),
        )
    )

    harm["improved_fraction"] = harm["improved_conditions"] / harm["n_conditions"]
    harm["harmed_fraction"] = harm["harmed_conditions"] / harm["n_conditions"]
    harm["ablation_block"] = block_name

    return harm


v3_ablation_main_harm_for_combined = make_v3_ablation_harm_summary(
    v3_ablation_delta_wide,
    block_name="main",
)

v3_ablation_loco_harm = make_v3_ablation_harm_summary(
    v3_ablation_loco_delta_wide,
    block_name="loco",
)

v3_ablation_loso_harm = make_v3_ablation_harm_summary(
    v3_ablation_loso_delta_wide,
    block_name="loso",
)

v3_ablation_combined_harm_summary = pd.concat(
    [
        v3_ablation_main_harm_for_combined,
        v3_ablation_loco_harm,
        v3_ablation_loso_harm,
    ],
    ignore_index=True,
)

v3_ablation_combined_harm_path = DIRS["v3_ablations"] / "v3_ablation_harm_summary.csv"
v3_ablation_combined_harm_summary.to_csv(v3_ablation_combined_harm_path, index=False)

print("Saved combined V3 ablation harm summary:", v3_ablation_combined_harm_path)

display(
    v3_ablation_combined_harm_summary
    .sort_values(["backbone", "ablation_block", "mean_delta_nll"])
    .round(6)
)

**Add LOCO/LOSO ablation CIs**

In [ ]:
v3_ablation_loco_ci = make_ci_summary(
    df=v3_ablation_loco_delta_wide,
    group_cols=[
        "split_name",
        "heldout_type",
        "information_regime",
        "backbone",
        "method",
        "ablation_name",
    ],
    value_cols=DELTA_COLS,
    seed_offset=510,
)

v3_ablation_loso_ci = make_ci_summary(
    df=v3_ablation_loso_delta_wide,
    group_cols=[
        "split_name",
        "heldout_type",
        "information_regime",
        "backbone",
        "method",
        "ablation_name",
    ],
    value_cols=DELTA_COLS,
    seed_offset=520,
)

v3_ablation_loco_ci_path = DIRS["stats"] / "v3_ablation_loco_delta_bootstrap_ci.csv"
v3_ablation_loso_ci_path = DIRS["stats"] / "v3_ablation_loso_delta_bootstrap_ci.csv"

v3_ablation_loco_ci.to_csv(v3_ablation_loco_ci_path, index=False)
v3_ablation_loso_ci.to_csv(v3_ablation_loso_ci_path, index=False)

print("Saved V3 ablation LOCO CI:", v3_ablation_loco_ci_path)
print("Saved V3 ablation LOSO CI:", v3_ablation_loso_ci_path)

display(v3_ablation_loco_ci.round(6).head(40))
display(v3_ablation_loso_ci.round(6).head(40))

**Combined main-method + ablation table**

In [ ]:
v3_ablation_main_for_combined = v3_ablation_summary.copy()
v3_ablation_main_for_combined["ablation_block"] = "main"

v3_ablation_loco_for_combined = v3_ablation_loco_summary.copy()
v3_ablation_loco_for_combined["ablation_block"] = "loco"

v3_ablation_loso_for_combined = v3_ablation_loso_summary.copy()
v3_ablation_loso_for_combined["ablation_block"] = "loso"

stage5_combined_ablation_summary = pd.concat(
    [
        v3_ablation_main_for_combined,
        v3_ablation_loco_for_combined,
        v3_ablation_loso_for_combined,
    ],
    ignore_index=True,
    sort=False,
)

stage5_combined_ablation_path = DIRS["tables"] / "stage5_v3_ablation_main_loco_loso_summary.csv"
stage5_combined_ablation_summary.to_csv(stage5_combined_ablation_path, index=False)

compact_ablation_cols = [
    "ablation_block",
    "split_name",
    "heldout_type",
    "backbone",
    "ablation_name",
    "method",
    "delta_nll_mean",
    "delta_brier_mean",
    "delta_ece15_mean",
    "delta_aece15_mean",
    "delta_abs_signed_gap_mean",
    "predicted_T_mean",
    "predicted_T_min",
    "predicted_T_max",
    "lower_fallback_rate",
    "high_extrapolation_rate",
    "inside_support_rate",
    "n_conditions",
]

stage5_compact_ablation_summary = stage5_combined_ablation_summary[
    [c for c in compact_ablation_cols if c in stage5_combined_ablation_summary.columns]
].copy()

stage5_compact_ablation_path = DIRS["tables"] / "stage5_compact_v3_ablation_summary.csv"
stage5_compact_ablation_summary.to_csv(stage5_compact_ablation_path, index=False)

print("Saved Stage 5 combined ablation summary:", stage5_combined_ablation_path)
print("Saved Stage 5 compact ablation summary:", stage5_compact_ablation_path)

display(
    stage5_compact_ablation_summary
    .sort_values(["backbone", "ablation_block", "delta_nll_mean"])
    .round(6)
)

In [ ]:
stage5_note = {
    "stage": "stage_5_v3_ablations_main_loco_loso",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "main_outputs": {
        "v3_ablation_configs": str(v3_ablation_config_path),

        "v3_ablation_main_predictions": str(v3_ablation_pred_path),
        "v3_ablation_main_eval_long": str(v3_ablation_eval_path),
        "v3_ablation_main_model_info": str(v3_ablation_model_path),
        "v3_ablation_main_entropy_bins": str(v3_ablation_bins_path),
        "v3_ablation_main_delta_wide": str(v3_ablation_delta_path),
        "v3_ablation_main_summary": str(v3_ablation_summary_path),
        "v3_ablation_main_harm_summary": str(v3_ablation_harm_path),
        "v3_ablation_main_support_summary": str(v3_ablation_support_path),
        "v3_ablation_main_bootstrap_ci": str(v3_ablation_ci_path),
        "v3_ablation_main_bootstrap_ci_wide": str(v3_ablation_ci_wide_path),

        "v3_ablation_loco_eval_long": str(v3_ablation_loco_eval_path),
        "v3_ablation_loco_predictions": str(v3_ablation_loco_pred_path),
        "v3_ablation_loco_delta_wide": str(v3_ablation_loco_delta_path),
        "v3_ablation_loco_summary": str(v3_ablation_loco_summary_path),
        "v3_ablation_loco_bootstrap_ci": str(v3_ablation_loco_ci_path),

        "v3_ablation_loso_eval_long": str(v3_ablation_loso_eval_path),
        "v3_ablation_loso_predictions": str(v3_ablation_loso_pred_path),
        "v3_ablation_loso_delta_wide": str(v3_ablation_loso_delta_path),
        "v3_ablation_loso_summary": str(v3_ablation_loso_summary_path),
        "v3_ablation_loso_bootstrap_ci": str(v3_ablation_loso_ci_path),

        "v3_ablation_combined_harm_summary": str(v3_ablation_combined_harm_path),
        "stage5_combined_ablation_summary": str(stage5_combined_ablation_path),
        "stage5_compact_ablation_summary": str(stage5_compact_ablation_path),
    },

    "checks": {
        "ablations_run": list(V3_ABLATION_CONFIGS.keys()),
        "main_ablations_done": True,
        "loco_ablations_done": True,
        "loso_ablations_done": True,
        "target_labels_used_only_for_evaluation": True,
        "deltas_use_corrected_refit_source_ts": True,
        "ece15_values_valid": True,
        "accuracy_invariant_checked": True,
        "harm_summary_created_for_main_loco_loso": True,
        "compact_ablation_table_created": True,
    },

    "important_note": (
        "Stage 5 evaluates whether V3 safety components are necessary: fallback, cap, monotonicity, "
        "and high-entropy extrapolation. Ablations are now evaluated in the main all-source-bank setting, "
        "leave-one-corruption-family-out, and leave-one-severity-out."
    ),
}

stage5_note_path = DIRS["logs"] / "stage5_v3_ablations_main_loco_loso_completion.json"

with open(stage5_note_path, "w") as f:
    json.dump(stage5_note, f, indent=2)

print("Saved Stage 5 completion note:", stage5_note_path)
print(json.dumps(stage5_note, indent=2))

display(
    stage5_compact_ablation_summary[
        [
            "ablation_block",
            "split_name",
            "backbone",
            "ablation_name",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "predicted_T_mean",
            "predicted_T_min",
            "predicted_T_max",
            "lower_fallback_rate",
            "high_extrapolation_rate",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "ablation_block", "delta_nll_mean"])
    .round(6)
)

**Stage 6**

**Run stream baselines in main all-source-bank setting**

In [ ]:
stream_main_eval_rows = []
stream_main_pred_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"\nMain stream baselines | {backbone} | seed={seed}")

        source_bank_seed_df = source_bank_stream_df[
            (source_bank_stream_df["backbone"] == backbone) &
            (source_bank_stream_df["seed"] == seed)
        ].copy()

        target_seed_df = target_stream_df[
            (target_stream_df["backbone"] == backbone) &
            (target_stream_df["seed"] == seed)
        ].copy()

        assert len(source_bank_seed_df) == 50
        assert len(target_seed_df) == 50

        for method, features in STREAM_FEATURE_SETS.items():
            model = fit_stream_model_by_method(
                method=method,
                train_df=source_bank_seed_df,
                features=features,
                seed=seed,
            )

            pred_T, pred_logT = predict_stream_temperature_model(
                model=model,
                test_df=target_seed_df,
                feature_names=features,
                t_min=0.8,
                t_max=2.5,
            )

            target_seed_df_reset = target_seed_df.reset_index(drop=True)

            for i, row in target_seed_df_reset.iterrows():
                corruption = row["corruption"]
                severity = int(row["severity"])
                condition = row["condition"]
                T = float(pred_T[i])

                stream_main_pred_rows.append({
                    "split_name": "main_all_source_bank_train",
                    "heldout_type": "none",
                    "heldout_value": "none",
                    "information_regime": "unlabeled_target_stream_main",
                    "backbone": backbone,
                    "seed": int(seed),
                    "method": method,
                    "corruption": corruption,
                    "severity": severity,
                    "condition": condition,
                    "predicted_T": T,
                    "predicted_logT": float(pred_logT[i]),
                })

                stream_main_eval_rows.extend(
                    evaluate_temperature_on_target_condition(
                        backbone=backbone,
                        seed=seed,
                        corruption=corruption,
                        severity=severity,
                        method=method,
                        information_regime="unlabeled_target_stream_main",
                        temperature=T,
                        split_name="main_all_source_bank_train",
                        heldout_type="none",
                        heldout_value="none",
                    )
                )

stream_main_eval_long = pd.DataFrame(stream_main_eval_rows)
stream_main_predictions = pd.DataFrame(stream_main_pred_rows)

expected_stream_main_predictions = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
    * len(STREAM_FEATURE_SETS)
)

assert len(stream_main_predictions) == expected_stream_main_predictions, (
    f"Expected {expected_stream_main_predictions} predictions, got {len(stream_main_predictions)}"
)

expected_stream_main_eval_rows = expected_stream_main_predictions * len(PROTOCOL["metrics"])

assert len(stream_main_eval_long) == expected_stream_main_eval_rows, (
    f"Expected {expected_stream_main_eval_rows} eval rows, got {len(stream_main_eval_long)}"
)

stream_main_eval_path = DIRS["stream_methods"] / "stream_baselines_main_eval_long.csv"
stream_main_pred_path = DIRS["stream_methods"] / "stream_baselines_main_predictions.csv"

stream_main_eval_long.to_csv(stream_main_eval_path, index=False)
stream_main_predictions.to_csv(stream_main_pred_path, index=False)

print("Saved stream main eval:", stream_main_eval_path)
print("Saved stream main predictions:", stream_main_pred_path)

display(stream_main_eval_long.head())
display(stream_main_predictions.head())

**Compute main stream baseline deltas vs corrected source TS**

In [ ]:
stream_main_condition_wide = eval_long_to_condition_wide(stream_main_eval_long)
stream_main_delta_wide = add_corrected_source_ts_deltas(stream_main_condition_wide)

stream_main_delta_path = DIRS["stream_methods"] / "stream_baselines_main_delta_vs_source_ts_wide.csv"
stream_main_delta_wide.to_csv(stream_main_delta_path, index=False)

stream_main_summary = summarize_delta_wide(stream_main_delta_wide)

stream_main_summary_path = DIRS["stream_methods"] / "stream_baselines_main_delta_summary.csv"
stream_main_summary.to_csv(stream_main_summary_path, index=False)

print("Saved stream main delta wide:", stream_main_delta_path)
print("Saved stream main summary:", stream_main_summary_path)

display(
    stream_main_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Build raw and source TS reference rows**

In [ ]:
reference_wide = corrected_condition_metric_wide.copy()

raw_wide = reference_wide[reference_wide["method"] == "raw"].copy()
source_wide = reference_wide[reference_wide["method"] == "source_global_ts"].copy()

required_metric_cols = [
    "accuracy",
    "mean_confidence",
    "signed_gap",
    "abs_signed_gap",
    "nll",
    "brier",
    "ece15",
    "aece15",
]

merge_keys = ["backbone", "seed", "corruption", "severity", "condition"]

source_ref = source_wide[
    merge_keys + required_metric_cols
].rename(columns={m: f"source_ts_{m}" for m in required_metric_cols})

dup_source_ref = source_ref.duplicated(subset=merge_keys).sum()
assert dup_source_ref == 0, f"Duplicate source reference rows: {dup_source_ref}"

raw_delta_wide = raw_wide.merge(
    source_ref,
    on=merge_keys,
    how="left",
)

assert len(raw_delta_wide) == len(raw_wide)
assert raw_delta_wide["source_ts_nll"].notna().all()

for metric in required_metric_cols:
    raw_delta_wide[f"delta_{metric}_vs_source_ts"] = (
        raw_delta_wide[metric] - raw_delta_wide[f"source_ts_{metric}"]
    )

# Source TS relative to itself gives zero deltas.
source_delta_wide = source_wide.merge(
    source_ref,
    on=merge_keys,
    how="left",
)

assert len(source_delta_wide) == len(source_wide)

for metric in required_metric_cols:
    source_delta_wide[f"delta_{metric}_vs_source_ts"] = 0.0

raw_delta_wide["split_name"] = "main_all_source_bank_train"
raw_delta_wide["heldout_type"] = "none"
raw_delta_wide["heldout_value"] = "none"

source_delta_wide["split_name"] = "main_all_source_bank_train"
source_delta_wide["heldout_type"] = "none"
source_delta_wide["heldout_value"] = "none"

raw_reference_delta_path = DIRS["tables"] / "raw_delta_vs_corrected_source_ts_wide.csv"
source_reference_delta_path = DIRS["tables"] / "source_ts_delta_vs_corrected_source_ts_wide.csv"

raw_delta_wide.to_csv(raw_reference_delta_path, index=False)
source_delta_wide.to_csv(source_reference_delta_path, index=False)

print("Saved raw delta wide:", raw_reference_delta_path)
print("Saved source TS delta wide:", source_reference_delta_path)

**Summarize raw/source reference rows**

In [ ]:
def summarize_reference_delta_wide(delta_wide: pd.DataFrame) -> pd.DataFrame:
    summary = (
        delta_wide
        .groupby(["split_name", "heldout_type", "information_regime", "backbone", "method"], as_index=False)
        .agg(
            delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
            delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
            delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
            delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
            delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
            delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

            nll_mean=("nll", "mean"),
            brier_mean=("brier", "mean"),
            ece15_mean=("ece15", "mean"),
            aece15_mean=("aece15", "mean"),

            source_ts_nll_mean=("source_ts_nll", "mean"),
            source_ts_brier_mean=("source_ts_brier", "mean"),
            source_ts_ece15_mean=("source_ts_ece15", "mean"),
            source_ts_aece15_mean=("source_ts_aece15", "mean"),

            temperature_mean=("temperature", "mean"),
            temperature_min=("temperature", "min"),
            temperature_max=("temperature", "max"),

            n_conditions=("delta_nll_vs_source_ts", "count"),
        )
    )

    assert summary["ece15_mean"].between(0, 1).all()
    assert summary["source_ts_ece15_mean"].between(0, 1).all()

    return summary


raw_reference_summary = summarize_reference_delta_wide(raw_delta_wide)
source_reference_summary = summarize_reference_delta_wide(source_delta_wide)

raw_reference_summary_path = DIRS["tables"] / "raw_delta_vs_corrected_source_ts_summary.csv"
source_reference_summary_path = DIRS["tables"] / "source_ts_delta_vs_corrected_source_ts_summary.csv"

raw_reference_summary.to_csv(raw_reference_summary_path, index=False)
source_reference_summary.to_csv(source_reference_summary_path, index=False)

print("Saved raw reference summary:", raw_reference_summary_path)
print("Saved source TS reference summary:", source_reference_summary_path)

display(raw_reference_summary.round(6))
display(source_reference_summary.round(6))

**Build main deployable comparison table**

In [ ]:
main_deployable_summary = pd.concat(
    [
        raw_reference_summary.assign(comparison_block="raw_reference"),
        source_reference_summary.assign(comparison_block="source_ts_reference"),
        stream_main_summary.assign(comparison_block="stream_baselines_main"),
        safe_v2_main_delta_summary.assign(comparison_block="safe_v2_main"),
        frozen_v3_main_delta_summary.assign(comparison_block="frozen_v3_main"),
    ],
    ignore_index=True,
    sort=False,
)

main_deployable_summary_path = DIRS["tables"] / "stage6_main_deployable_comparison_summary.csv"
main_deployable_summary.to_csv(main_deployable_summary_path, index=False)

print("Saved main deployable comparison summary:", main_deployable_summary_path)

display(
    main_deployable_summary[
        [
            "comparison_block",
            "split_name",
            "information_regime",
            "backbone",
            "method",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "temperature_mean",
            "temperature_min",
            "temperature_max",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Create wide delta table for all main deployable methods**

In [ ]:
raw_delta_for_combined = raw_delta_wide.copy()
source_delta_for_combined = source_delta_wide.copy()
stream_delta_for_combined = stream_main_delta_wide.copy()
safe_v2_delta_for_combined = safe_v2_delta_wide.copy()
v3_delta_for_combined = frozen_v3_delta_wide.copy()

raw_delta_for_combined["comparison_block"] = "raw_reference"
source_delta_for_combined["comparison_block"] = "source_ts_reference"
stream_delta_for_combined["comparison_block"] = "stream_baselines_main"
safe_v2_delta_for_combined["comparison_block"] = "safe_v2_main"
v3_delta_for_combined["comparison_block"] = "frozen_v3_main"

main_deployable_delta_wide = pd.concat(
    [
        raw_delta_for_combined,
        source_delta_for_combined,
        stream_delta_for_combined,
        safe_v2_delta_for_combined,
        v3_delta_for_combined,
    ],
    ignore_index=True,
    sort=False,
)

main_deployable_delta_path = DIRS["tables"] / "stage6_main_deployable_delta_vs_source_ts_wide.csv"
main_deployable_delta_wide.to_csv(main_deployable_delta_path, index=False)

print("Saved main deployable delta-wide table:", main_deployable_delta_path)
print("Shape:", main_deployable_delta_wide.shape)

display(main_deployable_delta_wide.head())

**Bootstrap CIs for main deployable comparison**

In [ ]:
stage6_main_ci = make_ci_summary(
    df=main_deployable_delta_wide,
    group_cols=[
        "comparison_block",
        "split_name",
        "information_regime",
        "backbone",
        "method",
    ],
    value_cols=DELTA_COLS,
    seed_offset=600,
)

stage6_main_ci_path = DIRS["stats"] / "stage6_main_deployable_delta_bootstrap_ci.csv"
stage6_main_ci.to_csv(stage6_main_ci_path, index=False)

print("Saved Stage 6 main deployable CI:", stage6_main_ci_path)

display(
    stage6_main_ci
    .sort_values(["backbone", "quantity", "mean"])
    .round(6)
    .head(80)
)

**Wide CI table**

In [ ]:
stage6_ci_wide_rows = []

for key, group in stage6_main_ci.groupby(
    [
        "comparison_block",
        "split_name",
        "information_regime",
        "backbone",
        "method",
    ]
):
    comparison_block, split_name, regime, backbone, method = key

    row = {
        "comparison_block": comparison_block,
        "split_name": split_name,
        "information_regime": regime,
        "backbone": backbone,
        "method": method,
    }

    for _, r in group.iterrows():
        q = r["quantity"]
        row[f"{q}_mean"] = r["mean"]
        row[f"{q}_ci_low"] = r["ci_low"]
        row[f"{q}_ci_high"] = r["ci_high"]
        row[f"{q}_n"] = r["n_units"]

    stage6_ci_wide_rows.append(row)

stage6_main_ci_wide = pd.DataFrame(stage6_ci_wide_rows)

stage6_main_ci_wide_path = DIRS["stats"] / "stage6_main_deployable_delta_bootstrap_ci_wide.csv"
stage6_main_ci_wide.to_csv(stage6_main_ci_wide_path, index=False)

print("Saved Stage 6 wide CI:", stage6_main_ci_wide_path)

display(
    stage6_main_ci_wide
    .sort_values(["backbone", "comparison_block", "method"])
    .round(6)
)

**Main comparison by severity**

In [ ]:
main_deployable_by_severity = (
    main_deployable_delta_wide
    .groupby(["comparison_block", "backbone", "method", "severity"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),
        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

main_deployable_by_severity_path = DIRS["tables"] / "stage6_main_deployable_delta_by_severity.csv"
main_deployable_by_severity.to_csv(main_deployable_by_severity_path, index=False)

print("Saved main deployable by severity:", main_deployable_by_severity_path)

display(
    main_deployable_by_severity
    .sort_values(["backbone", "severity", "delta_nll_mean"])
    .round(6)
    .head(80)
)

**Main comparison by corruption**

In [ ]:
main_deployable_by_corruption = (
    main_deployable_delta_wide
    .groupby(["comparison_block", "backbone", "method", "corruption"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),
        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

main_deployable_by_corruption_path = DIRS["tables"] / "stage6_main_deployable_delta_by_corruption.csv"
main_deployable_by_corruption.to_csv(main_deployable_by_corruption_path, index=False)

print("Saved main deployable by corruption:", main_deployable_by_corruption_path)

display(
    main_deployable_by_corruption
    .sort_values(["backbone", "corruption", "delta_nll_mean"])
    .round(6)
    .head(100)
)

**Compact paper-style table**

In [ ]:
compact_methods_order = [
    "raw",
    "source_global_ts",
    "stream_entropy_mean_only",
    "stream_entropy_stats",
    "stream_all_stats_ridge_no_class_entropy",
    "stream_all_stats_mlp_no_class_entropy",
    SAFE_V2_CONFIG["method"],
    FROZEN_V3_CONFIG["method"],
]

paper_compact = main_deployable_summary.copy()

paper_compact["method_order"] = paper_compact["method"].apply(
    lambda m: compact_methods_order.index(m) if m in compact_methods_order else 999
)

paper_compact = paper_compact.sort_values(
    ["backbone", "method_order"]
).reset_index(drop=True)

paper_compact_cols = [
    "backbone",
    "method",
    "delta_nll_mean",
    "delta_brier_mean",
    "delta_ece15_mean",
    "delta_aece15_mean",
    "delta_abs_signed_gap_mean",
    "nll_mean",
    "brier_mean",
    "ece15_mean",
    "aece15_mean",
    "temperature_mean",
    "temperature_min",
    "temperature_max",
    "n_conditions",
]

paper_compact = paper_compact[[c for c in paper_compact_cols if c in paper_compact.columns]]

paper_compact_path = DIRS["tables"] / "stage6_paper_compact_deployable_comparison.csv"
paper_compact.to_csv(paper_compact_path, index=False)

print("Saved compact paper-style deployable comparison:", paper_compact_path)

display(paper_compact.round(6))

**Plotting setup for main deployable comparison**

In [ ]:
import matplotlib.pyplot as plt

PLOT_DIR_STAGE6 = DIRS["figures"] / "stage6_main_deployable_comparison"
PLOT_DIR_STAGE6.mkdir(parents=True, exist_ok=True)

METHOD_LABELS = {
    "raw": "Raw",
    "source_global_ts": "Source TS",
    "stream_entropy_mean_only": "Entropy mean",
    "stream_entropy_stats": "Entropy stats",
    "stream_all_stats_ridge_no_class_entropy": "All stats ridge",
    "stream_all_stats_mlp_no_class_entropy": "All stats MLP",
    SAFE_V2_CONFIG["method"]: "Safe V2",
    FROZEN_V3_CONFIG["method"]: "Frozen V3",
}

PAPER_METHOD_ORDER = [
    "raw",
    "source_global_ts",
    "stream_entropy_mean_only",
    "stream_entropy_stats",
    "stream_all_stats_ridge_no_class_entropy",
    "stream_all_stats_mlp_no_class_entropy",
    SAFE_V2_CONFIG["method"],
    FROZEN_V3_CONFIG["method"],
]

print("Stage 6 plot directory:", PLOT_DIR_STAGE6)

**Bar plots for main metric deltas**

In [ ]:
plot_metrics = [
    ("delta_nll_mean", "Mean ΔNLL vs Source TS"),
    ("delta_brier_mean", "Mean ΔBrier vs Source TS"),
    ("delta_ece15_mean", "Mean ΔECE15 vs Source TS"),
    ("delta_aece15_mean", "Mean ΔAECE15 vs Source TS"),
    ("delta_abs_signed_gap_mean", "Mean Δ|Conf-Acc Gap| vs Source TS"),
]

plot_df = main_deployable_summary.copy()
plot_df["method_label"] = plot_df["method"].map(METHOD_LABELS).fillna(plot_df["method"])
plot_df["method_order"] = plot_df["method"].apply(
    lambda m: PAPER_METHOD_ORDER.index(m) if m in PAPER_METHOD_ORDER else 999
)

for backbone in PROTOCOL["backbones"]:
    sub = plot_df[plot_df["backbone"] == backbone].copy()
    sub = sub.sort_values("method_order")

    for metric_col, ylabel in plot_metrics:
        fig, ax = plt.subplots(figsize=(10, 5))

        ax.bar(sub["method_label"], sub[metric_col])
        ax.axhline(0.0, linestyle="--", linewidth=1)

        ax.set_title(f"{backbone}: {ylabel}")
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Method")
        ax.tick_params(axis="x", rotation=35)

        fig.tight_layout()

        out_path = PLOT_DIR_STAGE6 / f"{backbone}_{metric_col}_bar.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.show()

        print("Saved:", out_path)

**Severity trend plots**

In [ ]:
severity_plot_df = main_deployable_by_severity.copy()
severity_plot_df["method_label"] = severity_plot_df["method"].map(METHOD_LABELS).fillna(severity_plot_df["method"])
severity_plot_df["method_order"] = severity_plot_df["method"].apply(
    lambda m: PAPER_METHOD_ORDER.index(m) if m in PAPER_METHOD_ORDER else 999
)

severity_metrics = [
    ("delta_nll_mean", "Mean ΔNLL vs Source TS"),
    ("delta_brier_mean", "Mean ΔBrier vs Source TS"),
    ("delta_ece15_mean", "Mean ΔECE15 vs Source TS"),
    ("delta_aece15_mean", "Mean ΔAECE15 vs Source TS"),
]

for backbone in PROTOCOL["backbones"]:
    sub_backbone = severity_plot_df[severity_plot_df["backbone"] == backbone].copy()

    for metric_col, ylabel in severity_metrics:
        fig, ax = plt.subplots(figsize=(9, 5))

        for method in PAPER_METHOD_ORDER:
            sub_method = sub_backbone[sub_backbone["method"] == method].copy()
            if len(sub_method) == 0:
                continue

            sub_method = sub_method.sort_values("severity")
            label = METHOD_LABELS.get(method, method)

            ax.plot(
                sub_method["severity"],
                sub_method[metric_col],
                marker="o",
                label=label,
            )

        ax.axhline(0.0, linestyle="--", linewidth=1)
        ax.set_title(f"{backbone}: {ylabel} by severity")
        ax.set_xlabel("Severity")
        ax.set_ylabel(ylabel)
        ax.set_xticks(PROTOCOL["severities"])
        ax.legend(fontsize=8)

        fig.tight_layout()

        out_path = PLOT_DIR_STAGE6 / f"{backbone}_{metric_col}_by_severity.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.show()

        print("Saved:", out_path)

**Corruption heatmap-style table plots**

In [ ]:
def plot_corruption_metric_table(df, backbone, metric_col, title, out_path):
    sub = df[df["backbone"] == backbone].copy()
    sub["method_label"] = sub["method"].map(METHOD_LABELS).fillna(sub["method"])

    method_order_labels = [
        METHOD_LABELS[m] for m in PAPER_METHOD_ORDER
        if m in set(sub["method"])
    ]

    pivot = (
        sub
        .pivot_table(
            index="corruption",
            columns="method_label",
            values=metric_col,
            aggfunc="mean",
        )
        .reindex(index=PROTOCOL["corruptions"])
    )

    pivot = pivot[[c for c in method_order_labels if c in pivot.columns]]

    fig, ax = plt.subplots(figsize=(11, 6))
    im = ax.imshow(pivot.values, aspect="auto")

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")

    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    ax.set_title(title)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metric_col)

    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


for backbone in PROTOCOL["backbones"]:
    for metric_col, label in [
        ("delta_nll_mean", "Mean ΔNLL vs Source TS"),
        ("delta_ece15_mean", "Mean ΔECE15 vs Source TS"),
        ("delta_abs_signed_gap_mean", "Mean Δ|Conf-Acc Gap| vs Source TS"),
    ]:
        out_path = PLOT_DIR_STAGE6 / f"{backbone}_{metric_col}_by_corruption_heatmap.png"
        plot_corruption_metric_table(
            df=main_deployable_by_corruption,
            backbone=backbone,
            metric_col=metric_col,
            title=f"{backbone}: {label} by corruption",
            out_path=out_path,
        )

**Figure manifest for Stage 6**

In [ ]:
stage6_figure_rows = []

for path in sorted(PLOT_DIR_STAGE6.glob("*.png")):
    stage6_figure_rows.append({
        "stage": "stage_6",
        "figure_name": path.name,
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "role": "main deployable comparison plot",
    })

stage6_figure_manifest = pd.DataFrame(stage6_figure_rows)

stage6_figure_manifest_path = DIRS["manifests"] / "stage6_figure_manifest.csv"
stage6_figure_manifest.to_csv(stage6_figure_manifest_path, index=False)

print("Saved Stage 6 figure manifest:", stage6_figure_manifest_path)
display(stage6_figure_manifest)

**Stream-size sensitivity config**

In [ ]:
STREAM_SIZE_SENSITIVITY_CONFIG = {
    "method": FROZEN_V3_CONFIG["method"],
    "stream_sizes": [32, 64, 128, 256, 512, 1024, 2048, 4096],
    "n_repeats": 5,
    "sampling": "without_replacement_if_possible",
    "evaluation": "temperature_predicted_from_subset_entropy_but_evaluated_on_full_target_condition",
    "target_labels_used_for_prediction": False,
    "target_labels_used_for_evaluation": True,
    "note": (
        "This experiment tests how many unlabeled target samples are needed before "
        "the stream entropy estimate and the resulting V3 temperature become stable."
    ),
}

stream_size_config_path = DIRS["configs"] / "stream_size_sensitivity_config.json"

with open(stream_size_config_path, "w") as f:
    json.dump(STREAM_SIZE_SENSITIVITY_CONFIG, f, indent=2)

print("Saved stream-size sensitivity config:", stream_size_config_path)
print(json.dumps(STREAM_SIZE_SENSITIVITY_CONFIG, indent=2))

**Stream-size sensitivity helpers**

In [ ]:
def stream_descriptors_from_logits_subset(
    logits: torch.Tensor,
    requested_size: int,
    rng: np.random.Generator,
):
    logits = logits.detach().cpu().float()
    n_total = logits.shape[0]

    actual_size = int(min(requested_size, n_total))

    if actual_size < n_total:
        idx = rng.choice(n_total, size=actual_size, replace=False)
        idx = torch.as_tensor(idx, dtype=torch.long)
        subset_logits = logits[idx]
    else:
        subset_logits = logits

    desc = stream_descriptors_from_logits(subset_logits)
    desc["requested_stream_size"] = int(requested_size)
    desc["actual_stream_size"] = int(actual_size)
    desc["full_condition_size"] = int(n_total)

    return desc


def build_frozen_v3_model_cache():
    model_cache = {}

    for backbone in PROTOCOL["backbones"]:
        for seed in PROTOCOL["seeds"]:
            source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

            train_df = source_bank_stream_df[
                (source_bank_stream_df["backbone"] == backbone) &
                (source_bank_stream_df["seed"] == seed)
            ].copy()

            assert len(train_df) == 50

            model_info = fit_frozen_v3_model(
                train_df=train_df,
                source_T=source_T,
                config=FROZEN_V3_CONFIG,
            )

            model_cache[(backbone, seed)] = model_info

    return model_cache


frozen_v3_stream_size_model_cache = build_frozen_v3_model_cache()

print("Built frozen V3 model cache for stream-size sensitivity.")

**Run stream-size sensitivity experiment**

In [ ]:
stream_size_rows = []

stream_sizes = STREAM_SIZE_SENSITIVITY_CONFIG["stream_sizes"]
n_repeats = int(STREAM_SIZE_SENSITIVITY_CONFIG["n_repeats"])

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"\nStream-size sensitivity | {backbone} | seed={seed}")

        source_T = SOURCE_T_BY_BACKBONE[backbone][seed]
        model_info = frozen_v3_stream_size_model_cache[(backbone, seed)]

        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                logits, labels = load_cached_logits(
                    backbone=backbone,
                    seed=seed,
                    corruption=corruption,
                    severity=severity,
                )

                source_ts_logits = apply_temperature(logits, source_T)
                source_ts_metrics = metrics_from_logits(source_ts_logits, labels)

                for requested_size in stream_sizes:
                    for repeat_id in range(n_repeats):
                        rng_seed = (
                            10_000
                            + 1_000 * int(seed)
                            + 100 * int(severity)
                            + 10 * repeat_id
                            + abs(hash((backbone, corruption, requested_size))) % 10_000
                        )

                        rng = np.random.default_rng(rng_seed)

                        desc = stream_descriptors_from_logits_subset(
                            logits=logits,
                            requested_size=requested_size,
                            rng=rng,
                        )

                        pred = predict_frozen_v3_temperature(
                            model_info=model_info,
                            entropy_value=float(desc["entropy_mean"]),
                        )

                        predicted_T = float(pred["predicted_T"])

                        scaled_logits = apply_temperature(logits, predicted_T)
                        met = metrics_from_logits(scaled_logits, labels)

                        row = {
                            "backbone": backbone,
                            "seed": int(seed),
                            "corruption": corruption,
                            "severity": int(severity),
                            "condition": f"{corruption}_sev{severity}",

                            "requested_stream_size": int(requested_size),
                            "actual_stream_size": int(desc["actual_stream_size"]),
                            "full_condition_size": int(desc["full_condition_size"]),
                            "repeat_id": int(repeat_id),
                            "rng_seed": int(rng_seed),

                            "method": FROZEN_V3_CONFIG["method"],
                            "source_T": float(source_T),
                            "predicted_T": predicted_T,
                            "raw_predicted_T": float(pred["raw_predicted_T"]),
                            "entropy_mean_subset": float(desc["entropy_mean"]),
                            "entropy_std_subset": float(desc["entropy_std"]),

                            "support_region": pred["support_region"],
                            "used_lower_fallback": bool(pred["used_lower_fallback"]),
                            "used_high_extrapolation": bool(pred["used_high_extrapolation"]),
                            "inside_support": bool(pred["inside_support"]),
                            "below_support": bool(pred["below_support"]),
                            "above_support": bool(pred["above_support"]),

                            "accuracy": float(met["accuracy"]),
                            "mean_confidence": float(met["mean_confidence"]),
                            "signed_gap": float(met["signed_gap"]),
                            "abs_signed_gap": float(met["abs_signed_gap"]),
                            "nll": float(met["nll"]),
                            "brier": float(met["brier"]),
                            "ece15": float(met["ece15"]),
                            "aece15": float(met["aece15"]),

                            "source_ts_accuracy": float(source_ts_metrics["accuracy"]),
                            "source_ts_mean_confidence": float(source_ts_metrics["mean_confidence"]),
                            "source_ts_signed_gap": float(source_ts_metrics["signed_gap"]),
                            "source_ts_abs_signed_gap": float(source_ts_metrics["abs_signed_gap"]),
                            "source_ts_nll": float(source_ts_metrics["nll"]),
                            "source_ts_brier": float(source_ts_metrics["brier"]),
                            "source_ts_ece15": float(source_ts_metrics["ece15"]),
                            "source_ts_aece15": float(source_ts_metrics["aece15"]),
                        }

                        for metric in required_metric_cols:
                            row[f"delta_{metric}_vs_source_ts"] = (
                                row[metric] - row[f"source_ts_{metric}"]
                            )

                        stream_size_rows.append(row)

                del logits, labels

stream_size_sensitivity_wide = pd.DataFrame(stream_size_rows)

expected_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
    * len(stream_sizes)
    * n_repeats
)

assert len(stream_size_sensitivity_wide) == expected_rows, (
    f"Expected {expected_rows}, got {len(stream_size_sensitivity_wide)}"
)

assert stream_size_sensitivity_wide["predicted_T"].gt(0).all()
assert stream_size_sensitivity_wide["ece15"].between(0, 1).all()
assert stream_size_sensitivity_wide["aece15"].between(0, 1).all()
assert stream_size_sensitivity_wide["source_ts_ece15"].between(0, 1).all()
assert stream_size_sensitivity_wide["source_ts_aece15"].between(0, 1).all()

assert (
    stream_size_sensitivity_wide["delta_accuracy_vs_source_ts"].abs() < 1e-10
).all(), "Temperature scaling changed accuracy. This should not happen."

stream_size_wide_path = DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_wide.csv"
stream_size_sensitivity_wide.to_csv(stream_size_wide_path, index=False)

print("Saved stream-size sensitivity wide table:", stream_size_wide_path)
print("Shape:", stream_size_sensitivity_wide.shape)

display(stream_size_sensitivity_wide.head())

**Stream-size sensitivity summaries**

In [ ]:
stream_size_summary = (
    stream_size_sensitivity_wide
    .groupby(["backbone", "requested_stream_size"], as_index=False)
    .agg(
        actual_stream_size_mean=("actual_stream_size", "mean"),

        predicted_T_mean=("predicted_T", "mean"),
        predicted_T_std=("predicted_T", "std"),
        predicted_T_min=("predicted_T", "min"),
        predicted_T_max=("predicted_T", "max"),

        entropy_mean=("entropy_mean_subset", "mean"),
        entropy_std_across_repeats=("entropy_mean_subset", "std"),

        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_nll_std=("delta_nll_vs_source_ts", "std"),

        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_brier_std=("delta_brier_vs_source_ts", "std"),

        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_ece15_std=("delta_ece15_vs_source_ts", "std"),

        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_aece15_std=("delta_aece15_vs_source_ts", "std"),

        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),
        delta_abs_signed_gap_std=("delta_abs_signed_gap_vs_source_ts", "std"),

        lower_fallback_rate=("used_lower_fallback", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        inside_support_rate=("inside_support", "mean"),

        n_units=("predicted_T", "count"),
    )
)

stream_size_by_severity = (
    stream_size_sensitivity_wide
    .groupby(["backbone", "requested_stream_size", "severity"], as_index=False)
    .agg(
        predicted_T_mean=("predicted_T", "mean"),
        predicted_T_std=("predicted_T", "std"),
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        n_units=("predicted_T", "count"),
    )
)

stream_size_by_support = (
    stream_size_sensitivity_wide
    .groupby(["backbone", "requested_stream_size", "support_region"], as_index=False)
    .agg(
        count=("predicted_T", "count"),
        predicted_T_mean=("predicted_T", "mean"),
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
    )
)

stream_size_by_support["fraction"] = (
    stream_size_by_support["count"] /
    stream_size_by_support.groupby(["backbone", "requested_stream_size"])["count"].transform("sum")
)

frac_check = stream_size_by_support.groupby(["backbone", "requested_stream_size"])["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), "Support fractions do not sum to 1."

stream_size_summary_path = DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_summary.csv"
stream_size_by_severity_path = DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_by_severity.csv"
stream_size_by_support_path = DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_by_support_region.csv"

stream_size_summary.to_csv(stream_size_summary_path, index=False)
stream_size_by_severity.to_csv(stream_size_by_severity_path, index=False)
stream_size_by_support.to_csv(stream_size_by_support_path, index=False)

print("Saved stream-size summaries:")
print(stream_size_summary_path)
print(stream_size_by_severity_path)
print(stream_size_by_support_path)

display(stream_size_summary.round(6))
display(stream_size_by_severity.round(6).head(30))
display(stream_size_by_support.round(6).head(30))

**Stream-size sensitivity plots**

In [ ]:
PLOT_DIR_STREAM_SIZE = DIRS["figures"] / "stage6_stream_size_sensitivity"
PLOT_DIR_STREAM_SIZE.mkdir(parents=True, exist_ok=True)

stream_size_plot_metrics = [
    ("predicted_T_mean", "Predicted temperature"),
    ("predicted_T_std", "Predicted temperature std"),
    ("entropy_std_across_repeats", "Entropy mean std across repeats"),
    ("delta_nll_mean", "Mean ΔNLL vs Source TS"),
    ("delta_ece15_mean", "Mean ΔECE15 vs Source TS"),
    ("delta_abs_signed_gap_mean", "Mean Δ|Conf-Acc Gap| vs Source TS"),
    ("high_extrapolation_rate", "High-extrapolation rate"),
]

for backbone in PROTOCOL["backbones"]:
    sub = stream_size_summary[
        stream_size_summary["backbone"] == backbone
    ].copy()

    sub = sub.sort_values("requested_stream_size")

    for metric_col, ylabel in stream_size_plot_metrics:
        fig, ax = plt.subplots(figsize=(7, 5))

        ax.plot(
            sub["requested_stream_size"],
            sub[metric_col],
            marker="o",
        )

        if metric_col.startswith("delta_"):
            ax.axhline(0.0, linestyle="--", linewidth=1)

        ax.set_xscale("log", base=2)
        ax.set_xticks(STREAM_SIZE_SENSITIVITY_CONFIG["stream_sizes"])
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

        ax.set_xlabel("Unlabeled stream size")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{backbone}: stream-size sensitivity — {ylabel}")

        fig.tight_layout()

        out_path = PLOT_DIR_STREAM_SIZE / f"{backbone}_stream_size_{metric_col}.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.show()

        print("Saved:", out_path)


# Severity trend plot for ΔNLL by stream size
for backbone in PROTOCOL["backbones"]:
    sub = stream_size_by_severity[
        stream_size_by_severity["backbone"] == backbone
    ].copy()

    fig, ax = plt.subplots(figsize=(8, 5))

    for severity in PROTOCOL["severities"]:
        ssub = sub[sub["severity"] == severity].sort_values("requested_stream_size")

        ax.plot(
            ssub["requested_stream_size"],
            ssub["delta_nll_mean"],
            marker="o",
            label=f"severity {severity}",
        )

    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xscale("log", base=2)
    ax.set_xticks(STREAM_SIZE_SENSITIVITY_CONFIG["stream_sizes"])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    ax.set_xlabel("Unlabeled stream size")
    ax.set_ylabel("Mean ΔNLL vs Source TS")
    ax.set_title(f"{backbone}: ΔNLL by stream size and severity")
    ax.legend(fontsize=8)

    fig.tight_layout()

    out_path = PLOT_DIR_STREAM_SIZE / f"{backbone}_stream_size_delta_nll_by_severity.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)


stream_size_figure_rows = []

for path in sorted(PLOT_DIR_STREAM_SIZE.glob("*.png")):
    stream_size_figure_rows.append({
        "stage": "stage_6_stream_size_sensitivity",
        "figure_name": path.name,
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "role": "stream-size sensitivity plot",
    })

stream_size_figure_manifest = pd.DataFrame(stream_size_figure_rows)

stream_size_figure_manifest_path = DIRS["manifests"] / "stage6_stream_size_figure_manifest.csv"
stream_size_figure_manifest.to_csv(stream_size_figure_manifest_path, index=False)

print("Saved stream-size figure manifest:", stream_size_figure_manifest_path)
display(stream_size_figure_manifest)

In [ ]:
stage6_note = {
    "stage": "stage_6_closest_deployable_baseline_comparison",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "main_outputs": {
        "stream_main_eval_long": str(stream_main_eval_path),
        "stream_main_predictions": str(stream_main_pred_path),
        "stream_main_delta_wide": str(stream_main_delta_path),
        "stream_main_summary": str(stream_main_summary_path),

        "raw_delta_wide": str(raw_reference_delta_path),
        "source_ts_delta_wide": str(source_reference_delta_path),
        "raw_summary": str(raw_reference_summary_path),
        "source_ts_summary": str(source_reference_summary_path),

        "safe_v2_summary": str(safe_v2_summary_path),
        "frozen_v3_summary": str(v3_summary_path),

        "main_deployable_summary": str(main_deployable_summary_path),
        "main_deployable_delta_wide": str(main_deployable_delta_path),
        "main_deployable_bootstrap_ci": str(stage6_main_ci_path),
        "main_deployable_bootstrap_ci_wide": str(stage6_main_ci_wide_path),
        "main_deployable_by_severity": str(main_deployable_by_severity_path),
        "main_deployable_by_corruption": str(main_deployable_by_corruption_path),
        "paper_compact_deployable_comparison": str(paper_compact_path),
        "stage6_figure_manifest": str(stage6_figure_manifest_path),

        "stream_size_config": str(stream_size_config_path),
        "stream_size_sensitivity_wide": str(stream_size_wide_path),
        "stream_size_sensitivity_summary": str(stream_size_summary_path),
        "stream_size_sensitivity_by_severity": str(stream_size_by_severity_path),
        "stream_size_sensitivity_by_support": str(stream_size_by_support_path),
        "stream_size_figure_manifest": str(stream_size_figure_manifest_path),
    },

    "checks": {
        "main_stream_baselines_evaluated": True,
        "stream_entropy_mean_only_treated_as_adaptive_entropy_baseline": True,
        "stream_mlp_baseline_included": True,
        "raw_and_source_ts_reference_included": True,
        "safe_v2_included": True,
        "frozen_v3_included": True,
        "deltas_use_corrected_refit_source_ts": True,
        "target_labels_used_only_for_evaluation": True,
        "ece15_values_valid": True,
        "accuracy_invariant_checked": True,
        "main_deployable_plots_created": True,
        "stream_size_sensitivity_done": True,
    },

    "important_note": (
        "Stage 6 provides the closest deployable baseline comparison. "
        "It includes raw, source TS, stream entropy/ridge/MLP baselines, Safe V2, and frozen V3. "
        "stream_entropy_mean_only is treated as the closest adaptive entropy baseline. "
        "Oracle target TS is excluded here because it is diagnostic only."
    ),
}

stage6_note_path = DIRS["logs"] / "stage6_closest_deployable_baseline_comparison_completion.json"

with open(stage6_note_path, "w") as f:
    json.dump(stage6_note, f, indent=2)

print("Saved Stage 6 completion note:", stage6_note_path)
print(json.dumps(stage6_note, indent=2))

display(paper_compact.round(6))

**stage 7**

**Oracle output folders and config**

In [ ]:
from scipy.stats import pearsonr, spearmanr

ORACLE_DIR = DIRS["oracle_diagnostics"]
ORACLE_FIG_DIR = DIRS["figures"] / "oracle_diagnostics"

ORACLE_DIR.mkdir(parents=True, exist_ok=True)
ORACLE_FIG_DIR.mkdir(parents=True, exist_ok=True)

ORACLE_CONFIG = {
    "stage": "stage_7_target_oracle_diagnostics",
    "oracle_status": "diagnostic_only_not_deployable",
    "uses_target_labels": True,
    "fit_from": "cached_target_logits_and_labels",
    "oracle_levels": [
        "target_global",
        "target_family",
        "target_family_severity",
    ],
    "temperature_fit_objective": "target_label_nll",
    "important_note": (
        "Oracle temperatures use target labels. They are only diagnostic references "
        "for studying condition-dependent calibration structure. They must not be "
        "presented as deployable calibration methods."
    ),
}

oracle_config_path = DIRS["configs"] / "stage7_oracle_diagnostics_config.json"

with open(oracle_config_path, "w") as f:
    json.dump(ORACLE_CONFIG, f, indent=2)

print("Saved oracle diagnostics config:", oracle_config_path)
print(json.dumps(ORACLE_CONFIG, indent=2))

**Helper to concatenate target logits by group**

In [ ]:
def collect_target_logits_labels(
    backbone: str,
    seed: int,
    corruptions,
    severities,
):
    logits_list = []
    labels_list = []
    meta_rows = []

    for corruption in corruptions:
        for severity in severities:
            logits, labels = load_cached_logits(
                backbone=backbone,
                seed=seed,
                corruption=corruption,
                severity=int(severity),
            )

            logits_list.append(logits)
            labels_list.append(labels)

            meta_rows.append({
                "backbone": backbone,
                "seed": int(seed),
                "corruption": corruption,
                "severity": int(severity),
                "condition": f"{corruption}_sev{severity}",
                "n_samples": int(len(labels)),
            })

            del logits, labels

    all_logits = torch.cat(logits_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)

    meta_df = pd.DataFrame(meta_rows)

    assert all_logits.ndim == 2
    assert all_labels.ndim == 1
    assert len(all_logits) == len(all_labels)
    assert all_logits.shape[1] == PROTOCOL["num_classes"]

    return all_logits, all_labels, meta_df


print("Oracle collection helpers ready.")

**Fit target-global, family-wise, and family×severity oracle temperatures**

In [ ]:
oracle_fit_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        print(f"\n{'='*100}")
        print(f"Fitting target-oracle diagnostics | {backbone} | seed={seed}")
        print(f"{'='*100}")

        source_T = SOURCE_T_BY_BACKBONE[backbone][seed]

        # ----------------------------------------------------
        # 1. Target-global oracle: one T for all target conditions
        # ----------------------------------------------------
        global_logits, global_labels, global_meta = collect_target_logits_labels(
            backbone=backbone,
            seed=seed,
            corruptions=PROTOCOL["corruptions"],
            severities=PROTOCOL["severities"],
        )

        global_fit = fit_temperature_nll(
            logits=global_logits,
            labels=global_labels,
            init_temperature=source_T,
            max_iter=100,
            lr=0.05,
            min_temperature=0.05,
            max_temperature=10.0,
        )

        oracle_fit_rows.append({
            "oracle_level": "target_global",
            "oracle_status": "diagnostic_only_not_deployable",
            "uses_target_labels": True,
            "backbone": backbone,
            "seed": int(seed),
            "corruption": "all",
            "severity": "all",
            "condition_group": "all_target_conditions",
            "source_T": float(source_T),
            "oracle_T": float(global_fit["temperature"]),
            "oracle_logT": float(global_fit["log_temperature"]),
            "nll_before_oracle": float(global_fit["nll_before"]),
            "nll_after_oracle": float(global_fit["nll_after"]),
            "nll_improvement_oracle": float(global_fit["nll_improvement"]),
            "n_samples": int(len(global_labels)),
            "n_conditions": int(len(global_meta)),
        })

        del global_logits, global_labels, global_meta

        # ----------------------------------------------------
        # 2. Target-family oracle: one T per corruption family
        # ----------------------------------------------------
        for corruption in PROTOCOL["corruptions"]:
            family_logits, family_labels, family_meta = collect_target_logits_labels(
                backbone=backbone,
                seed=seed,
                corruptions=[corruption],
                severities=PROTOCOL["severities"],
            )

            family_fit = fit_temperature_nll(
                logits=family_logits,
                labels=family_labels,
                init_temperature=source_T,
                max_iter=100,
                lr=0.05,
                min_temperature=0.05,
                max_temperature=10.0,
            )

            oracle_fit_rows.append({
                "oracle_level": "target_family",
                "oracle_status": "diagnostic_only_not_deployable",
                "uses_target_labels": True,
                "backbone": backbone,
                "seed": int(seed),
                "corruption": corruption,
                "severity": "all",
                "condition_group": corruption,
                "source_T": float(source_T),
                "oracle_T": float(family_fit["temperature"]),
                "oracle_logT": float(family_fit["log_temperature"]),
                "nll_before_oracle": float(family_fit["nll_before"]),
                "nll_after_oracle": float(family_fit["nll_after"]),
                "nll_improvement_oracle": float(family_fit["nll_improvement"]),
                "n_samples": int(len(family_labels)),
                "n_conditions": int(len(family_meta)),
            })

            del family_logits, family_labels, family_meta

        # ----------------------------------------------------
        # 3. Target-family×severity oracle: one T per condition
        # ----------------------------------------------------
        for corruption in PROTOCOL["corruptions"]:
            for severity in PROTOCOL["severities"]:
                cond_logits, cond_labels, cond_meta = collect_target_logits_labels(
                    backbone=backbone,
                    seed=seed,
                    corruptions=[corruption],
                    severities=[severity],
                )

                condition_fit = fit_temperature_nll(
                    logits=cond_logits,
                    labels=cond_labels,
                    init_temperature=source_T,
                    max_iter=100,
                    lr=0.05,
                    min_temperature=0.05,
                    max_temperature=10.0,
                )

                oracle_fit_rows.append({
                    "oracle_level": "target_family_severity",
                    "oracle_status": "diagnostic_only_not_deployable",
                    "uses_target_labels": True,
                    "backbone": backbone,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition_group": f"{corruption}_sev{severity}",
                    "source_T": float(source_T),
                    "oracle_T": float(condition_fit["temperature"]),
                    "oracle_logT": float(condition_fit["log_temperature"]),
                    "nll_before_oracle": float(condition_fit["nll_before"]),
                    "nll_after_oracle": float(condition_fit["nll_after"]),
                    "nll_improvement_oracle": float(condition_fit["nll_improvement"]),
                    "n_samples": int(len(cond_labels)),
                    "n_conditions": int(len(cond_meta)),
                })

                del cond_logits, cond_labels, cond_meta

oracle_fit_df = pd.DataFrame(oracle_fit_rows)

oracle_fit_path = ORACLE_DIR / "target_oracle_temperature_fits.csv"
oracle_fit_df.to_csv(oracle_fit_path, index=False)

print("Saved oracle temperature fits:", oracle_fit_path)
print("Shape:", oracle_fit_df.shape)

display(oracle_fit_df.head())
display(
    oracle_fit_df
    .groupby(["oracle_level", "backbone"], as_index=False)
    .agg(
        n_rows=("oracle_T", "count"),
        oracle_T_mean=("oracle_T", "mean"),
        oracle_T_min=("oracle_T", "min"),
        oracle_T_max=("oracle_T", "max"),
        nll_improvement_mean=("nll_improvement_oracle", "mean"),
    )
    .round(6)
)

**Validate oracle fits**

In [ ]:
assert oracle_fit_df["oracle_T"].gt(0).all(), "Oracle T must be positive"
assert oracle_fit_df["oracle_logT"].notna().all(), "Oracle logT contains NaN"
assert oracle_fit_df["nll_after_oracle"].notna().all(), "Oracle after-NLL contains NaN"

bad_oracle_fits = oracle_fit_df[
    oracle_fit_df["nll_after_oracle"]
    > oracle_fit_df["nll_before_oracle"] + 1e-6
]

assert len(bad_oracle_fits) == 0, (
    "Some oracle temperature fits worsened NLL:\n"
    + bad_oracle_fits[
        [
            "oracle_level",
            "backbone",
            "seed",
            "corruption",
            "severity",
            "nll_before_oracle",
            "nll_after_oracle",
            "oracle_T",
        ]
    ].head().to_string(index=False)
)

expected_oracle_rows = (
    # global: 2 backbones * 3 seeds * 1
    len(PROTOCOL["backbones"]) * len(PROTOCOL["seeds"]) * 1
    +
    # family: 2 backbones * 3 seeds * 10 corruptions
    len(PROTOCOL["backbones"]) * len(PROTOCOL["seeds"]) * len(PROTOCOL["corruptions"])
    +
    # family severity: 2 backbones * 3 seeds * 10 corruptions * 5 severities
    len(PROTOCOL["backbones"]) * len(PROTOCOL["seeds"]) * len(PROTOCOL["corruptions"]) * len(PROTOCOL["severities"])
)

assert len(oracle_fit_df) == expected_oracle_rows, (
    f"Expected {expected_oracle_rows} oracle rows, got {len(oracle_fit_df)}"
)

print("Oracle fit validation passed.")

**Build condition-level oracle table**

In [ ]:
condition_rows = []

for backbone in PROTOCOL["backbones"]:
    for seed in PROTOCOL["seeds"]:
        global_row = oracle_fit_df[
            (oracle_fit_df["oracle_level"] == "target_global") &
            (oracle_fit_df["backbone"] == backbone) &
            (oracle_fit_df["seed"] == seed)
        ]

        assert len(global_row) == 1
        global_T = float(global_row.iloc[0]["oracle_T"])

        for corruption in PROTOCOL["corruptions"]:
            family_row = oracle_fit_df[
                (oracle_fit_df["oracle_level"] == "target_family") &
                (oracle_fit_df["backbone"] == backbone) &
                (oracle_fit_df["seed"] == seed) &
                (oracle_fit_df["corruption"] == corruption)
            ]

            assert len(family_row) == 1
            family_T = float(family_row.iloc[0]["oracle_T"])

            for severity in PROTOCOL["severities"]:
                fs_row = oracle_fit_df[
                    (oracle_fit_df["oracle_level"] == "target_family_severity") &
                    (oracle_fit_df["backbone"] == backbone) &
                    (oracle_fit_df["seed"] == seed) &
                    (oracle_fit_df["corruption"] == corruption) &
                    (oracle_fit_df["severity"].astype(str) == str(severity))
                ]

                assert len(fs_row) == 1
                family_severity_T = float(fs_row.iloc[0]["oracle_T"])

                condition_rows.append({
                    "oracle_status": "diagnostic_only_not_deployable",
                    "uses_target_labels": True,
                    "backbone": backbone,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": f"{corruption}_sev{severity}",
                    "source_T": float(SOURCE_T_BY_BACKBONE[backbone][seed]),
                    "oracle_target_global_T": float(global_T),
                    "oracle_target_family_T": float(family_T),
                    "oracle_target_family_severity_T": float(family_severity_T),
                })

oracle_condition_table = pd.DataFrame(condition_rows)

expected_condition_rows = (
    len(PROTOCOL["backbones"])
    * len(PROTOCOL["seeds"])
    * len(PROTOCOL["corruptions"])
    * len(PROTOCOL["severities"])
)

assert len(oracle_condition_table) == expected_condition_rows

oracle_condition_path = ORACLE_DIR / "target_oracle_condition_level_temperatures.csv"
oracle_condition_table.to_csv(oracle_condition_path, index=False)

print("Saved oracle condition-level table:", oracle_condition_path)
display(oracle_condition_table.head())

**Evaluate oracle temperatures downstream on each condition**

In [ ]:
oracle_eval_rows = []

for _, row in oracle_condition_table.iterrows():
    backbone = row["backbone"]
    seed = int(row["seed"])
    corruption = row["corruption"]
    severity = int(row["severity"])

    oracle_methods = {
        "target_global_oracle_ts_diagnostic": float(row["oracle_target_global_T"]),
        "target_family_oracle_ts_diagnostic": float(row["oracle_target_family_T"]),
        "target_family_severity_oracle_ts_diagnostic": float(row["oracle_target_family_severity_T"]),
    }

    for method, T in oracle_methods.items():
        oracle_eval_rows.extend(
            evaluate_temperature_on_target_condition(
                backbone=backbone,
                seed=seed,
                corruption=corruption,
                severity=severity,
                method=method,
                information_regime="target_oracle_diagnostic_only",
                temperature=T,
                split_name="target_oracle_diagnostic",
                heldout_type="diagnostic",
                heldout_value="uses_target_labels",
            )
        )

oracle_eval_long = pd.DataFrame(oracle_eval_rows)

expected_oracle_eval_rows = (
    expected_condition_rows
    * 3
    * len(PROTOCOL["metrics"])
)

assert len(oracle_eval_long) == expected_oracle_eval_rows, (
    f"Expected {expected_oracle_eval_rows}, got {len(oracle_eval_long)}"
)

oracle_eval_path = ORACLE_DIR / "target_oracle_eval_long.csv"
oracle_eval_long.to_csv(oracle_eval_path, index=False)

print("Saved oracle eval long:", oracle_eval_path)
print("Shape:", oracle_eval_long.shape)

display(oracle_eval_long.head())

**Oracle deltas vs corrected source TS**

In [ ]:
oracle_condition_wide = eval_long_to_condition_wide(oracle_eval_long)
oracle_delta_wide = add_corrected_source_ts_deltas(oracle_condition_wide)

oracle_delta_path = ORACLE_DIR / "target_oracle_delta_vs_source_ts_wide.csv"
oracle_delta_wide.to_csv(oracle_delta_path, index=False)

print("Saved oracle delta wide:", oracle_delta_path)
print("Shape:", oracle_delta_wide.shape)

display(oracle_delta_wide.head())

**Oracle summary table**

In [ ]:
oracle_delta_summary = summarize_delta_wide(oracle_delta_wide)

oracle_delta_summary_path = ORACLE_DIR / "target_oracle_delta_summary.csv"
oracle_delta_summary.to_csv(oracle_delta_summary_path, index=False)

print("Saved oracle delta summary:", oracle_delta_summary_path)

display(
    oracle_delta_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)


**Oracle temperature by severity**

In [ ]:
oracle_temperature_by_severity = (
    oracle_condition_table
    .groupby(["backbone", "severity"], as_index=False)
    .agg(
        source_T_mean=("source_T", "mean"),
        oracle_target_global_T_mean=("oracle_target_global_T", "mean"),
        oracle_target_family_T_mean=("oracle_target_family_T", "mean"),
        oracle_target_family_severity_T_mean=("oracle_target_family_severity_T", "mean"),
        oracle_target_family_severity_T_min=("oracle_target_family_severity_T", "min"),
        oracle_target_family_severity_T_max=("oracle_target_family_severity_T", "max"),
        n_conditions=("oracle_target_family_severity_T", "count"),
    )
)

oracle_temperature_by_severity_path = ORACLE_DIR / "target_oracle_temperature_by_severity.csv"
oracle_temperature_by_severity.to_csv(oracle_temperature_by_severity_path, index=False)

print("Saved oracle temperature by severity:", oracle_temperature_by_severity_path)
display(oracle_temperature_by_severity.round(6))

**Oracle temperature by corruption**

In [ ]:
oracle_temperature_by_corruption = (
    oracle_condition_table
    .groupby(["backbone", "corruption"], as_index=False)
    .agg(
        source_T_mean=("source_T", "mean"),
        oracle_target_global_T_mean=("oracle_target_global_T", "mean"),
        oracle_target_family_T_mean=("oracle_target_family_T", "mean"),
        oracle_target_family_severity_T_mean=("oracle_target_family_severity_T", "mean"),
        oracle_target_family_severity_T_min=("oracle_target_family_severity_T", "min"),
        oracle_target_family_severity_T_max=("oracle_target_family_severity_T", "max"),
        n_conditions=("oracle_target_family_severity_T", "count"),
    )
)

oracle_temperature_by_corruption_path = ORACLE_DIR / "target_oracle_temperature_by_corruption.csv"
oracle_temperature_by_corruption.to_csv(oracle_temperature_by_corruption_path, index=False)

print("Saved oracle temperature by corruption:", oracle_temperature_by_corruption_path)

display(
    oracle_temperature_by_corruption
    .sort_values(["backbone", "oracle_target_family_severity_T_mean"], ascending=[True, False])
    .round(6)
)

**Oracle heatmap tables**

In [ ]:
oracle_heatmap_rows = []

for backbone in PROTOCOL["backbones"]:
    sub = oracle_condition_table[oracle_condition_table["backbone"] == backbone].copy()

    heatmap_table = (
        sub
        .groupby(["corruption", "severity"], as_index=False)
        .agg(
            oracle_T_mean=("oracle_target_family_severity_T", "mean"),
            oracle_T_std=("oracle_target_family_severity_T", "std"),
            source_T_mean=("source_T", "mean"),
            n=("oracle_target_family_severity_T", "count"),
        )
    )

    heatmap_path = ORACLE_DIR / f"{backbone}_oracle_family_severity_temperature_heatmap_table.csv"
    heatmap_table.to_csv(heatmap_path, index=False)

    print("Saved:", heatmap_path)

    oracle_heatmap_rows.append(
        heatmap_table.assign(backbone=backbone)
    )

oracle_heatmap_table_all = pd.concat(oracle_heatmap_rows, ignore_index=True)

oracle_heatmap_all_path = ORACLE_DIR / "all_backbones_oracle_family_severity_temperature_heatmap_table.csv"
oracle_heatmap_table_all.to_csv(oracle_heatmap_all_path, index=False)

print("Saved all-backbone oracle heatmap table:", oracle_heatmap_all_path)
display(oracle_heatmap_table_all.head())

**Compare deployable frozen V3 temperature to oracle temperature**

In [ ]:
v3_oracle_compare = frozen_v3_predictions_df.merge(
    oracle_condition_table[
        [
            "backbone",
            "seed",
            "corruption",
            "severity",
            "condition",
            "oracle_target_global_T",
            "oracle_target_family_T",
            "oracle_target_family_severity_T",
        ]
    ],
    on=["backbone", "seed", "corruption", "severity", "condition"],
    how="left",
)

assert len(v3_oracle_compare) == len(frozen_v3_predictions_df)
assert v3_oracle_compare["oracle_target_family_severity_T"].notna().all()

v3_oracle_compare["pred_minus_oracle_family_severity_T"] = (
    v3_oracle_compare["predicted_T"]
    - v3_oracle_compare["oracle_target_family_severity_T"]
)

v3_oracle_compare["abs_pred_minus_oracle_family_severity_T"] = (
    v3_oracle_compare["pred_minus_oracle_family_severity_T"].abs()
)

v3_oracle_compare_path = ORACLE_DIR / "frozen_v3_predicted_temperature_vs_oracle_diagnostic.csv"
v3_oracle_compare.to_csv(v3_oracle_compare_path, index=False)

print("Saved V3 vs oracle diagnostic table:", v3_oracle_compare_path)
display(v3_oracle_compare.head())

**V3-vs-oracle correlation summary**

In [ ]:
v3_oracle_corr_rows = []

for backbone in PROTOCOL["backbones"]:
    sub = v3_oracle_compare[v3_oracle_compare["backbone"] == backbone].copy()

    x = sub["predicted_T"].to_numpy(dtype=float)
    y = sub["oracle_target_family_severity_T"].to_numpy(dtype=float)

    pearson_result = pearsonr(x, y)
    spearman_result = spearmanr(x, y)

    v3_oracle_corr_rows.append({
        "diagnostic_status": "diagnostic_only",
        "backbone": backbone,
        "method": FROZEN_V3_CONFIG["method"],
        "pearson_predT_vs_oracle_family_severity_T": float(pearson_result.statistic),
        "pearson_p": float(pearson_result.pvalue),
        "spearman_predT_vs_oracle_family_severity_T": float(spearman_result.statistic),
        "spearman_p": float(spearman_result.pvalue),
        "predicted_T_mean": float(sub["predicted_T"].mean()),
        "predicted_T_min": float(sub["predicted_T"].min()),
        "predicted_T_max": float(sub["predicted_T"].max()),
        "oracle_family_severity_T_mean": float(sub["oracle_target_family_severity_T"].mean()),
        "oracle_family_severity_T_min": float(sub["oracle_target_family_severity_T"].min()),
        "oracle_family_severity_T_max": float(sub["oracle_target_family_severity_T"].max()),
        "mean_abs_pred_minus_oracle_T": float(sub["abs_pred_minus_oracle_family_severity_T"].mean()),
        "high_extrapolation_rate": float(sub["used_high_extrapolation"].mean()),
        "lower_fallback_rate": float(sub["used_lower_fallback"].mean()),
        "n_conditions": int(len(sub)),
    })

v3_oracle_corr_df = pd.DataFrame(v3_oracle_corr_rows)

v3_oracle_corr_path = ORACLE_DIR / "frozen_v3_vs_oracle_temperature_correlation_summary.csv"
v3_oracle_corr_df.to_csv(v3_oracle_corr_path, index=False)

print("Saved V3-vs-oracle correlation summary:", v3_oracle_corr_path)
display(v3_oracle_corr_df.round(6))

**Combined deployable + oracle diagnostic summary**

In [ ]:
oracle_for_combined = oracle_delta_summary.copy()
oracle_for_combined["comparison_block"] = "target_oracle_diagnostic_only"
oracle_for_combined["deployable"] = False

deployable_for_combined = main_deployable_summary.copy()
deployable_for_combined["deployable"] = True

combined_deployable_oracle_summary = pd.concat(
    [
        deployable_for_combined,
        oracle_for_combined,
    ],
    ignore_index=True,
    sort=False,
)

combined_deployable_oracle_path = DIRS["tables"] / "stage7_combined_deployable_and_oracle_diagnostic_summary.csv"
combined_deployable_oracle_summary.to_csv(combined_deployable_oracle_path, index=False)

print("Saved combined deployable + oracle diagnostic summary:", combined_deployable_oracle_path)

display(
    combined_deployable_oracle_summary[
        [
            "comparison_block",
            "deployable",
            "split_name",
            "information_regime",
            "backbone",
            "method",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "deployable", "delta_nll_mean"])
    .round(6)
)

**Plotting setup**

In [ ]:
PLOT_DIR_STAGE7 = DIRS["figures"] / "stage7_oracle_diagnostics"
PLOT_DIR_STAGE7.mkdir(parents=True, exist_ok=True)

print("Stage 7 plot directory:", PLOT_DIR_STAGE7)

**Oracle temperature heatmaps**

In [ ]:
for backbone in PROTOCOL["backbones"]:
    sub = oracle_condition_table[
        oracle_condition_table["backbone"] == backbone
    ].copy()

    heatmap = (
        sub
        .groupby(["corruption", "severity"], as_index=False)
        .agg(
            oracle_T_mean=("oracle_target_family_severity_T", "mean")
        )
        .pivot(index="corruption", columns="severity", values="oracle_T_mean")
        .reindex(index=PROTOCOL["corruptions"], columns=PROTOCOL["severities"])
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(heatmap.values, aspect="auto")

    ax.set_xticks(np.arange(len(PROTOCOL["severities"])))
    ax.set_xticklabels(PROTOCOL["severities"])
    ax.set_yticks(np.arange(len(PROTOCOL["corruptions"])))
    ax.set_yticklabels(PROTOCOL["corruptions"])

    ax.set_xlabel("Severity")
    ax.set_ylabel("Corruption")
    ax.set_title(f"{backbone}: target-oracle T by corruption × severity")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Oracle temperature")

    for i in range(len(PROTOCOL["corruptions"])):
        for j in range(len(PROTOCOL["severities"])):
            value = heatmap.values[i, j]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7)

    fig.tight_layout()

    out_path = PLOT_DIR_STAGE7 / f"{backbone}_oracle_temperature_heatmap.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

**Signed confidence–accuracy gap heatmaps**

In [ ]:
gap_plot_methods = [
    "raw",
    "source_global_ts",
]

for backbone in PROTOCOL["backbones"]:
    for method in gap_plot_methods:
        sub = corrected_condition_metric_wide[
            (corrected_condition_metric_wide["backbone"] == backbone) &
            (corrected_condition_metric_wide["method"] == method)
        ].copy()

        heatmap = (
            sub
            .groupby(["corruption", "severity"], as_index=False)
            .agg(
                signed_gap_mean=("signed_gap", "mean")
            )
            .pivot(index="corruption", columns="severity", values="signed_gap_mean")
            .reindex(index=PROTOCOL["corruptions"], columns=PROTOCOL["severities"])
        )

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(heatmap.values, aspect="auto")

        ax.set_xticks(np.arange(len(PROTOCOL["severities"])))
        ax.set_xticklabels(PROTOCOL["severities"])
        ax.set_yticks(np.arange(len(PROTOCOL["corruptions"])))
        ax.set_yticklabels(PROTOCOL["corruptions"])

        ax.set_xlabel("Severity")
        ax.set_ylabel("Corruption")
        ax.set_title(f"{backbone}: signed confidence–accuracy gap ({method})")

        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Mean confidence - accuracy")

        for i in range(len(PROTOCOL["corruptions"])):
            for j in range(len(PROTOCOL["severities"])):
                value = heatmap.values[i, j]
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7)

        fig.tight_layout()

        out_path = PLOT_DIR_STAGE7 / f"{backbone}_{method}_signed_gap_heatmap.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.show()

        print("Saved:", out_path)

**V3 predicted temperature vs oracle temperature scatter**

In [ ]:
for backbone in PROTOCOL["backbones"]:
    sub = v3_oracle_compare[
        v3_oracle_compare["backbone"] == backbone
    ].copy()

    fig, ax = plt.subplots(figsize=(6, 6))

    for region in ["inside_support", "lower_fallback", "high_extrapolation"]:
        rsub = sub[sub["support_region"] == region].copy()
        if len(rsub) == 0:
            continue

        ax.scatter(
            rsub["oracle_target_family_severity_T"],
            rsub["predicted_T"],
            label=region,
            alpha=0.75,
        )

    min_val = min(
        sub["oracle_target_family_severity_T"].min(),
        sub["predicted_T"].min(),
    )

    max_val = max(
        sub["oracle_target_family_severity_T"].max(),
        sub["predicted_T"].max(),
    )

    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=1)

    ax.set_xlabel("Target-oracle family×severity T")
    ax.set_ylabel("Frozen V3 predicted T")
    ax.set_title(f"{backbone}: frozen V3 vs diagnostic oracle T")
    ax.legend(fontsize=8)

    fig.tight_layout()

    out_path = PLOT_DIR_STAGE7 / f"{backbone}_frozen_v3_predT_vs_oracleT_scatter.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

**V3 vs oracle error by severity**

In [ ]:
v3_oracle_error_by_severity = (
    v3_oracle_compare
    .groupby(["backbone", "severity"], as_index=False)
    .agg(
        predicted_T_mean=("predicted_T", "mean"),
        oracle_T_mean=("oracle_target_family_severity_T", "mean"),
        mean_signed_T_error=("pred_minus_oracle_family_severity_T", "mean"),
        mean_abs_T_error=("abs_pred_minus_oracle_family_severity_T", "mean"),
        high_extrapolation_rate=("used_high_extrapolation", "mean"),
        lower_fallback_rate=("used_lower_fallback", "mean"),
        n_conditions=("predicted_T", "count"),
    )
)

v3_oracle_error_by_severity_path = ORACLE_DIR / "frozen_v3_vs_oracle_error_by_severity.csv"
v3_oracle_error_by_severity.to_csv(v3_oracle_error_by_severity_path, index=False)

print("Saved:", v3_oracle_error_by_severity_path)
display(v3_oracle_error_by_severity.round(6))


for backbone in PROTOCOL["backbones"]:
    sub = v3_oracle_error_by_severity[
        v3_oracle_error_by_severity["backbone"] == backbone
    ].copy()

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(
        sub["severity"],
        sub["predicted_T_mean"],
        marker="o",
        label="Frozen V3 predicted T",
    )

    ax.plot(
        sub["severity"],
        sub["oracle_T_mean"],
        marker="o",
        label="Oracle family×severity T",
    )

    ax.set_xlabel("Severity")
    ax.set_ylabel("Temperature")
    ax.set_title(f"{backbone}: predicted vs oracle T by severity")
    ax.set_xticks(PROTOCOL["severities"])
    ax.legend()

    fig.tight_layout()

    out_path = PLOT_DIR_STAGE7 / f"{backbone}_v3_vs_oracleT_by_severity.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

**Figure manifest**

In [ ]:
stage7_figure_rows = []

for path in sorted(PLOT_DIR_STAGE7.glob("*.png")):
    stage7_figure_rows.append({
        "stage": "stage_7",
        "figure_name": path.name,
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "role": "oracle diagnostic plot",
    })

stage7_figure_manifest = pd.DataFrame(stage7_figure_rows)

stage7_figure_manifest_path = DIRS["manifests"] / "stage7_figure_manifest.csv"
stage7_figure_manifest.to_csv(stage7_figure_manifest_path, index=False)

print("Saved Stage 7 figure manifest:", stage7_figure_manifest_path)
display(stage7_figure_manifest)

In [ ]:
stage7_note = {
    "stage": "stage_7_target_oracle_diagnostics",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "main_outputs": {
        "oracle_config": str(oracle_config_path),
        "oracle_temperature_fits": str(oracle_fit_path),
        "oracle_condition_level_temperatures": str(oracle_condition_path),
        "oracle_eval_long": str(oracle_eval_path),
        "oracle_delta_wide": str(oracle_delta_path),
        "oracle_delta_summary": str(oracle_delta_summary_path),
        "oracle_temperature_by_severity": str(oracle_temperature_by_severity_path),
        "oracle_temperature_by_corruption": str(oracle_temperature_by_corruption_path),
        "oracle_heatmap_table_all": str(oracle_heatmap_all_path),
        "frozen_v3_vs_oracle_table": str(v3_oracle_compare_path),
        "frozen_v3_vs_oracle_corr_summary": str(v3_oracle_corr_path),
        "frozen_v3_vs_oracle_error_by_severity": str(v3_oracle_error_by_severity_path),
        "combined_deployable_and_oracle_summary": str(combined_deployable_oracle_path),
        "stage7_figure_manifest": str(stage7_figure_manifest_path),
    },

    "checks": {
        "oracle_uses_target_labels": True,
        "oracle_marked_diagnostic_only": True,
        "oracle_not_deployable": True,
        "oracle_temperatures_regenerated_from_cached_target_logits": True,
        "oracle_deltas_use_corrected_refit_source_ts": True,
        "ece15_values_valid": True,
        "v3_vs_oracle_correlation_reported_as_diagnostic": True,
        "oracle_temperature_heatmaps_created": True,
        "signed_confidence_accuracy_gap_heatmaps_created": True,
        "v3_vs_oracle_plots_created": True,
    },

    "important_note": (
        "Target-oracle temperatures are fitted with target labels and are diagnostic only. "
        "They are used to study condition-dependent calibration structure and the gap between "
        "deployable methods and target-label oracle references. They must not be presented as "
        "deployable methods."
    ),
}

stage7_note_path = DIRS["logs"] / "stage7_target_oracle_diagnostics_completion.json"

with open(stage7_note_path, "w") as f:
    json.dump(stage7_note, f, indent=2)

print("Saved Stage 7 completion note:", stage7_note_path)
print(json.dumps(stage7_note, indent=2))

display(oracle_delta_summary.sort_values(["backbone", "delta_nll_mean"]).round(6))
display(oracle_temperature_by_severity.round(6))
display(v3_oracle_corr_df.round(6))
display(v3_oracle_error_by_severity.round(6))

**stage 8**

**Final method-regime table**

In [ ]:
method_regime_rows = [
    {
        "method_group": "source_only",
        "method": "raw",
        "deployable": True,
        "uses_source_labels": False,
        "uses_target_unlabeled_stream": False,
        "uses_target_labels": False,
        "description": "Uncalibrated model outputs.",
    },
    {
        "method_group": "source_only",
        "method": "source_global_ts",
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": False,
        "uses_target_labels": False,
        "description": "Single temperature fitted on clean/source calibration logits.",
    },
    {
        "method_group": "unlabeled_target_stream",
        "method": "stream_entropy_mean_only",
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": True,
        "uses_target_labels": False,
        "description": "Closest simple adaptive entropy baseline: ridge mapping from stream entropy mean to log temperature, fitted on source-bank pseudo-domains.",
    },
    {
        "method_group": "unlabeled_target_stream",
        "method": "stream_entropy_stats",
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": True,
        "uses_target_labels": False,
        "description": "Ridge mapping using entropy distribution statistics.",
    },
    {
        "method_group": "unlabeled_target_stream",
        "method": "stream_all_stats_ridge_no_class_entropy",
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": True,
        "uses_target_labels": False,
        "description": "Richer ridge mapping using entropy, confidence, margin, and free-energy stream statistics.",
    },
    {
        "method_group": "unlabeled_target_stream",
        "method": "stream_all_stats_mlp_no_class_entropy",
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": True,
        "uses_target_labels": False,
        "description": "MLP mapping using entropy, confidence, margin, and free-energy stream statistics.",
    },
    {
        "method_group": "unlabeled_target_stream_safe",
        "method": SAFE_V2_CONFIG["method"],
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": True,
        "uses_target_labels": False,
        "description": "Conservative binned isotonic entropy-to-temperature mapping without high-entropy extrapolation.",
    },
    {
        "method_group": "unlabeled_target_stream_safe",
        "method": FROZEN_V3_CONFIG["method"],
        "deployable": True,
        "uses_source_labels": True,
        "uses_target_unlabeled_stream": True,
        "uses_target_labels": False,
        "description": "Frozen support-aware safe entropy mapping with monotonicity, bounded softening, lower fallback, and capped high-entropy extrapolation.",
    },
    {
        "method_group": "target_oracle_diagnostic",
        "method": "target_global_oracle_ts_diagnostic",
        "deployable": False,
        "uses_source_labels": False,
        "uses_target_unlabeled_stream": False,
        "uses_target_labels": True,
        "description": "Diagnostic oracle temperature fitted on all target labels for a seed/backbone.",
    },
    {
        "method_group": "target_oracle_diagnostic",
        "method": "target_family_oracle_ts_diagnostic",
        "deployable": False,
        "uses_source_labels": False,
        "uses_target_unlabeled_stream": False,
        "uses_target_labels": True,
        "description": "Diagnostic oracle temperature fitted per corruption family using target labels.",
    },
    {
        "method_group": "target_oracle_diagnostic",
        "method": "target_family_severity_oracle_ts_diagnostic",
        "deployable": False,
        "uses_source_labels": False,
        "uses_target_unlabeled_stream": False,
        "uses_target_labels": True,
        "description": "Diagnostic oracle temperature fitted per corruption family and severity using target labels.",
    },
]

method_regime_table = pd.DataFrame(method_regime_rows)

method_regime_path = DIRS["tables"] / "final_method_regime_table.csv"
method_regime_table.to_csv(method_regime_path, index=False)

print("Saved method-regime table:", method_regime_path)
display(method_regime_table)

**Final figure manifest**

In [ ]:
figure_rows = []

for path in sorted((DIRS["figures"]).rglob("*.png")):
    figure_rows.append({
        "figure_name": path.name,
        "path": str(path),
        "relative_path": str(path.relative_to(PAPER_ROOT)),
        "size_bytes": int(path.stat().st_size),
        "stage": path.parent.name,
    })

final_figure_manifest = pd.DataFrame(figure_rows)

final_figure_manifest_path = DIRS["manifests"] / "final_figure_manifest.csv"
final_figure_manifest.to_csv(final_figure_manifest_path, index=False)

print("Saved final figure manifest:", final_figure_manifest_path)
print("Number of figures:", len(final_figure_manifest))
display(final_figure_manifest)

In [ ]:
final_main_table = paper_compact.copy()

final_main_table = final_main_table.merge(
    method_regime_table[["method", "deployable", "method_group"]],
    on="method",
    how="left",
)

missing_flags = final_main_table[final_main_table["deployable"].isna()]
if len(missing_flags) > 0:
    print("Methods without deployable flag in final main table:")
    display(missing_flags[["method"]].drop_duplicates())

assert final_main_table["deployable"].notna().all(), "Missing deployable flag in final main table"

final_main_table_path = DIRS["tables"] / "stage8_final_main_deployable_table.csv"
final_main_table.to_csv(final_main_table_path, index=False)

print("Saved final main deployable table:", final_main_table_path)

display(
    final_main_table
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Final LOCO/LOSO table**

In [ ]:
final_loco_loso_table = stage4_condition_split_summary.copy()

final_loco_loso_table = final_loco_loso_table.merge(
    method_regime_table[["method", "deployable", "method_group"]],
    on="method",
    how="left",
)

# For V3 method name should match exactly.
missing_flags = final_loco_loso_table[final_loco_loso_table["deployable"].isna()]
if len(missing_flags) > 0:
    print("Methods without deployable flag:")
    display(missing_flags[["method"]].drop_duplicates())

assert final_loco_loso_table["deployable"].notna().all(), "Missing deployable flag in LOCO/LOSO table"

final_loco_loso_table = final_loco_loso_table[
    [
        "block",
        "split_name",
        "heldout_type",
        "method_group",
        "method",
        "deployable",
        "backbone",
        "delta_nll_mean",
        "delta_brier_mean",
        "delta_ece15_mean",
        "delta_aece15_mean",
        "delta_abs_signed_gap_mean",
        "temperature_mean",
        "temperature_min",
        "temperature_max",
        "n_conditions",
    ]
].copy()

final_loco_loso_path = DIRS["tables"] / "stage8_final_loco_loso_downstream_table.csv"
final_loco_loso_table.to_csv(final_loco_loso_path, index=False)

print("Saved final LOCO/LOSO downstream table:", final_loco_loso_path)

display(
    final_loco_loso_table
    .sort_values(["backbone", "split_name", "delta_nll_mean"])
    .round(6)
)

**Final ablation table**

In [ ]:
final_ablation_table = stage5_compact_ablation_summary.copy()

final_ablation_path = DIRS["tables"] / "stage8_final_v3_ablation_table.csv"
final_ablation_table.to_csv(final_ablation_path, index=False)

print("Saved final V3 ablation table:", final_ablation_path)

display(
    final_ablation_table
    .sort_values(["backbone", "ablation_block", "delta_nll_mean"])
    .round(6)
)

**Final support/extrapolation table**

In [ ]:
final_support_table = support_extrapolation_full.copy()

final_support_table = final_support_table[
    [
        "backbone",
        "corruption",
        "severity",
        "support_region",
        "count",
        "fraction",
        "mean_predicted_T",
        "mean_source_T",
        "mean_entropy",
        "mean_delta_nll",
        "mean_delta_brier",
        "mean_delta_ece15",
        "mean_delta_aece15",
        "mean_delta_abs_gap",
        "worst_harm_delta_nll",
        "best_improvement_delta_nll",
    ]
].copy()

final_support_path = DIRS["tables"] / "stage8_final_support_extrapolation_by_backbone_corruption_severity.csv"
final_support_table.to_csv(final_support_path, index=False)

print("Saved final support/extrapolation table:", final_support_path)
display(final_support_table.head(40).round(6))

**Final oracle diagnostic table**

In [ ]:
final_oracle_table = oracle_delta_summary.copy()

final_oracle_table = final_oracle_table.merge(
    method_regime_table[["method", "deployable", "method_group"]],
    on="method",
    how="left",
)

assert final_oracle_table["deployable"].notna().all(), "Missing deployable flag in oracle table"
assert (final_oracle_table["deployable"] == False).all(), "Oracle rows must not be deployable"

final_oracle_table = final_oracle_table[
    [
        "method_group",
        "method",
        "deployable",
        "backbone",
        "delta_nll_mean",
        "delta_brier_mean",
        "delta_ece15_mean",
        "delta_aece15_mean",
        "delta_abs_signed_gap_mean",
        "nll_mean",
        "brier_mean",
        "ece15_mean",
        "aece15_mean",
        "temperature_mean",
        "temperature_min",
        "temperature_max",
        "n_conditions",
    ]
].copy()

final_oracle_path = DIRS["tables"] / "stage8_final_oracle_diagnostic_table.csv"
final_oracle_table.to_csv(final_oracle_path, index=False)

print("Saved final oracle diagnostic table:", final_oracle_path)

display(
    final_oracle_table
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Final V3 vs oracle diagnostic table**

In [ ]:
final_v3_oracle_corr = v3_oracle_corr_df.copy()

final_v3_oracle_corr["diagnostic_note"] = (
    "Oracle temperatures use target labels and are diagnostic only. "
    "V3 is deployable but should not be expected to match oracle magnitude."
)

final_v3_oracle_corr_path = DIRS["tables"] / "stage8_final_v3_vs_oracle_correlation_table.csv"
final_v3_oracle_corr.to_csv(final_v3_oracle_corr_path, index=False)

print("Saved final V3 vs oracle correlation table:", final_v3_oracle_corr_path)
display(final_v3_oracle_corr.round(6))

In [ ]:
# feedback checklist

output_manifest_path = DIRS["manifests"] / "output_manifest.csv"

supervisor_checklist_rows = [
    {
        "feedback_item": "Fix ECE aggregation bug",
        "status": "done",
        "evidence": str(source_ref_path),
        "note": "ECE15/AECE15 regenerated from logits and asserted to be in [0,1].",
    },
    {
        "feedback_item": "Regenerate tables/figures",
        "status": "done",
        "evidence": str(output_manifest_path),
        "note": "Corrected tables and figure manifests generated.",
    },
    {
        "feedback_item": "Refit source TS from cached clean/source calibration logits",
        "status": "done",
        "evidence": str(source_ts_refit_path),
        "note": "Source TS no longer uses old CSV as final truth.",
    },
    {
        "feedback_item": "Reproducible from cached logits",
        "status": "done",
        "evidence": str(output_manifest_path),
        "note": "Pipeline regenerates source TS, source-bank descriptors, target descriptors, V2, V3, baselines, oracle diagnostics, and figures.",
    },
    {
        "feedback_item": "Freeze V3 design",
        "status": "done",
        "evidence": str(v3_config_path),
        "note": "Frozen V3 config saved and used for final rerun.",
    },
    {
        "feedback_item": "Compare closest baselines",
        "status": "done",
        "evidence": str(main_deployable_summary_path),
        "note": "Raw, source TS, stream entropy/ridge/MLP, V2, and V3 compared.",
    },
    {
        "feedback_item": "V3 ablations",
        "status": "done",
        "evidence": str(stage5_compact_ablation_path),
        "note": "Main, LOCO, and LOSO ablations included.",
    },
    {
        "feedback_item": "Support/extrapolation table",
        "status": "done",
        "evidence": str(final_support_path),
        "note": "Support regions summarized by backbone/corruption/severity.",
    },
    {
        "feedback_item": "LOCO/LOSO downstream metrics",
        "status": "done",
        "evidence": str(stage4_summary_path),
        "note": "Evaluated NLL, Brier, ECE15, AECE15, signed gap, and abs gap.",
    },
    {
        "feedback_item": "Oracle methods marked diagnostic only",
        "status": "done",
        "evidence": str(method_regime_path),
        "note": "Oracle methods use target labels and are marked non-deployable.",
    },
    {
        "feedback_item": "Plots",
        "status": "done",
        "evidence": str(final_figure_manifest_path),
        "note": "Main comparison plots and oracle diagnostic plots generated.",
    },
    {
        "feedback_item": "Extra architecture/dataset",
        "status": "deferred",
        "evidence": "not applicable",
        "note": "Supervisor said this is optional if time allows; it will be done after ResNet18/ResNet50 are finalized.",
    },

    {
    "feedback_item": "Stream-size sensitivity",
    "status": "done",
    "evidence": str(stream_size_summary_path),
    "note": "Tests how unlabeled stream size affects entropy estimate stability, predicted temperature, downstream deltas, and support/extrapolation behavior.",
    },
]

supervisor_checklist = pd.DataFrame(supervisor_checklist_rows)

supervisor_checklist_path = DIRS["manifests"] / "supervisor_feedback_checklist.csv"
supervisor_checklist.to_csv(supervisor_checklist_path, index=False)

print("Saved supervisor checklist:", supervisor_checklist_path)
display(supervisor_checklist)

In [ ]:
final_run_summary = {
    "stage": "stage_8_final_packaging_and_audit",
    "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "paper_root": str(PAPER_ROOT),
    "backbones": PROTOCOL["backbones"],
    "seeds": PROTOCOL["seeds"],
    "corruptions": PROTOCOL["corruptions"],
    "severities": PROTOCOL["severities"],

    "main_final_tables": {
        "main_deployable_methods": str(final_main_table_path),
        "loco_loso_methods": str(final_loco_loso_path),
        "v3_ablations": str(final_ablation_path),
        "v3_support_extrapolation": str(final_support_path),
        "deployable_vs_oracle_diagnostic": str(final_oracle_path),
        "method_regime": str(method_regime_path),
    },

    "main_manifests": {
        "input_manifest": str(input_manifest_path),
        "output_manifest": str(output_manifest_path),
        "figure_manifest": str(final_figure_manifest_path),
        "supervisor_checklist": str(supervisor_checklist_path),
    },

    "key_checks": {
        "ece_bug_fixed": True,
        "source_ts_refit_from_clean_source_logits": True,
        "v3_config_frozen": True,
        "target_oracle_marked_diagnostic_only": True,
        "main_deployable_baselines_included": True,
        "safe_v2_included": True,
        "safe_v3_included": True,
        "v3_ablations_main_loco_loso_included": True,
        "support_extrapolation_table_included": True,
        "figures_included": True,
        "extra_architecture_deferred": True,
        "stream_size_sensitivity_included": True,
    },
}

final_run_summary_path = DIRS["manifests"] / "stage8_final_run_summary.json"

with open(final_run_summary_path, "w") as f:
    json.dump(final_run_summary, f, indent=2)

print("Saved final run summary:", final_run_summary_path)
print(json.dumps(final_run_summary, indent=2))

In [ ]:
readme_text = f"""# AML Paper Freeze V3 Reproducibility Package

Root:
{PAPER_ROOT}

This package contains the corrected ResNet18/ResNet50 calibration-under-corruption-shift pipeline.

Main corrections:
1. ECE aggregation bug fixed.
2. Source TS refit from cached clean/source calibration logits.
3. Source-bank and target stream descriptors regenerated from cached logits.
4. Safe V2 and frozen V3 evaluated.
5. LOCO/LOSO downstream evaluation added.
6. V3 ablations added for main, LOCO, and LOSO.
7. Support/extrapolation tables added.
8. Stream-size sensitivity added.
9. Oracle TS kept diagnostic only.

Important note:
Target-oracle temperatures use target labels. They are diagnostic only and must not be presented as deployable methods.

Frozen V3 was developed during exploratory analysis, then frozen before the final paper-level rerun.
"""

readme_path = PAPER_ROOT / "README_reproducibility.md"

with open(readme_path, "w") as f:
    f.write(readme_text)

print("Saved README:", readme_path)

In [ ]:
STAGE_ROOT = PAPER_ROOT

def file_metadata(path: Path, role: str = "") -> dict:
    path = Path(path)
    exists = path.exists()

    if exists:
        stat = path.stat()
        return {
            "path": str(path),
            "name": path.name,
            "parent": str(path.parent),
            "suffix": path.suffix,
            "exists": True,
            "size_bytes": int(stat.st_size),
            "modified_time": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            "role": role,
        }

    return {
        "path": str(path),
        "name": path.name,
        "parent": str(path.parent),
        "suffix": path.suffix,
        "exists": False,
        "size_bytes": 0,
        "modified_time": "",
        "role": role,
    }


important_output_paths = [
    # configs/manifests
    (DIRS["configs"] / "paper_freeze_protocol_config.json", "main paper-freeze protocol config"),
    (DIRS["configs"] / "frozen_v3_config.json", "frozen V3 config"),
    (DIRS["configs"] / "safe_v2_config.json", "safe V2 config"),
    (DIRS["configs"] / "stage7_oracle_diagnostics_config.json", "oracle diagnostic config"),
    (DIRS["configs"] / "stream_feature_sets.json", "stream feature sets"),

    (DIRS["manifests"] / "input_manifest.csv", "input manifest"),
    (DIRS["manifests"] / "input_path_registry.csv", "input path registry"),
    (DIRS["manifests"] / "run_config.json", "run config"),
    (DIRS["manifests"] / "frozen_v3_config.json", "frozen V3 manifest copy"),
    (DIRS["manifests"] / "cached_logits_manifest.csv", "cached target logits manifest"),
    (DIRS["manifests"] / "source_bank_logits_manifest.csv", "source-bank logits manifest"),
    (DIRS["manifests"] / "stage6_figure_manifest.csv", "stage 6 figure manifest"),
    (DIRS["manifests"] / "stage7_figure_manifest.csv", "stage 7 figure manifest"),

    # Stage 1
    (DIRS["source_calibration"] / "source_calibration_logit_candidates.csv", "source calibration logit candidates"),
    (DIRS["source_calibration"] / "selected_source_calibration_logits.csv", "selected source calibration logits"),
    (DIRS["source_ts"] / "refit_source_ts_from_clean_source_logits.csv", "refit source TS from clean/source logits"),
    (DIRS["tables"] / "corrected_baseline_eval_long.csv", "corrected raw/source TS eval long"),
    (DIRS["tables"] / "corrected_condition_metric_wide.csv", "corrected raw/source TS condition-wide metrics"),
    (DIRS["tables"] / "corrected_source_reference_summary.csv", "corrected source reference summary"),
    (DIRS["tables"] / "corrected_source_ts_delta_vs_raw_summary.csv", "corrected source TS delta vs raw summary"),

    # Stage 2
    (DIRS["stream_methods"] / "regenerated_source_bank_stream_descriptor_table.csv", "regenerated source-bank descriptors and temperatures"),
    (DIRS["stream_methods"] / "regenerated_target_stream_descriptor_table.csv", "regenerated target stream descriptors"),
    (DIRS["safe_v2"] / "safe_v2_target_temperature_predictions_unlabeled_only.csv", "safe V2 target temperature predictions"),
    (DIRS["safe_v3"] / "frozen_v3_target_temperature_predictions_unlabeled_only.csv", "frozen V3 target temperature predictions"),

    # Stage 3
    (DIRS["safe_v2"] / "safe_v2_main_delta_summary.csv", "safe V2 main summary"),
    (DIRS["safe_v3"] / "frozen_v3_main_delta_summary.csv", "frozen V3 main summary"),
    (DIRS["safe_v3"] / "frozen_v3_main_harm_summary.csv", "frozen V3 harm summary"),
    (DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_backbone_corruption_severity.csv", "frozen V3 full support table"),
    (DIRS["tables"] / "stage3_safe_v2_v3_main_delta_summary.csv", "safe V2/V3 main summary"),

    # Stage 4
    (DIRS["tables"] / "stage4_loco_loso_downstream_delta_summary.csv", "LOCO/LOSO downstream summary"),
    (DIRS["tables"] / "stage4_all_loco_loso_delta_vs_source_ts_wide.csv", "LOCO/LOSO all deltas wide"),
    (DIRS["stats"] / "stage4_loco_loso_delta_bootstrap_ci_wide.csv", "LOCO/LOSO bootstrap CI wide"),

    # Stage 5
    (DIRS["v3_ablations"] / "v3_ablation_main_delta_summary.csv", "V3 ablation main summary"),
    (DIRS["v3_ablations"] / "v3_ablation_loco_delta_summary.csv", "V3 ablation LOCO summary"),
    (DIRS["v3_ablations"] / "v3_ablation_loso_delta_summary.csv", "V3 ablation LOSO summary"),
    (DIRS["v3_ablations"] / "v3_ablation_harm_summary.csv", "V3 ablation combined harm summary"),
    (DIRS["tables"] / "stage5_compact_v3_ablation_summary.csv", "compact V3 ablation summary"),

    # Stage 6
    (DIRS["tables"] / "stage6_main_deployable_comparison_summary.csv", "main deployable comparison summary"),
    (DIRS["tables"] / "stage6_main_deployable_delta_vs_source_ts_wide.csv", "main deployable delta wide"),
    (DIRS["tables"] / "stage6_paper_compact_deployable_comparison.csv", "paper compact deployable comparison"),
    (DIRS["tables"] / "stage6_main_deployable_delta_by_severity.csv", "deployable comparison by severity"),
    (DIRS["tables"] / "stage6_main_deployable_delta_by_corruption.csv", "deployable comparison by corruption"),
    (DIRS["stats"] / "stage6_main_deployable_delta_bootstrap_ci_wide.csv", "main deployable bootstrap CI wide"),

    # Stage 7
    (DIRS["oracle_diagnostics"] / "target_oracle_temperature_fits.csv", "target oracle temperature fits"),
    (DIRS["oracle_diagnostics"] / "target_oracle_condition_level_temperatures.csv", "target oracle condition-level temperatures"),
    (DIRS["oracle_diagnostics"] / "target_oracle_delta_summary.csv", "target oracle delta summary"),
    (DIRS["oracle_diagnostics"] / "target_oracle_temperature_by_severity.csv", "oracle temperature by severity"),
    (DIRS["oracle_diagnostics"] / "target_oracle_temperature_by_corruption.csv", "oracle temperature by corruption"),
    (DIRS["oracle_diagnostics"] / "frozen_v3_vs_oracle_temperature_correlation_summary.csv", "frozen V3 vs oracle diagnostic correlation"),
    (DIRS["oracle_diagnostics"] / "frozen_v3_vs_oracle_error_by_severity.csv", "frozen V3 vs oracle error by severity"),
    (DIRS["tables"] / "stage7_combined_deployable_and_oracle_diagnostic_summary.csv", "combined deployable and oracle diagnostic summary"),

    (DIRS["configs"] / "stream_size_sensitivity_config.json", "stream-size sensitivity config"),
    (DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_wide.csv", "stream-size sensitivity wide table"),
    (DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_summary.csv", "stream-size sensitivity summary"),
    (DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_by_severity.csv", "stream-size sensitivity by severity"),
    (DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_by_support_region.csv", "stream-size sensitivity by support region"),
    (DIRS["manifests"] / "stage6_stream_size_figure_manifest.csv", "stream-size sensitivity figure manifest"),

    (readme_path, "reproducibility README"),
    (final_run_summary_path, "final run summary"),
    (supervisor_checklist_path, "supervisor feedback checklist"),
    (final_main_table_path, "final main deployable table"),
    (final_loco_loso_path, "final LOCO/LOSO downstream table"),
    (final_ablation_path, "final V3 ablation table"),
    (final_support_path, "final support/extrapolation table"),
    (final_oracle_path, "final oracle diagnostic table"),
    (final_v3_oracle_corr_path, "final V3 vs oracle correlation table"),
]

output_manifest_rows = []

for path, role in important_output_paths:
    output_manifest_rows.append(file_metadata(path, role=role))

for path in sorted(STAGE_ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in [".csv", ".json", ".md", ".txt", ".png"]:
        output_manifest_rows.append(file_metadata(path, role="auto-discovered output"))

output_manifest_df = (
    pd.DataFrame(output_manifest_rows)
    .drop_duplicates(subset=["path"])
    .reset_index(drop=True)
)

missing_outputs = output_manifest_df[output_manifest_df["exists"] == False]

assert len(missing_outputs) == 0, (
    "Some expected outputs are missing:\n"
    + missing_outputs[["role", "path"]].to_string(index=False)
)

output_manifest_path = DIRS["manifests"] / "output_manifest.csv"
output_manifest_df.to_csv(output_manifest_path, index=False)

print("Saved output manifest:", output_manifest_path)
print("Number of output files in manifest:", len(output_manifest_df))
display(output_manifest_df.head(30))

In [ ]:
# ECE sanity
assert final_main_table["ece15_mean"].between(0, 1).all(), "Final main table has invalid ECE"
assert final_loco_loso_table["delta_ece15_mean"].notna().all(), "LOCO/LOSO table has NaN ECE delta"
assert final_ablation_table["delta_ece15_mean"].notna().all(), "Ablation table has NaN ECE delta"


oracle_rows = final_oracle_table[
    final_oracle_table["method_group"] == "target_oracle_diagnostic"
].copy()

assert len(oracle_rows) > 0, "No oracle rows found in final oracle table"
assert (oracle_rows["deployable"] == False).all(), "Oracle diagnostic rows must have deployable=False"

if "ece15_mean" in oracle_rows.columns:
    assert oracle_rows["ece15_mean"].between(0, 1).all(), "Oracle table has invalid ECE"

assert len(oracle_rows) > 0, "No oracle rows found in final oracle table"

if "ece15_mean" in oracle_rows.columns:
    assert oracle_rows["ece15_mean"].between(0, 1).all(), "Oracle table has invalid ECE"

# V3 support sanity
frac_check = final_support_table.groupby(["backbone", "corruption", "severity"])["fraction"].sum()
assert np.allclose(frac_check.values, 1.0), "Final support fractions do not sum to 1"

# Required final files sanity
required_final_files = [
    readme_path,
    output_manifest_path,
    final_run_summary_path,
    method_regime_path,
    final_main_table_path,
    final_loco_loso_path,
    final_ablation_path,
    final_support_path,
    final_oracle_path,
    supervisor_checklist_path,
    final_figure_manifest_path,
]

missing_final = [str(p) for p in required_final_files if not Path(p).exists()]

assert len(missing_final) == 0, (
    "Missing final files:\n" + "\n".join(missing_final)
)

# Required figures sanity
assert len(final_figure_manifest) > 0, "No figures found in final figure manifest"

# Required checklist sanity
not_done_required = supervisor_checklist[
    (supervisor_checklist["status"] != "done") &
    (supervisor_checklist["feedback_item"] != "Extra architecture/dataset")
]

assert len(not_done_required) == 0, (
    "Some required supervisor feedback items are not marked done:\n"
    + not_done_required.to_string(index=False)
)

print("Final Stage 8 sanity checks passed.")
print("Paper-freeze package root:", STAGE_ROOT)

In [ ]:
# ============================================================
# FINAL LAST CELL FOR RESNET-18 / RESNET-50
# Collect supervisor-ready final deliverables in the same style as ViT
# ============================================================

import shutil
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. Export folders
# ============================================================

RESNET_REQUIRED_EXPORT_DIR = PAPER_ROOT / "resnet_required_final_deliverables_for_supervisor"

COMBINED_EXPORT_DIR = RESNET_REQUIRED_EXPORT_DIR / "combined_resnet18_resnet50"
COMBINED_TABLE_DIR = COMBINED_EXPORT_DIR / "tables"
COMBINED_FIGURE_DIR = COMBINED_EXPORT_DIR / "figures"
COMBINED_MANIFEST_DIR = COMBINED_EXPORT_DIR / "manifests"

PER_BACKBONE_EXPORT_DIRS = {}

for backbone in PROTOCOL["backbones"]:
    b_root = RESNET_REQUIRED_EXPORT_DIR / backbone
    PER_BACKBONE_EXPORT_DIRS[backbone] = {
        "root": b_root,
        "tables": b_root / "tables",
        "figures": b_root / "figures",
        "manifests": b_root / "manifests",
    }

for d in [RESNET_REQUIRED_EXPORT_DIR, COMBINED_EXPORT_DIR, COMBINED_TABLE_DIR, COMBINED_FIGURE_DIR, COMBINED_MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for backbone, bdirs in PER_BACKBONE_EXPORT_DIRS.items():
    for d in bdirs.values():
        d.mkdir(parents=True, exist_ok=True)

print("ResNet export root:")
print(RESNET_REQUIRED_EXPORT_DIR)


# ============================================================
# 1. Helpers
# ============================================================

def first_existing_path(candidates):
    for p in candidates:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None


def read_csv_if_exists(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None


def get_df_from_global_or_csv(global_name, candidate_paths):
    if global_name in globals():
        obj = globals()[global_name]
        if isinstance(obj, pd.DataFrame):
            return obj.copy()

    path = first_existing_path(candidate_paths)
    if path is not None:
        return pd.read_csv(path)

    return None


def save_combined_and_per_backbone(df, combined_name, per_backbone_name=None):
    if df is None:
        return None

    if per_backbone_name is None:
        per_backbone_name = combined_name

    combined_path = COMBINED_TABLE_DIR / combined_name
    df.to_csv(combined_path, index=False)

    if "backbone" in df.columns:
        for backbone in PROTOCOL["backbones"]:
            sub = df[df["backbone"] == backbone].copy()
            if len(sub) > 0:
                out_path = PER_BACKBONE_EXPORT_DIRS[backbone]["tables"] / per_backbone_name
                sub.to_csv(out_path, index=False)

    return combined_path


def copy_table_to_exports(src_path, combined_name, per_backbone_name=None):
    src_path = Path(src_path)

    if not src_path.exists():
        return None

    df = pd.read_csv(src_path)
    return save_combined_and_per_backbone(df, combined_name, per_backbone_name)


def safe_corr(x, y, method="pearson"):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)

    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan
    if x.nunique() < 2 or y.nunique() < 2:
        return np.nan

    return float(x.corr(y, method=method))


def make_bar_plot(df, x_col, y_col, title, xlabel, ylabel, out_path, rotation=30):
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.bar(df[x_col].astype(str), df[y_col].to_numpy(dtype=float))

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotation)

    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def make_line_plot(df, x_col, y_col, title, xlabel, ylabel, out_path, label_col=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    if label_col is None:
        sub = df.sort_values(x_col)
        ax.plot(sub[x_col], sub[y_col], marker="o")
    else:
        for label in sorted(df[label_col].dropna().unique()):
            sub = df[df[label_col] == label].sort_values(x_col)
            ax.plot(sub[x_col], sub[y_col], marker="o", label=str(label))
        ax.legend(fontsize=8)

    if str(y_col).startswith("delta_") or "delta" in str(y_col).lower():
        ax.axhline(0.0, linestyle="--", linewidth=1)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def copy_figure_to_backbone(src_path, backbone):
    src_path = Path(src_path)
    if src_path.exists():
        dst = PER_BACKBONE_EXPORT_DIRS[backbone]["figures"] / src_path.name
        shutil.copy2(src_path, dst)


# ============================================================
# 2. Load important existing outputs
# ============================================================

main_deployable_summary_df = get_df_from_global_or_csv(
    "main_deployable_summary",
    [
        DIRS["tables"] / "stage6_main_deployable_comparison_summary.csv",
    ],
)

stage6_main_ci_wide_df = get_df_from_global_or_csv(
    "stage6_main_ci_wide",
    [
        DIRS["stats"] / "stage6_main_deployable_delta_bootstrap_ci_wide.csv",
    ],
)

stage4_loco_loso_summary_df = get_df_from_global_or_csv(
    "stage4_condition_split_summary",
    [
        DIRS["tables"] / "stage4_loco_loso_downstream_delta_summary.csv",
    ],
)

v3_ablation_summary_df = get_df_from_global_or_csv(
    "stage5_compact_ablation_summary",
    [
        DIRS["tables"] / "stage5_compact_v3_ablation_summary.csv",
        DIRS["tables"] / "stage5_v3_ablation_main_loco_loso_summary.csv",
        DIRS["v3_ablations"] / "v3_ablation_main_delta_summary.csv",
    ],
)

support_backbone_df = get_df_from_global_or_csv(
    "support_extrapolation_by_backbone",
    [
        DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_backbone.csv",
    ],
)

support_severity_df = get_df_from_global_or_csv(
    "support_extrapolation_by_severity",
    [
        DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_severity.csv",
    ],
)

support_corruption_df = get_df_from_global_or_csv(
    "support_extrapolation_by_corruption",
    [
        DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_corruption.csv",
    ],
)

support_full_df = get_df_from_global_or_csv(
    "support_extrapolation_full",
    [
        DIRS["support_extrapolation"] / "frozen_v3_support_eval_by_backbone_corruption_severity.csv",
    ],
)

stream_size_summary_df = get_df_from_global_or_csv(
    "stream_size_summary",
    [
        DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_summary.csv",
    ],
)

stream_size_by_support_df = get_df_from_global_or_csv(
    "stream_size_by_support",
    [
        DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_by_support_region.csv",
    ],
)

source_ts_refit_df_loaded = get_df_from_global_or_csv(
    "source_ts_refit_df",
    [
        DIRS["source_ts"] / "refit_source_ts_from_clean_source_logits.csv",
    ],
)

target_oracle_summary_df = get_df_from_global_or_csv(
    "target_oracle_summary",
    [
        DIRS["oracle_diagnostics"] / "target_oracle_delta_summary.csv",
        DIRS["oracle_diagnostics"] / "target_oracle_summary.csv",
        DIRS["tables"] / "target_oracle_delta_summary.csv",
    ],
)

oracle_condition_table_df = get_df_from_global_or_csv(
    "oracle_condition_table",
    [
        DIRS["oracle_diagnostics"] / "oracle_condition_table.csv",
        DIRS["oracle_diagnostics"] / "target_oracle_condition_table.csv",
    ],
)

oracle_heatmap_table_all_df = get_df_from_global_or_csv(
    "oracle_heatmap_table_all",
    [
        DIRS["oracle_diagnostics"] / "all_backbones_oracle_family_severity_temperature_heatmap_table.csv",
    ],
)

v3_oracle_compare_df = get_df_from_global_or_csv(
    "v3_oracle_compare",
    [
        DIRS["oracle_diagnostics"] / "frozen_v3_predicted_temperature_vs_oracle_diagnostic.csv",
    ],
)

v3_oracle_corr_df_loaded = get_df_from_global_or_csv(
    "v3_oracle_corr_df",
    [
        DIRS["oracle_diagnostics"] / "frozen_v3_vs_oracle_temperature_correlation_summary.csv",
    ],
)


# ============================================================
# 3. Recreate oracle corruption×severity table if needed
# ============================================================

if oracle_heatmap_table_all_df is None and oracle_condition_table_df is not None:
    oracle_temp_col = None

    for c in [
        "oracle_target_family_severity_T",
        "target_oracle_T",
        "oracle_T",
    ]:
        if c in oracle_condition_table_df.columns:
            oracle_temp_col = c
            break

    assert oracle_temp_col is not None, "Could not find oracle temperature column in oracle_condition_table."

    oracle_heatmap_table_all_df = (
        oracle_condition_table_df
        .groupby(["backbone", "corruption", "severity"], as_index=False)
        .agg(
            oracle_T_mean=(oracle_temp_col, "mean"),
            oracle_T_std=(oracle_temp_col, "std"),
            source_T_mean=("source_T", "mean"),
            n_conditions=(oracle_temp_col, "count"),
        )
    )

if oracle_heatmap_table_all_df is not None:
    oracle_corruption_severity_path = save_combined_and_per_backbone(
        oracle_heatmap_table_all_df,
        "09_oracle_temperature_by_corruption_severity.csv",
    )
else:
    oracle_corruption_severity_path = None


# ============================================================
# 4. Recreate V3-vs-oracle correlation / underestimation summary
# ============================================================

if v3_oracle_compare_df is None:
    if "frozen_v3_predictions_df" in globals() and oracle_condition_table_df is not None:
        oracle_cols = [
            "backbone",
            "seed",
            "corruption",
            "severity",
            "condition",
        ]

        oracle_temp_col = None
        for c in [
            "oracle_target_family_severity_T",
            "target_oracle_T",
            "oracle_T",
        ]:
            if c in oracle_condition_table_df.columns:
                oracle_temp_col = c
                break

        assert oracle_temp_col is not None, "Could not find oracle temperature column for V3-vs-oracle comparison."

        v3_oracle_compare_df = frozen_v3_predictions_df.merge(
            oracle_condition_table_df[oracle_cols + [oracle_temp_col]],
            on=oracle_cols,
            how="left",
        )

        if oracle_temp_col != "oracle_target_family_severity_T":
            v3_oracle_compare_df["oracle_target_family_severity_T"] = v3_oracle_compare_df[oracle_temp_col]

if v3_oracle_compare_df is not None:
    if "oracle_target_family_severity_T" not in v3_oracle_compare_df.columns:
        for c in ["target_oracle_T", "oracle_T"]:
            if c in v3_oracle_compare_df.columns:
                v3_oracle_compare_df["oracle_target_family_severity_T"] = v3_oracle_compare_df[c]
                break

    assert "oracle_target_family_severity_T" in v3_oracle_compare_df.columns, (
        "V3-vs-oracle table exists but no oracle temperature column was found."
    )

    v3_oracle_compare_df["pred_minus_oracle_family_severity_T"] = (
        v3_oracle_compare_df["predicted_T"]
        - v3_oracle_compare_df["oracle_target_family_severity_T"]
    )

    v3_oracle_compare_df["abs_pred_minus_oracle_family_severity_T"] = (
        v3_oracle_compare_df["pred_minus_oracle_family_severity_T"].abs()
    )

    v3_oracle_compare_df["v3_underestimates_oracle_T"] = (
        v3_oracle_compare_df["predicted_T"]
        < v3_oracle_compare_df["oracle_target_family_severity_T"]
    )

    v3_oracle_compare_df["v3_overestimates_oracle_T"] = (
        v3_oracle_compare_df["predicted_T"]
        > v3_oracle_compare_df["oracle_target_family_severity_T"]
    )

    v3_oracle_corr_rows = []

    for backbone in PROTOCOL["backbones"]:
        sub = v3_oracle_compare_df[
            v3_oracle_compare_df["backbone"] == backbone
        ].copy()

        if len(sub) == 0:
            continue

        row = {
            "diagnostic_status": "diagnostic_only",
            "backbone": backbone,
            "method": FROZEN_V3_CONFIG["method"],

            "pearson_predT_vs_oracle_family_severity_T": safe_corr(
                sub["predicted_T"],
                sub["oracle_target_family_severity_T"],
                method="pearson",
            ),
            "spearman_predT_vs_oracle_family_severity_T": safe_corr(
                sub["predicted_T"],
                sub["oracle_target_family_severity_T"],
                method="spearman",
            ),
            "pearson_entropy_vs_oracle_family_severity_T": safe_corr(
                sub["entropy_mean"],
                sub["oracle_target_family_severity_T"],
                method="pearson",
            ) if "entropy_mean" in sub.columns else np.nan,
            "spearman_entropy_vs_oracle_family_severity_T": safe_corr(
                sub["entropy_mean"],
                sub["oracle_target_family_severity_T"],
                method="spearman",
            ) if "entropy_mean" in sub.columns else np.nan,

            "predicted_T_mean": float(sub["predicted_T"].mean()),
            "predicted_T_min": float(sub["predicted_T"].min()),
            "predicted_T_max": float(sub["predicted_T"].max()),

            "oracle_family_severity_T_mean": float(sub["oracle_target_family_severity_T"].mean()),
            "oracle_family_severity_T_min": float(sub["oracle_target_family_severity_T"].min()),
            "oracle_family_severity_T_max": float(sub["oracle_target_family_severity_T"].max()),

            "mean_pred_minus_oracle_T": float(sub["pred_minus_oracle_family_severity_T"].mean()),
            "median_pred_minus_oracle_T": float(sub["pred_minus_oracle_family_severity_T"].median()),
            "mean_abs_pred_minus_oracle_T": float(sub["abs_pred_minus_oracle_family_severity_T"].mean()),

            "underestimation_rate": float(sub["v3_underestimates_oracle_T"].mean()),
            "overestimation_rate": float(sub["v3_overestimates_oracle_T"].mean()),

            "high_extrapolation_rate": float(sub["used_high_extrapolation"].mean()) if "used_high_extrapolation" in sub.columns else np.nan,
            "lower_fallback_rate": float(sub["used_lower_fallback"].mean()) if "used_lower_fallback" in sub.columns else np.nan,
            "n_conditions": int(len(sub)),
        }

        v3_oracle_corr_rows.append(row)

    v3_oracle_corr_df = pd.DataFrame(v3_oracle_corr_rows)

    v3_oracle_by_severity = (
        v3_oracle_compare_df
        .groupby(["backbone", "severity"], as_index=False)
        .agg(
            n_conditions=("condition", "count"),
            predicted_T_mean=("predicted_T", "mean"),
            oracle_family_severity_T_mean=("oracle_target_family_severity_T", "mean"),
            mean_pred_minus_oracle_T=("pred_minus_oracle_family_severity_T", "mean"),
            mean_abs_pred_minus_oracle_T=("abs_pred_minus_oracle_family_severity_T", "mean"),
            underestimation_rate=("v3_underestimates_oracle_T", "mean"),
            high_extrapolation_rate=("used_high_extrapolation", "mean") if "used_high_extrapolation" in v3_oracle_compare_df.columns else ("condition", "count"),
        )
    )

    v3_oracle_by_corruption = (
        v3_oracle_compare_df
        .groupby(["backbone", "corruption"], as_index=False)
        .agg(
            n_conditions=("condition", "count"),
            predicted_T_mean=("predicted_T", "mean"),
            oracle_family_severity_T_mean=("oracle_target_family_severity_T", "mean"),
            mean_pred_minus_oracle_T=("pred_minus_oracle_family_severity_T", "mean"),
            mean_abs_pred_minus_oracle_T=("abs_pred_minus_oracle_family_severity_T", "mean"),
            underestimation_rate=("v3_underestimates_oracle_T", "mean"),
            high_extrapolation_rate=("used_high_extrapolation", "mean") if "used_high_extrapolation" in v3_oracle_compare_df.columns else ("condition", "count"),
        )
    )

    v3_oracle_condition_path = save_combined_and_per_backbone(
        v3_oracle_compare_df,
        "10b_v3_vs_oracle_condition_wide.csv",
    )

    v3_oracle_corr_path = save_combined_and_per_backbone(
        v3_oracle_corr_df,
        "10_v3_vs_oracle_correlation_underestimation_summary.csv",
    )

    v3_oracle_by_severity_path = save_combined_and_per_backbone(
        v3_oracle_by_severity,
        "10c_v3_vs_oracle_by_severity.csv",
    )

    v3_oracle_by_corruption_path = save_combined_and_per_backbone(
        v3_oracle_by_corruption,
        "10d_v3_vs_oracle_by_corruption.csv",
    )
else:
    v3_oracle_corr_df = None
    v3_oracle_by_severity = None
    v3_oracle_by_corruption = None
    v3_oracle_condition_path = None
    v3_oracle_corr_path = None
    v3_oracle_by_severity_path = None
    v3_oracle_by_corruption_path = None


# ============================================================
# 5. Save/copy required final tables
# ============================================================

created_paths = {}

created_paths["01_main_deployable_table.csv"] = save_combined_and_per_backbone(
    main_deployable_summary_df,
    "01_main_deployable_table.csv",
)

created_paths["02_main_bootstrap_ci_table.csv"] = save_combined_and_per_backbone(
    stage6_main_ci_wide_df,
    "02_main_bootstrap_ci_table.csv",
)

created_paths["03_loco_loso_table.csv"] = save_combined_and_per_backbone(
    stage4_loco_loso_summary_df,
    "03_loco_loso_table.csv",
)

created_paths["04_v3_ablation_table.csv"] = save_combined_and_per_backbone(
    v3_ablation_summary_df,
    "04_v3_ablation_table.csv",
)

created_paths["05_support_extrapolation_by_backbone.csv"] = save_combined_and_per_backbone(
    support_backbone_df,
    "05_support_extrapolation_by_backbone.csv",
)

created_paths["05b_support_extrapolation_by_severity.csv"] = save_combined_and_per_backbone(
    support_severity_df,
    "05b_support_extrapolation_by_severity.csv",
)

created_paths["05c_support_extrapolation_by_corruption.csv"] = save_combined_and_per_backbone(
    support_corruption_df,
    "05c_support_extrapolation_by_corruption.csv",
)

created_paths["05d_support_extrapolation_full.csv"] = save_combined_and_per_backbone(
    support_full_df,
    "05d_support_extrapolation_full.csv",
)

created_paths["06_stream_size_sensitivity_table.csv"] = save_combined_and_per_backbone(
    stream_size_summary_df,
    "06_stream_size_sensitivity_table.csv",
)

created_paths["06b_stream_size_by_support_region.csv"] = save_combined_and_per_backbone(
    stream_size_by_support_df,
    "06b_stream_size_by_support_region.csv",
)

created_paths["07_source_ts_refit_table.csv"] = save_combined_and_per_backbone(
    source_ts_refit_df_loaded,
    "07_source_ts_refit_table.csv",
)

created_paths["08_target_oracle_diagnostic_table.csv"] = save_combined_and_per_backbone(
    target_oracle_summary_df,
    "08_target_oracle_diagnostic_table.csv",
)

created_paths["09_oracle_temperature_by_corruption_severity.csv"] = oracle_corruption_severity_path
created_paths["10_v3_vs_oracle_correlation_underestimation_summary.csv"] = v3_oracle_corr_path
created_paths["10b_v3_vs_oracle_condition_wide.csv"] = v3_oracle_condition_path
created_paths["10c_v3_vs_oracle_by_severity.csv"] = v3_oracle_by_severity_path
created_paths["10d_v3_vs_oracle_by_corruption.csv"] = v3_oracle_by_corruption_path


# Extra useful detail tables if they exist
extra_table_candidates = {
    "02b_stage4_loco_loso_bootstrap_ci_table.csv": DIRS["stats"] / "stage4_loco_loso_delta_bootstrap_ci_wide.csv",
    "02c_v3_ablation_bootstrap_ci_table.csv": DIRS["stats"] / "v3_ablation_main_delta_bootstrap_ci_wide.csv",
    "08b_target_oracle_condition_table.csv": DIRS["oracle_diagnostics"] / "oracle_condition_table.csv",
    "08c_target_oracle_heatmap_all_backbones.csv": DIRS["oracle_diagnostics"] / "all_backbones_oracle_family_severity_temperature_heatmap_table.csv",
    "11_stage6_main_deployable_delta_wide.csv": DIRS["tables"] / "stage6_main_deployable_delta_vs_source_ts_wide.csv",
    "12_stream_size_by_severity.csv": DIRS["stream_size_sensitivity"] / "frozen_v3_stream_size_sensitivity_by_severity.csv",
}

for export_name, src in extra_table_candidates.items():
    src = Path(src)
    if src.exists():
        created_paths[export_name] = copy_table_to_exports(src, export_name)


# ============================================================
# 6. Create requested figures
# ============================================================

figure_rows = []

# ---------- Main deployable bar plots ----------
if main_deployable_summary_df is not None:
    for backbone in PROTOCOL["backbones"]:
        sub = main_deployable_summary_df[
            main_deployable_summary_df["backbone"] == backbone
        ].copy()

        if len(sub) == 0:
            continue

        if "comparison_block" in sub.columns:
            sub["plot_label"] = sub["comparison_block"].astype(str) + "\n" + sub["method"].astype(str)
        else:
            sub["plot_label"] = sub["method"].astype(str)

        sub = sub.sort_values("delta_nll_mean") if "delta_nll_mean" in sub.columns else sub

        for metric_col, ylabel in [
            ("delta_nll_mean", "Mean ΔNLL vs Source TS"),
            ("delta_brier_mean", "Mean ΔBrier vs Source TS"),
            ("delta_ece15_mean", "Mean ΔECE@15 vs Source TS"),
            ("delta_aece15_mean", "Mean ΔAECE@15 vs Source TS"),
            ("delta_abs_signed_gap_mean", "Mean Δ|confidence - accuracy|"),
        ]:
            if metric_col not in sub.columns:
                continue

            out_path = COMBINED_FIGURE_DIR / f"{backbone}_main_deployable_bar_{metric_col}.png"

            make_bar_plot(
                df=sub,
                x_col="plot_label",
                y_col=metric_col,
                title=f"{backbone}: main deployable comparison — {ylabel}",
                xlabel="Method",
                ylabel=ylabel,
                out_path=out_path,
                rotation=45,
            )

            copy_figure_to_backbone(out_path, backbone)


# ---------- Severity trends ----------
if "frozen_v3_by_severity" in globals():
    severity_trend_df = frozen_v3_by_severity.copy()
else:
    severity_trend_df = None

if severity_trend_df is not None:
    for backbone in PROTOCOL["backbones"]:
        sub = severity_trend_df[
            severity_trend_df["backbone"] == backbone
        ].copy()

        if len(sub) == 0:
            continue

        for metric_col, ylabel in [
            ("delta_nll_mean", "Mean ΔNLL vs Source TS"),
            ("delta_ece15_mean", "Mean ΔECE@15 vs Source TS"),
            ("predicted_T_mean", "Mean predicted T"),
            ("high_extrapolation_rate", "High-extrapolation rate"),
        ]:
            if metric_col not in sub.columns:
                continue

            out_path = COMBINED_FIGURE_DIR / f"{backbone}_severity_trend_{metric_col}.png"

            make_line_plot(
                df=sub,
                x_col="severity",
                y_col=metric_col,
                title=f"{backbone}: V3 severity trend — {ylabel}",
                xlabel="Severity",
                ylabel=ylabel,
                out_path=out_path,
            )

            copy_figure_to_backbone(out_path, backbone)


# ---------- Oracle temperature heatmaps ----------
if oracle_heatmap_table_all_df is not None:
    oracle_temp_col = None

    for c in [
        "oracle_T_mean",
        "oracle_target_family_severity_T_mean",
        "target_oracle_T_mean",
    ]:
        if c in oracle_heatmap_table_all_df.columns:
            oracle_temp_col = c
            break

    if oracle_temp_col is not None:
        for backbone in PROTOCOL["backbones"]:
            sub = oracle_heatmap_table_all_df[
                oracle_heatmap_table_all_df["backbone"] == backbone
            ].copy()

            if len(sub) == 0:
                continue

            heatmap_df = sub.pivot_table(
                index="corruption",
                columns="severity",
                values=oracle_temp_col,
                aggfunc="mean",
            )

            fig, ax = plt.subplots(figsize=(9, 6))

            im = ax.imshow(heatmap_df.to_numpy(dtype=float), aspect="auto")

            ax.set_xticks(np.arange(len(heatmap_df.columns)))
            ax.set_xticklabels(heatmap_df.columns)

            ax.set_yticks(np.arange(len(heatmap_df.index)))
            ax.set_yticklabels(heatmap_df.index)

            ax.set_xlabel("Severity")
            ax.set_ylabel("Corruption")
            ax.set_title(f"{backbone}: target-oracle temperature heatmap")

            cbar = fig.colorbar(im, ax=ax)
            cbar.set_label("Oracle T")

            fig.tight_layout()

            out_path = COMBINED_FIGURE_DIR / f"{backbone}_oracle_temperature_heatmap_corruption_severity.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)

            copy_figure_to_backbone(out_path, backbone)


# ---------- V3-vs-oracle scatter and severity plots ----------
if v3_oracle_compare_df is not None:
    for backbone in PROTOCOL["backbones"]:
        sub = v3_oracle_compare_df[
            v3_oracle_compare_df["backbone"] == backbone
        ].copy()

        if len(sub) == 0:
            continue

        fig, ax = plt.subplots(figsize=(6, 6))

        ax.scatter(
            sub["oracle_target_family_severity_T"],
            sub["predicted_T"],
            alpha=0.7,
        )

        min_t = float(min(sub["oracle_target_family_severity_T"].min(), sub["predicted_T"].min()))
        max_t = float(max(sub["oracle_target_family_severity_T"].max(), sub["predicted_T"].max()))

        ax.plot([min_t, max_t], [min_t, max_t], linestyle="--", linewidth=1)

        ax.set_xlabel("Target-oracle family×severity T")
        ax.set_ylabel("Frozen V3 predicted T")
        ax.set_title(f"{backbone}: V3 predicted T vs oracle T")

        fig.tight_layout()

        out_path = COMBINED_FIGURE_DIR / f"{backbone}_v3_vs_oracle_temperature_scatter.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

        copy_figure_to_backbone(out_path, backbone)

    if v3_oracle_by_severity is not None:
        for backbone in PROTOCOL["backbones"]:
            sub = v3_oracle_by_severity[
                v3_oracle_by_severity["backbone"] == backbone
            ].copy()

            if len(sub) == 0:
                continue

            fig, ax = plt.subplots(figsize=(8, 5))

            ax.plot(
                sub["severity"],
                sub["predicted_T_mean"],
                marker="o",
                label="Frozen V3 predicted T",
            )

            ax.plot(
                sub["severity"],
                sub["oracle_family_severity_T_mean"],
                marker="o",
                label="Target-oracle T",
            )

            ax.set_xlabel("Severity")
            ax.set_ylabel("Temperature")
            ax.set_title(f"{backbone}: V3 vs oracle temperature by severity")
            ax.legend()

            fig.tight_layout()

            out_path = COMBINED_FIGURE_DIR / f"{backbone}_v3_vs_oracle_temperature_by_severity.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)

            copy_figure_to_backbone(out_path, backbone)

            if "underestimation_rate" in sub.columns:
                out_path = COMBINED_FIGURE_DIR / f"{backbone}_v3_oracle_underestimation_rate_by_severity.png"

                make_line_plot(
                    df=sub,
                    x_col="severity",
                    y_col="underestimation_rate",
                    title=f"{backbone}: V3 underestimation of oracle T by severity",
                    xlabel="Severity",
                    ylabel="Underestimation rate",
                    out_path=out_path,
                )

                copy_figure_to_backbone(out_path, backbone)


# ---------- Support/extrapolation plots ----------
if support_backbone_df is not None:
    for backbone in PROTOCOL["backbones"]:
        sub = support_backbone_df[
            support_backbone_df["backbone"] == backbone
        ].copy()

        if len(sub) == 0:
            continue

        out_path = COMBINED_FIGURE_DIR / f"{backbone}_support_extrapolation_region_fraction.png"

        make_bar_plot(
            df=sub,
            x_col="support_region",
            y_col="fraction",
            title=f"{backbone}: V3 support/extrapolation distribution",
            xlabel="Support region",
            ylabel="Fraction of conditions",
            out_path=out_path,
            rotation=25,
        )

        copy_figure_to_backbone(out_path, backbone)

if support_severity_df is not None:
    for backbone in PROTOCOL["backbones"]:
        sub = support_severity_df[
            support_severity_df["backbone"] == backbone
        ].copy()

        if len(sub) == 0:
            continue

        out_path = COMBINED_FIGURE_DIR / f"{backbone}_support_extrapolation_by_severity.png"

        make_line_plot(
            df=sub,
            x_col="severity",
            y_col="fraction",
            label_col="support_region",
            title=f"{backbone}: support/extrapolation by severity",
            xlabel="Severity",
            ylabel="Fraction",
            out_path=out_path,
        )

        copy_figure_to_backbone(out_path, backbone)


# ---------- Copy existing stream-size figures ----------
if "PLOT_DIR_STREAM_SIZE" in globals():
    fig_dir = Path(PLOT_DIR_STREAM_SIZE)

    if fig_dir.exists():
        for src in sorted(fig_dir.glob("*.png")):
            dst = COMBINED_FIGURE_DIR / src.name
            shutil.copy2(src, dst)

            for backbone in PROTOCOL["backbones"]:
                if src.name.startswith(backbone):
                    copy_figure_to_backbone(dst, backbone)


# ============================================================
# 7. Build manifests
# ============================================================

def build_manifest_for_folder(root_dir, backbone_label):
    rows = []

    for subdir in ["tables", "figures", "manifests"]:
        d = root_dir / subdir
        if not d.exists():
            continue

        for p in sorted(d.glob("*")):
            if p.is_file():
                rows.append({
                    "backbone_group": backbone_label,
                    "artifact_group": subdir,
                    "artifact_name": p.name,
                    "path": str(p),
                    "size_bytes": int(p.stat().st_size),
                })

    return pd.DataFrame(rows)


combined_manifest_df = build_manifest_for_folder(COMBINED_EXPORT_DIR, "combined_resnet18_resnet50")

combined_manifest_path = COMBINED_MANIFEST_DIR / "resnet_required_final_deliverables_manifest.csv"
combined_manifest_df.to_csv(combined_manifest_path, index=False)

for backbone in PROTOCOL["backbones"]:
    b_root = PER_BACKBONE_EXPORT_DIRS[backbone]["root"]
    b_manifest_df = build_manifest_for_folder(b_root, backbone)

    b_manifest_path = PER_BACKBONE_EXPORT_DIRS[backbone]["manifests"] / f"{backbone}_required_final_deliverables_manifest.csv"
    b_manifest_df.to_csv(b_manifest_path, index=False)


missing_required = []

required_exports = [
    "01_main_deployable_table.csv",
    "02_main_bootstrap_ci_table.csv",
    "03_loco_loso_table.csv",
    "04_v3_ablation_table.csv",
    "05_support_extrapolation_by_backbone.csv",
    "06_stream_size_sensitivity_table.csv",
    "07_source_ts_refit_table.csv",
    "08_target_oracle_diagnostic_table.csv",
    "09_oracle_temperature_by_corruption_severity.csv",
    "10_v3_vs_oracle_correlation_underestimation_summary.csv",
]

for name in required_exports:
    if not (COMBINED_TABLE_DIR / name).exists():
        missing_required.append(name)

missing_required_df = pd.DataFrame({
    "missing_required_export": missing_required
})

missing_required_path = COMBINED_MANIFEST_DIR / "missing_required_exports.csv"
missing_required_df.to_csv(missing_required_path, index=False)


export_note = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "paper_root": str(PAPER_ROOT),
    "export_root": str(RESNET_REQUIRED_EXPORT_DIR),
    "combined_export_root": str(COMBINED_EXPORT_DIR),
    "per_backbone_export_roots": {
        backbone: str(PER_BACKBONE_EXPORT_DIRS[backbone]["root"])
        for backbone in PROTOCOL["backbones"]
    },
    "required_tables": required_exports,
    "n_combined_artifacts": int(len(combined_manifest_df)),
    "n_missing_required_exports": int(len(missing_required)),
    "missing_required_exports_path": str(missing_required_path),
    "combined_manifest_path": str(combined_manifest_path),
    "important_note": (
        "This folder mirrors the ViT deliverables folder structure but for ResNet-18 and ResNet-50. "
        "It includes combined ResNet outputs and separate per-backbone folders with tables, figures, and manifests. "
        "Target-oracle outputs are diagnostic only."
    ),
}

export_note_path = COMBINED_MANIFEST_DIR / "resnet_required_final_deliverables_export_note.json"

with open(export_note_path, "w") as f:
    json.dump(export_note, f, indent=2)


print("\nDONE.")
print("Combined ResNet deliverables folder:")
print(COMBINED_EXPORT_DIR)

for backbone in PROTOCOL["backbones"]:
    print(f"\n{backbone} deliverables folder:")
    print(PER_BACKBONE_EXPORT_DIRS[backbone]["root"])

print("\nCombined manifest:")
print(combined_manifest_path)

print("\nMissing required exports:")
display(missing_required_df)

print("\nCombined artifacts:")
display(combined_manifest_df)